# Integrated Financial Statement Modeling (CFA Level 1)

A from-scratch capstone notebook that ties together income statement analysis, balance sheet mechanics, cash flow derivation, and forecasting into a **single, coherent three-statement model**.

**Prerequisites:** All prior notebooks in the Financial Statement Analysis series — Introduction to Financial Statements, Income Statement Analysis, Balance Sheet & Working Capital, Cash Flow Analysis, Ratio Analysis & DuPont, and Earnings Quality.

**Outline**
1. Why build financial models
2. Setup
3. Historical financial data
4. Revenue forecasting
5. Cost structure modeling
6. Working capital modeling
7. Capital expenditure and depreciation
8. Income statement projection
9. Balance sheet projection
10. Cash flow statement projection
11. Circular reference resolution
12. Scenario analysis
13. Monte Carlo simulation
14. Model integrity checks
15. References

### What This Notebook Covers

This capstone notebook is the culmination of the entire Financial Statement Analysis module. Where previous notebooks examined each statement in isolation, here we **integrate** all three statements into a single, self-consistent forecasting model. This is how financial analysis is actually practised in industry — at investment banks, private equity firms, hedge funds, and corporate finance departments.

The integrated three-statement model is arguably the most important practical skill in finance. It forms the foundation for:

- **Discounted Cash Flow (DCF) valuation** — the projected free cash flows from our model feed directly into a DCF analysis.
- **Leveraged Buyout (LBO) modelling** — an LBO model is essentially a three-statement model with a detailed debt schedule.
- **Merger & Acquisition (M&A) analysis** — both buyer and target are modelled, then combined.
- **Credit analysis** — lenders use projected financials to assess debt service capacity.

> **CFA Exam Tip:** The CFA Level 1 curriculum emphasises the *linkages* between financial statements. Expect questions that test whether you understand how a change in one statement flows through to the others — for example, how an increase in depreciation affects the income statement (lower net income), balance sheet (lower PP&E, lower retained earnings), and cash flow statement (added back as non-cash charge).

### Model Architecture Overview

A three-statement model has a clear information hierarchy:

| Layer | Description | Key Inputs |
|-------|-------------|------------|
| **Assumptions** | Revenue growth, cost ratios, capex plans, working capital days | Historical analysis, industry research, management guidance |
| **Income Statement** | Revenue → Gross Profit → EBIT → EBT → Net Income | Assumption layer |
| **Balance Sheet** | Assets, liabilities, equity — must balance | IS (retained earnings), assumptions (WC, capex) |
| **Cash Flow Statement** | Derived from IS changes and BS changes | IS and BS |
| **Debt Schedule** | Interest, repayments, new borrowings | CF (cash available), BS (debt levels) |

The **circular reference** between interest expense (IS) → debt/cash (BS) → cash flow (CF) → debt/cash (BS) → interest expense (IS) is the central technical challenge, which we solve iteratively in Section 11.

> **Key Concept:** The model is *not* three separate forecasts stitched together. It is a single system where every number is connected. Change one assumption — say, revenue growth — and it ripples through cost of goods sold, working capital, tax, retained earnings, cash flow, debt levels, and interest expense simultaneously.

### Users of Financial Models

Different stakeholders build and use financial models for distinct purposes:

| User | Primary Goal | Key Metrics Examined | Model Horizon |
|------|-------------|---------------------|---------------|
| **Equity Research Analyst** | Fair value estimate for stock recommendation | EPS, P/E, revenue growth, margins | 2-5 years |
| **Investment Banker (M&A)** | Advise on deal pricing and structure | Enterprise value, synergies, accretion/dilution | 3-5 years |
| **Private Equity Analyst** | Assess leveraged return potential | IRR, MOIC, debt paydown, FCF yield | 5-7 years |
| **Credit Analyst / Lender** | Evaluate debt service capacity | DSCR, leverage ratios, interest coverage | Debt maturity |
| **Corporate FP&A** | Budget, forecast, and strategic planning | Revenue, EBITDA, capex, working capital | 1-3 years |
| **Hedge Fund Analyst** | Identify mispriced securities | FCF, earnings quality, variant perception | 1-3 years |

> **Common Mistake:** Beginners often focus on making the model "look right" (matching consensus estimates) rather than ensuring internal consistency. A model that produces a plausible EPS but has an unbalanced balance sheet is fundamentally broken. Always prioritise **structural integrity** over output aesthetics.

### Why this notebook is the capstone

This notebook synthesises every concept from the prior six notebooks in the Financial Statement Analysis series:
* The **income statement** structure (multi-step format, margin analysis)
* The **balance sheet** mechanics (working capital, depreciation, classification)
* The **cash flow statement** construction (indirect method, FCFF/FCFE)
* **Ratio analysis** (DuPont decomposition, efficiency ratios)
* **Earnings quality** considerations (accruals, sustainability)

The capstone task is to build a working **3-statement financial model** that projects all three statements forward simultaneously, with all the cross-statement linkages enforced. This is the single most important practical skill in financial analysis — used by equity analysts, credit analysts, M&A bankers, corporate development teams, and CFOs.

> **Key Concept:** A financial model is not a forecasting tool — it is a *simulation* tool. The model's purpose is to translate a set of operating assumptions into a complete picture of financial outcomes (income, balance sheet, cash flows). This lets the analyst test "what if" questions: What if revenue grows 5% slower? What if margins compress? What if the company makes a large acquisition? The model answers these questions with full statement-level detail.

---
## 1. Why Build Financial Models

Financial modelling is the practice of creating a **mathematical representation of a company's financial performance** — past, present, and future. The three-statement model (income statement, balance sheet, cash flow statement) is the foundation upon which virtually all corporate finance analysis is built.

### 1.1 Core use cases

| Use Case | Description |
|----------|-------------|
| **Forecasting** | Project future revenues, earnings, and cash flows to estimate where the business is heading |
| **Valuation** | DCF models require projected free cash flows; comparable company analysis needs forward multiples |
| **Credit analysis** | Lenders assess interest coverage, leverage ratios, and debt service ability under stress |
| **Scenario planning** | Management and investors test "what if" questions — recession, new product launch, acquisition |
| **Capital allocation** | Boards decide on dividends, buybacks, and reinvestment by modelling their financial impact |
| **M&A analysis** | Acquirers build models of the target to assess synergies and determine bid price |

### 1.2 The three-statement linkage

The power of an integrated model lies in its **internal consistency**. The three statements are not independent; they form a closed system:

$$\text{Net Income} \xrightarrow{\text{flows into}} \text{Retained Earnings (BS)} \xrightarrow{\text{changes drive}} \text{Cash Flow Statement}$$

More precisely:

- The **income statement** produces net income, which feeds retained earnings on the balance sheet.
- **Balance sheet** changes between periods (e.g., increase in receivables, decrease in payables) drive the operating and investing sections of the cash flow statement.
- The **cash flow statement** reconciles net income back to cash, and the ending cash balance feeds back to the balance sheet.
- **Interest expense** on the income statement depends on debt levels from the balance sheet, creating a **circular reference** that must be resolved iteratively.

> **Key Concept:** An integrated three-statement model is internally self-consistent: every dollar earned, spent, borrowed, or invested appears on all three statements in a way that satisfies the accounting equation $A = L + E$ at every point in time.

### 1.3 Model architecture

A well-structured model separates **inputs** (assumptions) from **calculations** from **outputs**:

1. **Historical data** — 3 to 5 years of actual financial statements
2. **Assumptions / drivers** — growth rates, margins, efficiency ratios, capex intensity
3. **Projection engine** — formulas that translate assumptions into projected statements
4. **Integrity checks** — automated tests that the model balances
5. **Scenario layer** — ability to toggle between base, bull, and bear assumptions

> **CFA Exam Tip:** The CFA curriculum emphasises that a good model is *transparent* (assumptions clearly stated), *flexible* (easy to change inputs), and *internally consistent* (balance sheet balances, cash flow reconciles). When answering exam questions about model design, focus on these three qualities.

### 1.4 Common pitfalls

- **Hard-coded numbers** buried in formulas instead of centralised assumption cells
- **Ignoring circular references** — modelling interest expense as a fixed number rather than linking it to average debt
- **Unbalanced balance sheets** — failing to use a "plug" (cash or revolver) to force $A = L + E$
- **Over-precision** — forecasting to the dollar when the inputs have wide uncertainty bands
- **No sanity checks** — blindly trusting outputs without testing whether implied margins, growth rates, and ratios are realistic

> **Common Mistake:** Beginners often project each statement independently and then wonder why the balance sheet does not balance. The statements *must* be built simultaneously with explicit linkages — you cannot project the income statement in isolation.

### 1.4 The Model-Building Process

Building a three-statement model follows a systematic workflow. While variations exist across firms, the core steps are universal:

1. **Gather historical data** — Collect 3-5 years of historical financials from 10-K filings, ensuring consistency in accounting treatment across periods.
2. **Analyse historical trends** — Compute margins, growth rates, efficiency ratios, and leverage metrics to establish baseline assumptions.
3. **Develop forward assumptions** — Use historical trends, industry analysis, and management guidance to project key drivers.
4. **Build the income statement** — Project revenue first, then each expense line as a function of revenue or as a fixed amount.
5. **Build the balance sheet** — Project working capital from efficiency ratios, PP&E from capex/depreciation, and equity from retained earnings.
6. **Derive cash flows** — The cash flow statement falls out mechanically from IS and BS changes.
7. **Resolve circular references** — Iteratively solve for interest expense, debt, and cash.
8. **Stress-test with scenarios** — Run bull/bear/base cases and Monte Carlo simulations.
9. **Validate with integrity checks** — Confirm the balance sheet balances and cash flows reconcile.

> **CFA Exam Tip:** When the exam asks about the "order of projection" in a financial model, remember: Income Statement first (top-down from revenue), Balance Sheet second (using IS outputs like net income), Cash Flow Statement third (derived from IS and BS). The debt/interest circular reference is resolved last.

### 1.5 Assumptions: The Heart of Every Model

Every output of a financial model is only as good as its inputs. The key assumption categories are:

| Category | Typical Assumptions | Source |
|----------|-------------------|--------|
| **Revenue** | Growth rate, volume, pricing, market share | Industry reports, management guidance, regression |
| **Costs** | COGS %, SG&A fixed/variable split, R&D intensity | Historical trends, peer comparison |
| **Working Capital** | DSO, DIO, DPO (in days) | Historical efficiency ratios |
| **Capital Expenditure** | Capex/revenue ratio, maintenance vs growth split | Management guidance, depreciation analysis |
| **Capital Structure** | Debt repayment schedule, interest rate, dividend payout | Loan agreements, company policy |
| **Tax** | Effective tax rate | Historical rate, statutory rate, tax reform |

> **Key Concept:** Assumptions should be **transparent**, **justifiable**, and **auditable**. Every number in the model should trace back to either a historical data point, an external source, or a clearly stated judgement. "Black box" assumptions destroy model credibility.

### 1.6 Common Modelling Pitfalls

Before we begin building, it is worth cataloguing the most frequent errors made by novice modellers:

1. **Hardcoded values buried in formulas** — Every assumption should live in a clearly labelled assumptions section, not embedded within calculations. In our Python implementation, we use named variables and dictionaries.
2. **Unbalanced balance sheet** — If assets do not equal liabilities plus equity, there is an error. We build automated checks in Section 14.
3. **Ignoring circular references** — Using a placeholder interest rate without iterating to convergence produces materially wrong results, especially for highly leveraged companies.
4. **Inconsistent sign conventions** — Mixing positive and negative signs for expenses, capex, and debt changes is a perennial source of errors.
5. **Over-precision** — Projecting revenue to six decimal places creates false confidence. Round appropriately and acknowledge uncertainty.

> **Common Mistake:** Many beginners project each line item independently without checking that the resulting balance sheet balances. The three statements are a *system* — you cannot project them in isolation.

### The five primary uses of financial models

Different stakeholders use 3-statement models for different purposes. Understanding the audience shapes the model's design:

| User | Primary Goal | Key Outputs | Modelling Focus |
|------|--------------|-------------|------------------|
| **Equity analyst** | Estimate fair value | EPS forecast, FCFE, target price | Revenue/margin accuracy, terminal value |
| **Credit analyst** | Assess debt service capacity | EBITDA, FCF, leverage ratios | Downside scenarios, covenants |
| **M&A banker** | Value an acquisition | Synergy estimates, accretion/dilution | Pro-forma combined financials |
| **Corporate finance team** | Strategic planning | Capital allocation, funding gap | Multi-year cash flow projection |
| **CFO / Treasurer** | Liquidity management | Cash burn rate, covenant compliance | Working capital, debt schedule |

Each use case emphasises different parts of the model. An equity analyst cares deeply about EPS and the relationship between assumed growth and required investment. A credit analyst focuses on EBITDA and the debt service coverage ratio in stressed scenarios. The same model framework serves all these users — only the assumptions and outputs of interest differ.

> **CFA Exam Tip:** The CFA curriculum emphasises that financial models must be **internally consistent** — assumptions made in one part of the model must flow through the entire system without contradiction. For example, if you assume revenue grows 10%, the corresponding inventory build, receivables increase, and capex requirement must all be consistent with that growth assumption. Inconsistency is the most common modelling error.

### The three principles of integrated modelling

Three principles distinguish a professional 3-statement model from a casual spreadsheet:

1. **Articulation:** All three statements are linked. Net income flows to retained earnings on the balance sheet. Balance sheet changes determine working capital adjustments in the cash flow statement. Ending cash on the cash flow statement equals cash on the balance sheet. Every linkage is enforced — never typed manually.

2. **Drivers, not numbers:** The model is structured around economic *drivers* (revenue growth rate, gross margin, days receivable), not directly entered dollar figures. Changing one driver should ripple through all three statements appropriately.

3. **Balance enforcement:** The accounting equation (A = L + E) must hold in every projected period. The "balancing mechanism" — using cash or debt as the plug — ensures this enforcement automatically.

> **Common Mistake:** A common beginner error is to project each statement independently — forecasting revenue and net income on the income statement, then trying to fill in the balance sheet separately. This always produces inconsistencies. The correct approach is to project drivers, derive the income statement, derive the balance sheet from working capital ratios, and let the cash flow statement reconcile the two.

---
## 2. Setup

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# Tolerances
ATOL = 1e-8
RTOL = 1e-6

# Colour palette
PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print("Setup complete.")

### Setup Notes

We use only **NumPy**, **SciPy**, and **Matplotlib** — no financial modelling libraries. This is intentional: by building every calculation from scratch, we develop a deep understanding of how each number flows through the model.

The key libraries serve specific roles:

| Library | Role in This Notebook |
|---------|----------------------|
| `numpy` | Array operations for vectorised financial calculations across projection years |
| `scipy.stats` | Probability distributions for Monte Carlo simulation (normal, triangular) |
| `scipy.optimize` | Regression for macro-driven revenue forecasting |
| `matplotlib` | Visualisation of projections, scenarios, fan charts, and sensitivity analysis |

> **Key Concept:** In practice, most financial models are built in Excel. We use Python/NumPy here for pedagogical clarity and reproducibility. The logic is identical — only the implementation medium differs. Understanding the Python version makes you *better* at Excel modelling because you must explicitly code every linkage that Excel handles implicitly through cell references.

---
## 3. Historical Financial Data

We define three years of historical financial statements for **Apex Manufacturing Corp**, a fictional mid-cap industrial company. The data is synthetic but calibrated to be realistic for a company with approximately \$2 billion in revenue.

### 3.1 Data design principles

When constructing historical data for a model:

- **Internal consistency** — the balance sheet must balance ($A = L + E$) in every historical year
- **Cash flow derivation** — historical cash flows should be derivable from changes in the balance sheet and income statement
- **Realistic ratios** — margins, turns, and leverage should fall within industry norms for manufacturing (gross margin 30-40%, operating margin 10-15%, asset turnover 0.8-1.2x)

> **Key Concept:** Historical financial data serves two purposes in modelling: (1) it provides the **base year** from which projections begin, and (2) it reveals **trends and ratios** that inform assumptions. A model is only as good as its understanding of the past.

### 3.2 Income statement (historical)

| Line Item | Year 1 | Year 2 | Year 3 |
|-----------|--------|--------|--------|
| Revenue | 1,800 | 1,950 | 2,100 |
| COGS | (1,170) | (1,248) | (1,323) |
| Gross Profit | 630 | 702 | 777 |
| SG&A | (270) | (293) | (315) |
| R&D | (90) | (98) | (105) |
| D&A | (72) | (78) | (84) |
| EBIT | 198 | 233 | 273 |
| Interest Expense | (36) | (33) | (30) |
| EBT | 162 | 200 | 243 |
| Tax (25%) | (41) | (50) | (61) |
| Net Income | 122 | 150 | 182 |

*(All figures in millions)*

### 3.3 Balance sheet (historical)

| Line Item | Year 1 | Year 2 | Year 3 |
|-----------|--------|--------|--------|
| Cash | 120 | 145 | 175 |
| Accounts Receivable | 250 | 271 | 292 |
| Inventory | 195 | 208 | 221 |
| **Current Assets** | **565** | **624** | **688** |
| Net PP&E | 900 | 942 | 978 |
| **Total Assets** | **1,465** | **1,566** | **1,666** |
| Accounts Payable | 130 | 139 | 147 |
| Short-term Debt | 50 | 45 | 40 |
| **Current Liabilities** | **180** | **184** | **187** |
| Long-term Debt | 450 | 415 | 380 |
| **Total Liabilities** | **630** | **599** | **567** |
| Common Equity | 835 | 967 | 1,099 |
| **Total L + E** | **1,465** | **1,566** | **1,666** |

> **Common Mistake:** Many students forget to verify that the balance sheet balances in the historical data before building projections. If Total Assets does not equal Total Liabilities plus Equity in the base year, every projection year will be wrong.

### 3.4 Cash flow statement (historical, indirect method)

The cash flow statement is *derived* from changes in the income statement and balance sheet. We will compute it from first principles.

### 3.3 Interpreting Historical Financials

Before projecting forward, a skilled analyst examines historical data for patterns and anomalies. Key questions to ask:

**Revenue trajectory:**
- Is growth accelerating, decelerating, or stable?
- Are there any one-time items (acquisitions, divestitures) that distort organic growth?

**Margin trends:**
- Is the gross margin expanding (pricing power, scale economies) or compressing (input cost inflation, competition)?
- Is SG&A growing faster or slower than revenue (operating leverage)?

**Balance sheet health:**
- Is working capital consuming more cash as the company grows, or is management improving efficiency?
- Is PP&E growing in line with revenue, or is the company under/over-investing?
- How is the company financed — predominantly equity or debt?

**Cash flow quality:**
- Is operating cash flow consistently exceeding net income? (A sign of high earnings quality.)
- Is free cash flow positive after capex?

> **CFA Exam Tip:** The CFA curriculum emphasises **earnings quality** — the degree to which reported earnings reflect sustainable, cash-generating operations. Key red flags include: net income consistently exceeding CFO, rising accruals, and growing divergence between revenue and cash collected.

### 3.4 Historical Ratio Benchmarks

Before building projections, we should compute key ratios from the historical data to establish baselines:

| Ratio | Formula | What It Tells Us |
|-------|---------|-----------------|
| Gross Margin | $\frac{\text{Revenue} - \text{COGS}}{\text{Revenue}}$ | Pricing power and production efficiency |
| Operating Margin | $\frac{\text{EBIT}}{\text{Revenue}}$ | Core business profitability |
| Net Margin | $\frac{\text{Net Income}}{\text{Revenue}}$ | Bottom-line profitability after all costs |
| Asset Turnover | $\frac{\text{Revenue}}{\text{Total Assets}}$ | How efficiently assets generate revenue |
| Equity Multiplier | $\frac{\text{Total Assets}}{\text{Equity}}$ | Financial leverage |
| ROE (DuPont) | Net Margin $\times$ Asset Turnover $\times$ Equity Multiplier | Return on equity decomposed into drivers |

> **Key Concept:** Historical ratios are not destiny, but they provide the **anchor** for projections. Any assumption that deviates significantly from historical norms requires explicit justification — for example, "gross margin will expand by 200 bps due to the new automated production line."

### Why historical data is the foundation

Every projection starts with three years of historical financial statements. The historical period serves four purposes:

1. **Establish trends:** Is revenue growing? Are margins expanding or compressing? Is the business becoming more or less efficient?
2. **Compute base ratios:** Days receivable, days inventory, days payable, capex-to-revenue, depreciation-to-PP&E — these ratios become the projection inputs.
3. **Identify cyclicality:** Some businesses have predictable cycles (retail seasonality, construction project timing). Three years may not capture this; longer histories may be needed.
4. **Validate assumptions:** When you project a 5% revenue growth, the historical CAGR provides a reality check. A projection that bears no relationship to history is suspect.

### About Apex Manufacturing Corp.

Apex is a hypothetical mid-cap industrial manufacturer with the following profile:
* **Revenue:** ~\$1B, growing 7-9% per year
* **Gross margin:** ~38%, slowly compressing due to input cost inflation
* **Operating margin:** ~14%
* **Capital structure:** Moderate leverage, regular dividend payer
* **Working capital intensity:** High — ~25% of revenue tied up in inventory and receivables

This profile creates a rich modelling exercise: every assumption matters because the company is capital-intensive enough that small changes in working capital efficiency or capex policy materially affect free cash flow.

> **Key Concept:** When building a real model, the "drivers" should be assumptions you can defend. Revenue growth driven by industry forecasts (from IBISWorld, Bloomberg, or company guidance) is more defensible than a number pulled from thin air. Similarly, margin assumptions should reference historical levels and discuss any expected changes (cost programs, mix shifts, pricing actions). The defensibility of assumptions matters more than the precision of the spreadsheet.

In [ ]:
# ── Historical Income Statement ($ millions) ──
years_hist = np.array([1, 2, 3])
labels_hist = ['Year 1', 'Year 2', 'Year 3']

revenue_hist      = np.array([1800.0, 1950.0, 2100.0])
cogs_hist         = np.array([1170.0, 1248.0, 1323.0])
gross_profit_hist = revenue_hist - cogs_hist
sga_hist          = np.array([270.0,  293.0,  315.0])
rd_hist           = np.array([90.0,   98.0,   105.0])
da_hist           = np.array([72.0,   78.0,   84.0])
ebit_hist         = gross_profit_hist - sga_hist - rd_hist - da_hist
interest_hist     = np.array([36.0,   33.0,   30.0])
ebt_hist          = ebit_hist - interest_hist
tax_rate          = 0.25
tax_hist          = np.round(ebt_hist * tax_rate, 1)
net_income_hist   = ebt_hist - tax_hist

print("=== Historical Income Statement ($M) ===")
for i, yr in enumerate(labels_hist):
    print(f"\n{yr}:")
    print(f"  Revenue:        {revenue_hist[i]:>8.1f}")
    print(f"  COGS:           {-cogs_hist[i]:>8.1f}")
    print(f"  Gross Profit:   {gross_profit_hist[i]:>8.1f}   ({gross_profit_hist[i]/revenue_hist[i]*100:.1f}%)")
    print(f"  SG&A:           {-sga_hist[i]:>8.1f}")
    print(f"  R&D:            {-rd_hist[i]:>8.1f}")
    print(f"  D&A:            {-da_hist[i]:>8.1f}")
    print(f"  EBIT:           {ebit_hist[i]:>8.1f}   ({ebit_hist[i]/revenue_hist[i]*100:.1f}%)")
    print(f"  Interest:       {-interest_hist[i]:>8.1f}")
    print(f"  EBT:            {ebt_hist[i]:>8.1f}")
    print(f"  Tax:            {-tax_hist[i]:>8.1f}")
    print(f"  Net Income:     {net_income_hist[i]:>8.1f}   ({net_income_hist[i]/revenue_hist[i]*100:.1f}%)")

### Interpretation: Historical Income Statement

The historical income statement for Apex Manufacturing reveals several important patterns:

- **Revenue growth** is in the mid-single digits, consistent with a mature industrial company. Year-over-year growth rates should be computed to check for acceleration or deceleration.
- **Gross margin** (Revenue minus COGS, divided by Revenue) indicates the company's fundamental pricing power and production efficiency. A stable or improving gross margin suggests the company has some ability to pass cost increases to customers.
- **SG&A** includes both fixed overhead (management salaries, rent) and variable components (sales commissions, distribution). The split matters for operating leverage analysis.
- **R&D spending** as a percentage of revenue indicates innovation intensity. For an industrial company, 3-4% is typical; technology companies might spend 15-25%.
- **Interest expense** depends on the company's debt level and cost of debt. It is *not* an operating item and should be analysed separately from operating profitability.
- **Effective tax rate** should be compared to the statutory rate. Persistent differences may indicate tax credits, international operations, or aggressive tax planning.

> **Key Concept:** When examining historical financials, always compute both **absolute changes** ($\Delta$) and **percentage changes** for each line item. A cost that grows 10% when revenue grows 5% signals margin compression — a critical input for your projection assumptions.

> **Common Mistake:** Do not confuse *accounting* profitability with *economic* profitability. A company can report positive net income while destroying value if its return on invested capital (ROIC) is below its weighted average cost of capital (WACC).

### Reading the historical income statement

Several patterns deserve attention in Apex's three-year history:

* **Revenue growth:** Trace the year-over-year growth rates. Is growth accelerating, decelerating, or stable? Apex's revenue is growing at roughly 7-9% — solid for an industrial company but not exceptional.

* **Gross margin trend:** Compute COGS / Revenue for each year. If gross margin is declining, the company is either losing pricing power or absorbing input cost inflation. Apex shows mild compression — a common pattern for capital-intensive manufacturers.

* **Operating leverage:** As revenue grows, fixed costs (rent, salaries, depreciation) become a smaller percentage of revenue. This should expand operating margin even if gross margin is stable. Apex's pattern shows whether operating leverage is working.

* **Below-the-line items:** Interest expense reflects capital structure. Tax rate consistency suggests no unusual jurisdictional shifts. A stable effective tax rate provides a reliable input for the projection.

### What to forecast vs what to derive

The historical analysis identifies which items to *forecast directly* vs which to *derive from other items*:

| Item | Approach | Reason |
|------|----------|--------|
| Revenue | Forecast directly | Top-line driver of everything else |
| COGS | % of revenue | Direct production cost scales with sales |
| SG&A | Mix of fixed + variable | Some costs fixed (executives, rent), some variable (commissions) |
| R&D | % of revenue or fixed budget | Strategic choice, often planned |
| Depreciation | Derive from PP&E + capex | Mechanical schedule |
| Interest expense | Derive from debt × interest rate | Mechanical |
| Tax | Effective tax rate × pre-tax income | Standard approach |

> **CFA Exam Tip:** When projecting an income statement, distinguish between **forecastable items** (revenue, margins, R&D budget) and **derived items** (depreciation, interest, tax). Forecastable items require judgment and assumptions; derived items are mechanical calculations from other inputs. Mixing the two leads to circular logic and inconsistent projections.

In [ ]:
# ── Historical Balance Sheet ($M) ──
cash_hist       = np.array([120.0, 145.0, 175.0])
ar_hist         = np.array([250.0, 271.0, 292.0])
inventory_hist  = np.array([195.0, 208.0, 221.0])
current_assets_hist = cash_hist + ar_hist + inventory_hist

ppe_hist        = np.array([900.0, 942.0, 978.0])
total_assets_hist = current_assets_hist + ppe_hist

ap_hist         = np.array([130.0, 139.0, 147.0])
st_debt_hist    = np.array([50.0,  45.0,  40.0])
current_liab_hist = ap_hist + st_debt_hist

lt_debt_hist    = np.array([450.0, 415.0, 380.0])
total_liab_hist = current_liab_hist + lt_debt_hist

equity_hist     = np.array([835.0, 967.0, 1099.0])
total_le_hist   = total_liab_hist + equity_hist

print("=== Historical Balance Sheet ($M) ===")
for i, yr in enumerate(labels_hist):
    print(f"\n{yr}:")
    print(f"  Cash:               {cash_hist[i]:>8.1f}")
    print(f"  Accounts Receivable:{ar_hist[i]:>8.1f}")
    print(f"  Inventory:          {inventory_hist[i]:>8.1f}")
    print(f"  Current Assets:     {current_assets_hist[i]:>8.1f}")
    print(f"  Net PP&E:           {ppe_hist[i]:>8.1f}")
    print(f"  Total Assets:       {total_assets_hist[i]:>8.1f}")
    print(f"  ---")
    print(f"  Accounts Payable:   {ap_hist[i]:>8.1f}")
    print(f"  Short-term Debt:    {st_debt_hist[i]:>8.1f}")
    print(f"  Current Liabilities:{current_liab_hist[i]:>8.1f}")
    print(f"  Long-term Debt:     {lt_debt_hist[i]:>8.1f}")
    print(f"  Total Liabilities:  {total_liab_hist[i]:>8.1f}")
    print(f"  Equity:             {equity_hist[i]:>8.1f}")
    print(f"  Total L+E:          {total_le_hist[i]:>8.1f}")
    bal = total_assets_hist[i] - total_le_hist[i]
    check = 'PASS' if abs(bal) < ATOL else 'FAIL'
    print(f"  Balance check (A-L-E): {bal:.1f} [{check}]")

### Interpretation: Historical Balance Sheet

The balance sheet snapshot reveals the company's financial position at each year-end:

- **Cash** is growing, suggesting the company generates more cash than it deploys — a positive signal for financial flexibility, but potentially a sign of under-investment if excessive.
- **Accounts Receivable** should grow roughly in line with revenue. If AR grows faster, it may indicate deteriorating collection practices or aggressive revenue recognition.
- **Inventory** growth relative to COGS indicates whether the company is managing its supply chain efficiently. Rising inventory-to-COGS ratios can foreshadow write-downs.
- **PP&E (net)** reflects cumulative capex minus accumulated depreciation. Compare PP&E growth to revenue growth — if PP&E grows slower, the company may be sweating its assets.
- **Accounts Payable** relative to COGS indicates the company's payment practices. Stretching payables conserves cash but may strain supplier relationships.
- **Debt** levels relative to equity and EBITDA indicate leverage. The debt-to-equity and net debt-to-EBITDA ratios are critical for credit analysis.
- **Equity** growth comes from retained earnings (net income minus dividends). Verify that retained earnings equal prior retained earnings plus net income minus dividends.

The fundamental identity must hold in every year:

$$\text{Total Assets} = \text{Total Liabilities} + \text{Shareholders' Equity}$$

$$\text{Cash} + \text{AR} + \text{Inventory} + \text{PP\&E} = \text{AP} + \text{Debt} + \text{Equity}$$

> **CFA Exam Tip:** Balance sheet analysis on the CFA exam often focuses on **liquidity** (current ratio, quick ratio), **solvency** (debt-to-equity, interest coverage), and **efficiency** (asset turnover, working capital ratios). Be prepared to compute these from raw financial data.

### Reading the historical balance sheet

The balance sheet tells a different story than the income statement — one about *capital deployment* and *capital structure*. Key observations for Apex:

**Asset composition:**
* **Cash:** Track absolute level and trend. A growing cash balance suggests strong free cash flow; declining cash may signal trouble or aggressive shareholder returns.
* **Working capital items (AR, inventory, AP):** Compute as percentage of revenue or COGS. These ratios drive the projections and must be stable or trending in a predictable direction.
* **PP&E:** Net PP&E = Gross PP&E − Accumulated Depreciation. Growth in net PP&E indicates the company is investing faster than it is depreciating — a sign of expansion.

**Capital structure:**
* **Debt levels:** Track total debt (current + long-term). The trend matters more than the level — is the company adding leverage or paying down debt?
* **Equity composition:** Common stock typically stable (no major issuances). Retained earnings should grow each year by net income minus dividends.

**Balance sheet linkages:**
The balance sheet shows the *cumulative* effect of all past income statement and cash flow activity. Three years of retained earnings growth equals three years of net income minus three years of dividends. This identity is a useful check on the historical data integrity.

### Computing key ratios for projection

From the historical balance sheet, compute the following ratios that will serve as projection inputs:

| Ratio | Formula | Why |
|-------|---------|-----|
| Days Sales Outstanding (DSO) | AR / Revenue × 365 | Days to collect from customers |
| Days Inventory Outstanding (DIO) | Inventory / COGS × 365 | Days inventory sits before being sold |
| Days Payable Outstanding (DPO) | AP / COGS × 365 | Days to pay suppliers |
| CapEx / Revenue | CapEx / Revenue | Investment intensity |
| Depreciation / Net PP&E | Depreciation / Net PP&E | Effective depreciation rate |
| Debt / Equity | Total Debt / Equity | Capital structure |

> **Key Concept:** These ratios are the bridge between the income statement projection and the balance sheet projection. Once you've forecasted revenue, the working capital ratios automatically tell you what AR, inventory, and AP must be. The "automatically" is critical — manually entering balance sheet items invites inconsistency.

In [ ]:
# ── Derive Historical Cash Flow Statement ($M) ──
# We need Year 0 balance sheet for change computations.
cash_y0, ar_y0, inv_y0 = 100.0, 230.0, 180.0
ap_y0, st_debt_y0, lt_debt_y0 = 120.0, 55.0, 480.0
ppe_y0 = 870.0

# Capex = D&A + change in net PP&E (since Net PP&E_t = Net PP&E_{t-1} + Capex - D&A)
ppe_prev = np.array([ppe_y0, ppe_hist[0], ppe_hist[1]])
capex_hist = ppe_hist - ppe_prev + da_hist

# Dividends = Equity_{t-1} + Net Income - Equity_t
equity_prev_arr = np.array([713.0, 835.0, 967.0])
dividends_hist = equity_prev_arr + net_income_hist - equity_hist

# Cash from operations (indirect)
delta_ar  = np.diff(np.concatenate([[ar_y0], ar_hist]))
delta_inv = np.diff(np.concatenate([[inv_y0], inventory_hist]))
delta_ap  = np.diff(np.concatenate([[ap_y0], ap_hist]))

cfo_hist = net_income_hist + da_hist - delta_ar - delta_inv + delta_ap

# Cash from investing
cfi_hist = -capex_hist

# Cash from financing
delta_st = np.diff(np.concatenate([[st_debt_y0], st_debt_hist]))
delta_lt = np.diff(np.concatenate([[lt_debt_y0], lt_debt_hist]))
cff_hist = delta_st + delta_lt - dividends_hist

# Net change in cash
net_cash_change = cfo_hist + cfi_hist + cff_hist

print("=== Historical Cash Flow Statement ($M) ===")
for i, yr in enumerate(labels_hist):
    print(f"\n{yr}:")
    print(f"  Net Income:              {net_income_hist[i]:>8.1f}")
    print(f"  + D&A:                   {da_hist[i]:>8.1f}")
    print(f"  - Increase in AR:        {-delta_ar[i]:>8.1f}")
    print(f"  - Increase in Inventory: {-delta_inv[i]:>8.1f}")
    print(f"  + Increase in AP:        {delta_ap[i]:>8.1f}")
    print(f"  Cash from Operations:    {cfo_hist[i]:>8.1f}")
    print(f"  Capital Expenditures:    {-capex_hist[i]:>8.1f}")
    print(f"  Cash from Investing:     {cfi_hist[i]:>8.1f}")
    print(f"  Change in ST Debt:       {delta_st[i]:>8.1f}")
    print(f"  Change in LT Debt:       {delta_lt[i]:>8.1f}")
    print(f"  Dividends Paid:          {-dividends_hist[i]:>8.1f}")
    print(f"  Cash from Financing:     {cff_hist[i]:>8.1f}")
    print(f"  Net Change in Cash:      {net_cash_change[i]:>8.1f}")

# Verify cash reconciliation
cash_prev_arr = np.array([cash_y0, cash_hist[0], cash_hist[1]])
print("\n--- Cash Reconciliation ---")
for i in range(3):
    implied = cash_prev_arr[i] + net_cash_change[i]
    check = 'PASS' if abs(implied - cash_hist[i]) < 0.1 else 'FAIL'
    print(f"  {labels_hist[i]}: Beginning {cash_prev_arr[i]:.1f} + Change "
          f"{net_cash_change[i]:.1f} = {implied:.1f} vs Actual {cash_hist[i]:.1f} [{check}]")

### Interpretation: Historical Cash Flow Statement

The derived cash flow statement provides crucial insights that the income statement alone cannot reveal:

- **Cash from Operations (CFO)** starts with net income and adds back non-cash charges (D&A), then adjusts for working capital changes. CFO exceeding net income is a hallmark of high earnings quality.
- **Working capital changes** can be a significant source or use of cash. A growing company typically *consumes* working capital (more AR and inventory), while efficiency improvements *release* working capital.
- **Cash from Investing (CFI)** is dominated by capital expenditure. Negative CFI indicates the company is investing for the future — this is expected and healthy for a growing industrial firm.
- **Free Cash Flow (FCF)** = CFO + CFI (where CFI is negative for capex). This is the cash available to service debt and reward equity holders.
- **Cash from Financing (CFF)** captures debt issuance/repayment and dividends. A company that consistently borrows to fund dividends is living beyond its means.

The **cash flow reconciliation** must hold:

$$\Delta\text{Cash} = \text{CFO} + \text{CFI} + \text{CFF}$$

> **Key Concept:** The cash flow statement is the hardest to manipulate because cash is objective — it is either in the bank or it is not. This makes cash flow analysis the most reliable tool for assessing financial health.

> **CFA Exam Tip:** The CFA curriculum distinguishes between **free cash flow to the firm (FCFF)** and **free cash flow to equity (FCFE)**. FCFF = CFO + Interest(1-t) - Capex, while FCFE = CFO - Capex + Net Borrowing. Know both formulas and when each is appropriate.

### Reading the derived cash flow statement

The historical cash flow statement is *derived* from the income statement and the change in balance sheets — exactly the indirect method covered in the Cash Flow Statement Analysis notebook. Key observations:

* **CFO vs Net Income:** Compare these two measures across the historical period. Is CFO consistently above net income (high earnings quality, depreciation add-back exceeds working capital consumption)? Or below (working capital growth absorbing cash)?

* **Working capital impact:** The sum of working capital changes (negative for current asset increases, positive for current liability increases) reveals whether the business is consuming or generating cash through working capital management. For growing companies, this is typically a *use* of cash.

* **Free cash flow trajectory:** Compute FCF = CFO − CapEx for each year. This is the cash available after maintaining and expanding the asset base. FCF that grows faster than revenue indicates improving capital efficiency; FCF that grows slower (or declines) indicates the opposite.

* **Financing activity:** Net financing cash flow shows whether the company is raising capital (positive — issuing debt or equity) or returning it (negative — paying dividends, buybacks, debt repayment). The pattern reveals capital allocation priorities.

### Validating cash flow consistency

A key integrity check: the cash on the balance sheet must equal the prior cash plus the net change from the cash flow statement.

$$\text{Cash}_t = \text{Cash}_{t-1} + \Delta\text{Cash from CFS}$$

If this identity fails in the historical data, there is a data error — possibly a missing line item or a timing issue. Always verify this before using the historical data as the basis for projection.

> **CFA Exam Tip:** When the exam presents historical financial statements and asks you to project cash flow, the indirect method approach is virtually always expected. Start with projected net income, add back projected depreciation, and adjust for the projected changes in working capital (which come from your projected balance sheet). The cash flow statement is the *output* of the income statement and balance sheet projections — never a separately forecasted statement.

---
## 4. Revenue Forecasting

Revenue is the single most important line item in a financial model — virtually every other item is derived from it, either directly (as a percentage of revenue) or indirectly (through balance sheet items that scale with revenue).

### 4.1 Top-down approach

Start with the **total addressable market (TAM)** and work down:

$$\text{Revenue} = \text{Industry Size} \times \text{Market Share} \times \text{Average Selling Price}$$

This approach is useful when you have macro industry data. It forces the analyst to think about whether growth comes from market expansion or share gains.

### 4.2 Bottom-up approach (segment buildup)

Start with the company's **business segments** and build up:

$$\text{Revenue} = \sum_{s=1}^{S} \text{Units}_s \times \text{Price}_s$$

or equivalently, for a multi-segment company:

$$\text{Revenue} = \sum_{s=1}^{S} \text{Revenue}_{s,t-1} \times (1 + g_s)$$

where $g_s$ is the segment-specific growth rate.

### 4.3 Growth rate method

The simplest approach — apply a **compound annual growth rate** to the base year:

$$\text{Revenue}_t = \text{Revenue}_0 \times (1 + g)^t$$

The growth rate $g$ can be estimated from:
- Historical average growth
- Analyst consensus
- Management guidance
- Regression on a macro variable (e.g., GDP)

### 4.4 Regression-based forecasting

If revenue is correlated with a macroeconomic variable $X$ (e.g., industrial production index), we can estimate:

$$\text{Revenue}_t = \alpha + \beta \cdot X_t + \varepsilon_t$$

Then use forecasted values of $X$ to project revenue.

> **Key Concept:** The best models use multiple revenue forecasting methods and compare them for reasonableness. If your growth-rate method says 12% growth but the regression says 4%, that discrepancy is a signal to investigate further.

> **CFA Exam Tip:** The CFA curriculum distinguishes between **top-down** (macro to micro) and **bottom-up** (company-specific) approaches. Top-down is better for understanding industry dynamics; bottom-up provides more granular, actionable projections. A good analyst uses both.

We implement three methods below and compare them.

### 4.4 Selecting the Right Method

The choice of revenue forecasting method depends on the company, industry, and available data:

| Method | Best For | Strengths | Weaknesses |
|--------|----------|-----------|------------|
| **Growth Rate Extrapolation** | Stable, mature companies | Simple, transparent | Ignores structural changes |
| **Regression / Macro-Linked** | Cyclical companies (industrials, autos) | Captures economic sensitivity | Requires reliable macro forecasts |
| **Segment Buildup** | Diversified companies | Granular, captures mix shifts | Data-intensive, more assumptions |
| **TAM × Market Share** | High-growth / early-stage | Anchored to market sizing | TAM estimates are often unreliable |
| **Unit × Price** | Companies with disclosed volume data | Separates volume and pricing effects | Requires granular disclosures |

### 4.5 Growth Rate Analysis in Detail

When using historical growth rates, several variants exist:

$$\text{Simple Average Growth} = \frac{1}{n} \sum_{t=1}^{n} g_t$$

$$\text{CAGR} = \left(\frac{\text{Revenue}_T}{\text{Revenue}_0}\right)^{1/T} - 1$$

$$\text{Weighted Average} = \sum_{t=1}^{n} w_t \cdot g_t \quad \text{where} \sum w_t = 1$$

The **weighted average** is often preferred because it allows the analyst to place more emphasis on recent periods, which are more likely to reflect current business conditions.

> **Common Mistake:** Using CAGR blindly can be misleading if the start or end year is unusual (e.g., a recession year or a year with a large acquisition). Always examine the individual year-by-year growth rates alongside the CAGR.

### 4.6 Regression-Based Forecasting

A regression model links revenue to one or more macroeconomic variables:

$$\text{Revenue}_t = \alpha + \beta \cdot X_t + \varepsilon_t$$

where $X_t$ might be GDP, industrial production, or a sector-specific indicator. The regression provides:

- **$\beta$** — the sensitivity of revenue to the macro variable (economic elasticity)
- **$R^2$** — how much of revenue variation is explained by the macro factor
- **Forecast** — plug in the macro forecast to get a revenue forecast

> **Key Concept:** Regression-based revenue forecasting is particularly valuable for **cyclical companies** whose fortunes are tied to the economic cycle. If you can reliably forecast the macro variable, you can produce a more informed revenue forecast than simple trend extrapolation.

### 4.7 Reconciling Multiple Forecasts

When multiple methods produce different forecasts (as they almost always do), the analyst must make a judgement call. Common approaches:

1. **Simple average** — gives equal weight to each method
2. **Weighted average** — gives more weight to the method deemed most reliable
3. **Judgemental override** — select one method as primary and use others as sanity checks
4. **Range creation** — use the spread across methods to define bull/bear/base cases

> **CFA Exam Tip:** The CFA curriculum expects candidates to understand that revenue forecasting is inherently uncertain. The appropriate response is not to pick the "best" single number but to **acknowledge the range** and conduct sensitivity analysis around it.

### The hierarchy of forecasting approaches

Revenue forecasting methods range from simple to sophisticated. The right choice depends on data availability, the business model, and the analytical purpose:

| Method | When to Use | Pros | Cons |
|--------|-------------|------|------|
| **Constant growth rate** | Stable, mature businesses | Simple, transparent | Ignores cyclicality and inflection points |
| **Declining growth rate (S-curve)** | Maturing growth companies | Captures growth deceleration | Requires curve fitting assumptions |
| **Regression on macro variable** | Cyclical industries (autos, housing) | Anchored to economic indicators | Requires forecasting the macro variable |
| **Segment buildup** | Diversified companies | Captures mix shifts | Data-intensive, requires segment disclosure |
| **Customer/unit-level model** | Subscription, retail | Most granular | Requires detailed operational data |

In practice, professional analysts often use *multiple* methods and triangulate. If three different approaches all converge on similar growth, confidence is high. If they diverge significantly, the analyst must investigate why and exercise judgment.

### Top-down vs bottom-up forecasting

Two philosophical approaches to revenue forecasting:

**Top-down:**
1. Forecast the total addressable market (TAM)
2. Estimate the company's market share
3. Revenue = TAM × Market Share

**Bottom-up:**
1. Forecast units sold by product/region
2. Forecast price per unit
3. Revenue = $\sum$ (Units × Price)

Top-down is faster and useful for high-level strategic analysis. Bottom-up is more accurate but data-intensive. Many models use bottom-up for the near-term (years 1-3, where granular data is available) and transition to top-down for the long-term (years 4-10, where granular forecasting becomes speculative).

> **Key Concept:** Revenue is the single most important forecasted variable in any financial model — every other line item ultimately depends on it. Spend disproportionate effort on revenue assumptions and document them carefully. A model with a wrong revenue forecast cannot produce useful insights, no matter how sophisticated the rest of the methodology.

> **Common Mistake:** Beginners often pick a growth rate, multiply it by current revenue, and call it done. The challenge is that growth rates are *unstable* — they vary with the business cycle, competitive dynamics, and product life cycles. A more rigorous approach decomposes revenue growth into its components: volume growth, price growth, mix shift, currency effects, and acquisitions. Each component has different drivers and different sustainability profiles.

In [ ]:
# ── Method 1: Growth rate projection ──
hist_growth = np.diff(revenue_hist) / revenue_hist[:-1]
avg_growth = np.mean(hist_growth)

n_proj = 5
years_proj = np.arange(4, 4 + n_proj)  # Years 4-8
labels_proj = [f'Year {y}' for y in years_proj]

rev_growth_method = revenue_hist[-1] * (1 + avg_growth) ** np.arange(1, n_proj + 1)

print(f"Historical growth rates: {hist_growth}")
print(f"Average growth rate: {avg_growth:.4f} ({avg_growth*100:.2f}%)")
print(f"\nMethod 1 -- Growth Rate Projection:")
for i, yr in enumerate(labels_proj):
    print(f"  {yr}: ${rev_growth_method[i]:,.1f}M")

### Interpretation: Growth Rate Projection

The growth rate extrapolation method produces a forecast by carrying forward historical momentum. Key observations:

- The **mean historical growth rate** provides the central tendency, but examine the standard deviation — high variance suggests the mean may not be a reliable predictor.
- The **projected trajectory** should be sanity-checked against industry growth forecasts. If our company is projected to grow significantly faster than the industry, we are implicitly assuming market share gains.
- Growth rate extrapolation works best for companies with **stable, predictable** revenue streams — think consumer staples or utilities. For cyclical industrials like Apex, this method may miss turning points.

> **Key Concept:** Always ask: "Is the historical growth rate sustainable?" A company that grew 15% per year for three years may have been capturing market share from a failing competitor — a one-time event that cannot repeat.

### What the constant growth method assumes

The constant growth method projects revenue as:

$$\text{Revenue}_t = \text{Revenue}_{t-1} \times (1 + g)$$

The simplicity is deceptive — embedded in this formula are several strong assumptions:

1. **No business cycle effects:** Real economies experience recessions, expansions, and recoveries. A constant growth rate smooths over these fluctuations, potentially understating risk in downturns.

2. **No competitive dynamics:** Market shares shift as competitors enter, exit, innovate, or consolidate. A constant rate assumes the company maintains its position perfectly.

3. **No product lifecycle:** New products grow rapidly, mature, then decline. A constant rate assumes the company's portfolio is in steady-state.

4. **No structural changes:** Acquisitions, divestitures, geographic expansion, and pricing changes can cause discrete jumps that constant growth misses.

> **CFA Exam Tip:** The constant growth assumption is most appropriate for **terminal value** calculations in DCF valuation — projecting many years into the future where granular forecasting is impossible. For the explicit forecast period (typically 5-10 years), more sophisticated methods are usually warranted.

Despite these limitations, the constant growth method serves as a useful **baseline** — a number to compare other methods against. If your sophisticated bottom-up forecast diverges significantly from the simple constant growth projection, you should be able to explain why.

In [ ]:
# ── Method 2: Regression on macro variable (Industrial Production Index) ──
# Synthetic macro data correlated with revenue
ipi_hist = np.array([98.0, 102.5, 107.0])  # Industrial Production Index
ipi_proj = np.array([110.0, 113.0, 115.5, 118.0, 121.0])  # Forecasted IPI

slope, intercept, r_value, p_value, std_err = stats.linregress(ipi_hist, revenue_hist)
rev_regression_method = intercept + slope * ipi_proj

print(f"Regression: Revenue = {intercept:.1f} + {slope:.2f} * IPI")
print(f"R-squared: {r_value**2:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"\nMethod 2 -- Regression Projection:")
for i, yr in enumerate(labels_proj):
    print(f"  {yr}: ${rev_regression_method[i]:,.1f}M  (IPI = {ipi_proj[i]})")

### Interpretation: Regression Forecast

The regression approach links Apex's revenue to the Industrial Production Index (IPI), producing a macro-sensitive forecast:

- A **positive $\beta$** coefficient confirms that Apex's revenue moves with industrial activity — expected for a manufacturing company.
- The **$R^2$** value indicates how much of revenue variability is explained by the macro factor. A high $R^2$ (above 0.8) suggests strong economic sensitivity.
- The **forecast** depends on the IPI projection. If the macro forecast is wrong, the revenue forecast will be wrong — this is the method's key vulnerability.
- Compare the regression forecast to the growth-rate forecast: significant divergence indicates that one method is capturing information the other misses.

> **Common Mistake:** A high $R^2$ in-sample does not guarantee good out-of-sample forecasts, especially with only a few data points. Be cautious about over-interpreting regression results from short time series.

### Why regression-based forecasting matters

The regression method ties revenue to an external economic variable — in this case, the Industrial Production Index (IPI). The logic: Apex Manufacturing's products are sold to industrial customers, so their demand should correlate with broader industrial activity.

The regression model takes the form:

$$\text{Revenue} = \alpha + \beta \cdot \text{IPI} + \varepsilon$$

The slope coefficient $\beta$ measures how sensitive Apex's revenue is to changes in industrial production. A $\beta > 1$ means Apex grows faster than the broader economy in expansions (and shrinks faster in contractions); $\beta < 1$ means Apex is more defensive.

### Selecting the right macro variable

Different industries correlate with different macro variables:

| Industry | Best Macro Anchor |
|----------|-------------------|
| Industrial manufacturing | Industrial Production Index, PMI |
| Auto manufacturing | Light vehicle sales, consumer confidence |
| Homebuilders | Housing starts, mortgage rates |
| Retail | Personal consumption expenditures, retail sales |
| Hospitality | RevPAR, business travel volume |
| Financial services | Yield curve, credit spreads |

> **Key Concept:** A regression-based forecast is only as good as the forecast of the macro variable. If you regress revenue on GDP growth, you've shifted the forecasting problem to predicting GDP — which may be no easier. The technique is most useful when the macro variable is forecasted by specialised entities (Federal Reserve, IMF, IBISWorld) whose forecasts you can adopt.

> **Common Mistake:** Spurious correlation is a real risk. Two unrelated time-series can show high correlation simply because both have trends. Always test the relationship with sufficient data (at least 5-10 years) and check that the economic logic makes sense. A statistically significant regression coefficient with no business rationale is more dangerous than no regression at all.

In [ ]:
# ── Method 3: Segment-level buildup ──
# Apex has 3 segments: Industrial (55%), Automotive (30%), Aerospace (15%)
seg_shares = np.array([0.55, 0.30, 0.15])
seg_names = ['Industrial', 'Automotive', 'Aerospace']
seg_growth = np.array([0.06, 0.08, 0.12])  # Different growth by segment

seg_revenue_base = revenue_hist[-1] * seg_shares

rev_segment_method = np.zeros(n_proj)
print("Method 3 -- Segment Buildup Projection:\n")
for t in range(n_proj):
    seg_rev_t = seg_revenue_base * (1 + seg_growth) ** (t + 1)
    rev_segment_method[t] = seg_rev_t.sum()
    print(f"  {labels_proj[t]}:")
    for s in range(3):
        print(f"    {seg_names[s]:>12s}: ${seg_rev_t[s]:>8.1f}M  (g={seg_growth[s]*100:.0f}%)")
    print(f"    {'Total':>12s}: ${rev_segment_method[t]:>8.1f}M")
    print()

### Interpretation: Segment Buildup

The segment buildup approach provides the most granular view of revenue by modelling each business unit separately:

- **Different growth rates** for each segment allow us to capture structural shifts in the business mix. If the highest-margin segment is also the fastest-growing, consolidated margins will improve even if individual segment margins are flat.
- The **sum of segments** should reconcile to total revenue. Any "corporate" or "other" category should be modelled separately.
- This method is particularly useful when segments have **different economic drivers** — for example, industrial revenues may track manufacturing PMI while aerospace revenues track defence budgets.

> **Key Concept:** Segment-level modelling reveals information hidden in consolidated figures. A company growing revenue at 5% might have one segment growing at 15% and another declining at 5% — very different strategic implications than uniform 5% growth across all segments.

### The segment buildup approach

Diversified companies sell different products into different markets, each with their own growth dynamics. A segment-level forecast aggregates these:

$$\text{Total Revenue} = \sum_{i=1}^{N} \text{Revenue}_i = \sum_{i=1}^{N} \text{Revenue}_{i,t-1} \times (1 + g_i)$$

The advantage: each segment's growth rate can reflect its specific dynamics. Apex's industrial equipment may grow at industrial-economy rates, while its services segment may grow faster (services typically expand faster than manufactured goods in mature economies).

### The reporting requirement

US public companies must report segment-level revenue under ASC 280 (Segment Reporting). This data is found in the 10-K footnotes and provides the historical inputs for segment forecasting. International companies follow IFRS 8 (Operating Segments) with similar disclosure requirements.

The challenge: companies define segments based on internal management structure, which may not align with how investors think about the business. A "Services" segment may include very different services with different growth profiles. Analysts must interpret segment data carefully and sometimes redefine segments based on disclosure detail.

> **Key Concept:** Segment-level forecasting often produces a different (usually lower) total than aggregate forecasting. This is because the *fastest-growing segment* tends to dominate aggregate growth — its share of revenue increases each year, pulling the average growth rate up. A naive aggregate forecast that assumes constant growth misses this **mix shift** effect. The segment buildup correctly captures it.

### Limitations of segment buildup

* **Data quality:** Segment definitions change over time as companies reorganize, making historical comparisons difficult.
* **Allocations:** Corporate overhead and shared costs may be allocated to segments in ways that distort underlying economics.
* **Granularity:** Most companies report 3-5 segments, which may be insufficient to capture truly different growth dynamics.

In [ ]:
# ── Compare the three methods ──
fig, ax = plt.subplots(figsize=(10, 6))

all_years = np.concatenate([years_hist, years_proj])
ax.plot(years_hist, revenue_hist, 'ko-', markersize=8, linewidth=2, label='Historical')
ax.plot(years_proj, rev_growth_method, 's--', color=PRIMARY, markersize=7, label='Growth Rate')
ax.plot(years_proj, rev_regression_method, 'D--', color=SECONDARY, markersize=7, label='Regression (IPI)')
ax.plot(years_proj, rev_segment_method, '^--', color=TERTIARY, markersize=7, label='Segment Buildup')

ax.axvline(x=3.5, color='gray', linestyle=':', alpha=0.7)
ax.text(3.55, revenue_hist[-1] * 1.15, 'Forecast\nperiod', fontsize=10, color='gray')
ax.set_xlabel('Year')
ax.set_ylabel('Revenue ($M)')
ax.set_title('Revenue Forecast Comparison -- Three Methods')
ax.legend()
ax.set_xticks(all_years)
ax.set_xticklabels(['Y1','Y2','Y3','Y4','Y5','Y6','Y7','Y8'])
plt.tight_layout()
plt.show()

# Use weighted average for the model
weights = np.array([0.3, 0.3, 0.4])  # Slight preference for segment buildup
revenue_proj = (weights[0] * rev_growth_method +
                weights[1] * rev_regression_method +
                weights[2] * rev_segment_method)
print("Blended revenue forecast (30% growth, 30% regression, 40% segment):")
for i, yr in enumerate(labels_proj):
    print(f"  {yr}: ${revenue_proj[i]:,.1f}M")

### Interpretation: Revenue Forecast Comparison

The chart above compares our three revenue forecasting methods side-by-side. This visual comparison is a critical step in the modelling process:

- If all three methods converge on a similar trajectory, we can have **higher confidence** in the forecast.
- If they diverge, the spread defines a natural **uncertainty range** that can inform our scenario analysis in Section 12.
- The method selected for the base case should be the one most appropriate for Apex's business characteristics. For a cyclical industrial company, the regression-based approach may be most theoretically sound, but requires reliable macro forecasts.
- In practice, many analysts use a **blended approach** — perhaps 50% weight on the segment buildup (for granularity), 30% on the regression (for macro sensitivity), and 20% on the growth rate (for simplicity and mean-reversion).

> **CFA Exam Tip:** When presented with multiple forecasting methods on the exam, be prepared to discuss the **trade-offs** of each. The exam often asks which method is most appropriate given specific company characteristics (cyclical vs defensive, diversified vs focused, data-rich vs data-poor).

### Triangulating across methods

The three forecast methods produce different revenue trajectories. The question is: which to use?

In practice, professional analysts:
1. Run all three (or more) methods
2. Investigate divergences — why does the regression forecast differ from the segment buildup?
3. Build a "consensus view" that may blend methods or pick one based on which makes the most defensible assumptions
4. Document the choice and its rationale

The output is rarely a single point estimate. More commonly, analysts present a **range** with high/low/mid scenarios, each tied to specific assumptions.

### Sensitivity to method choice

A 1-percentage-point difference in growth rate compounds dramatically over 5 years:
* 7% growth: \$1.0B grows to \$1.40B (+40%)
* 8% growth: \$1.0B grows to \$1.47B (+47%)
* 9% growth: \$1.0B grows to \$1.54B (+54%)

The 2-percentage-point spread (7% to 9%) creates a 14-percentage-point spread in terminal revenue. Every downstream metric (operating income, cash flow, valuation) inherits this uncertainty.

> **CFA Exam Tip:** When the exam asks you to forecast revenue, justify your choice of method. State why the chosen approach is appropriate for the company's industry and stage, and acknowledge what the alternative methods would have implied. This demonstrates the analytical maturity the exam rewards.

> **Common Mistake:** Forecasters sometimes "anchor" on management guidance — adopting whatever growth rate the company itself projects. This is dangerous because management has incentives to be optimistic. A rigorous analyst treats management guidance as one input among many, not as the answer.

---
## 5. Cost Structure Modeling

Once revenue is projected, we model each major expense category. The key insight is that costs have both **fixed** and **variable** components, and understanding the mix determines **operating leverage**.

### 5.1 Fixed vs variable costs

$$\text{Total Cost} = \text{Fixed Cost} + \text{Variable Cost per Unit} \times \text{Units}$$

In percentage-of-revenue terms:

$$\frac{\text{Cost}}{\text{Revenue}} = \frac{\text{Fixed Cost}}{\text{Revenue}} + \text{Variable Ratio}$$

As revenue grows, the fixed-cost ratio *declines* (operating leverage), improving margins.

### 5.2 Operating leverage

The **degree of operating leverage (DOL)** measures how sensitive EBIT is to revenue changes:

$$\text{DOL} = \frac{\%\Delta \text{EBIT}}{\%\Delta \text{Revenue}} = \frac{\text{Contribution Margin}}{\text{EBIT}}$$

A company with high fixed costs (high DOL) sees profits swing more dramatically with revenue changes — amplifying gains in good times and losses in bad times.

### 5.3 Modelling approach

For each cost line, we compute the historical **cost-as-a-percentage-of-revenue**, identify trends, and project forward:

| Cost Item | Historical Range | Projection Assumption |
|-----------|-----------------|----------------------|
| COGS | 63.0% - 65.0% | Gradual improvement due to scale |
| SG&A | 14.5% - 15.0% | Slight leverage from fixed component |
| R&D | 5.0% - 5.0% | Held constant as percentage of revenue |
| D&A | 3.8% - 4.0% | Linked to PP&E balance |

> **Key Concept:** COGS is primarily variable (raw materials, direct labour) while SG&A has a significant fixed component (rent, management salaries). This distinction matters enormously for scenario analysis — in a downturn, a company with mostly variable costs can flex expenses down, while one with mostly fixed costs cannot.

> **CFA Exam Tip:** When the exam asks about operating leverage, remember: high DOL means high fixed costs, which means volatile earnings. A cyclical manufacturer with high DOL is much riskier than a service company with mostly variable costs.

### 5.3 Operating Leverage: A Critical Concept

Operating leverage measures how sensitive operating income is to changes in revenue:

$$\text{Degree of Operating Leverage (DOL)} = \frac{\%\Delta \text{EBIT}}{\%\Delta \text{Revenue}} = \frac{\text{Revenue} - \text{Variable Costs}}{\text{Revenue} - \text{Variable Costs} - \text{Fixed Costs}}$$

A company with high operating leverage (high fixed costs relative to variable costs) will see large swings in profitability from small changes in revenue:

| Scenario | Low Operating Leverage | High Operating Leverage |
|----------|----------------------|------------------------|
| Revenue +10% | EBIT +12% | EBIT +25% |
| Revenue -10% | EBIT -12% | EBIT -25% |

This is why cost structure analysis is so important for forecasting — it determines how revenue uncertainty translates into profit uncertainty.

> **Key Concept:** Operating leverage is a **double-edged sword**. In good times, fixed costs are spread over more units and margins expand dramatically. In bad times, those same fixed costs cannot be cut quickly, and margins compress sharply. This makes cost structure critical for scenario analysis.

### 5.4 Cost-to-Revenue Ratio Analysis

The simplest and most common approach to cost projection is to express each cost category as a percentage of revenue:

$$\text{Cost Ratio} = \frac{\text{Cost Category}}{\text{Revenue}}$$

For projection, we can:
1. **Hold constant** — assume the ratio stays at the most recent historical level
2. **Trend** — assume the ratio continues its historical trajectory (improving or deteriorating)
3. **Mean-revert** — assume the ratio returns to its long-term average
4. **Target** — set a target ratio based on management guidance or peer comparison

| Cost Category | Typical Approach | Rationale |
|---------------|-----------------|-----------|
| COGS | Trend with mean-reversion | Scale economies vs input cost inflation |
| SG&A (variable) | Hold constant | Scales proportionally with revenue |
| SG&A (fixed) | Grow below revenue growth | Operating leverage benefit |
| R&D | Hold constant or target | Strategic choice, often guided by management |
| D&A | Derived from PP&E schedule | Mechanical, not assumed |

> **Common Mistake:** Projecting COGS as a fixed percentage of revenue ignores operating leverage. If a company has significant fixed manufacturing costs (rent, equipment leases, salaried workers), COGS as a percentage of revenue should *decline* as revenue grows. Decomposing into fixed and variable components captures this effect.

### 5.5 Margin Expansion vs Compression

When projecting costs, we are implicitly projecting **margins**. The analyst should have a clear thesis on margin direction:

**Margin expansion** (costs growing slower than revenue) is driven by:
- Scale economies in production
- Fixed cost leverage (SG&A, R&D)
- Pricing power exceeding input cost inflation
- Operational efficiency improvements

**Margin compression** (costs growing faster than revenue) is driven by:
- Input cost inflation not passed to customers
- Competitive pressure forcing price cuts
- Regulatory compliance costs
- Investment in growth (higher SG&A for new market entry)

> **CFA Exam Tip:** Margin analysis is a staple of CFA exam questions. Be prepared to identify drivers of margin change from financial data and discuss whether observed trends are sustainable.

### The cost structure projection problem

Once revenue is forecasted, the next task is projecting costs. The challenge: not all costs scale with revenue in the same way.

### Fixed vs variable costs — the foundation

* **Variable costs** scale proportionally with revenue (raw materials, sales commissions, freight). The variable cost ratio (VC / Revenue) tends to be stable.
* **Fixed costs** are constant within a relevant range (executive salaries, office rent, R&D budgets). Fixed costs do not increase with revenue in the short run.
* **Semi-variable costs** have both components (utilities have a fixed connection charge plus a usage charge).

The split matters because of **operating leverage** — the magnification effect of fixed costs:

$$\text{DOL} = \frac{\%\Delta\text{Operating Income}}{\%\Delta\text{Revenue}}$$

A company with high fixed costs has high DOL: a 10% revenue increase produces a much larger operating income increase. Conversely, a 10% revenue decrease produces a much larger operating income decrease — making high-DOL companies more cyclical.

### How to identify fixed vs variable costs

Several techniques:

1. **Account analysis:** Examine each cost line item and classify based on its nature. Salaries are fixed, raw materials are variable.

2. **High-low method:** Look at the highest and lowest revenue periods in history. The cost increase per dollar of revenue increase is the variable cost rate.

3. **Regression:** Regress costs on revenue. The slope is the variable rate; the intercept is the fixed component.

4. **Footnote disclosures:** Some companies disclose fixed vs variable cost ratios in earnings calls or investor presentations.

### Cost projection methodology

Our approach for Apex Manufacturing:

| Cost Item | Projection Method | Rationale |
|-----------|-------------------|-----------|
| COGS | % of revenue | Mostly variable (raw materials, direct labour) |
| SG&A | Fixed + variable mix | Mix of executive salaries (fixed) + sales commissions (variable) |
| R&D | % of revenue | Strategic budget that scales with company size |
| Depreciation | Derived from PP&E schedule | Mechanical, not driver-based |

> **Key Concept:** The cost projection should be *defensible* — based on historical patterns, with explicit assumptions about whether margins will expand, contract, or remain stable. A projection that simply holds gross margin constant at the historical average implicitly assumes that input costs and pricing will move in lockstep. This may or may not be realistic.

In [ ]:
# ── Historical cost ratios ──
cogs_pct_hist = cogs_hist / revenue_hist
sga_pct_hist  = sga_hist / revenue_hist
rd_pct_hist   = rd_hist / revenue_hist
da_pct_hist   = da_hist / revenue_hist

print("Historical Cost Ratios (% of Revenue):")
print(f"{'':>6} {'COGS':>8} {'SGA':>8} {'R&D':>8} {'D&A':>8}")
for i in range(3):
    print(f"  Y{i+1}: {cogs_pct_hist[i]*100:>7.2f}% {sga_pct_hist[i]*100:>7.2f}% "
          f"{rd_pct_hist[i]*100:>7.2f}% {da_pct_hist[i]*100:>7.2f}%")

print(f"\n  Avg: {cogs_pct_hist.mean()*100:>7.2f}% {sga_pct_hist.mean()*100:>7.2f}% "
      f"{rd_pct_hist.mean()*100:>7.2f}% {da_pct_hist.mean()*100:>7.2f}%")

# DOL calculation
pct_rev_change = np.diff(revenue_hist) / revenue_hist[:-1]
pct_ebit_change = np.diff(ebit_hist) / ebit_hist[:-1]
dol = pct_ebit_change / pct_rev_change
print(f"\nDegree of Operating Leverage:")
for i in range(len(dol)):
    print(f"  Y{i+1} to Y{i+2}: DOL = {dol[i]:.2f}x")

### Interpretation: Historical Cost Ratios

The historical cost ratio analysis reveals the underlying cost structure of Apex Manufacturing:

- **COGS as % of revenue** — examine whether this ratio is stable, improving, or deteriorating. For an industrial company, a declining COGS ratio suggests scale economies or production efficiency gains. An increasing ratio may signal raw material cost inflation.
- **SG&A as % of revenue** — if this ratio is declining while revenue grows, the company is exhibiting **operating leverage** on its selling and administrative costs. This is a positive signal.
- **R&D as % of revenue** — stability here suggests management maintains a consistent innovation budget. Cuts to R&D as a percentage of revenue may boost short-term profitability but risk long-term competitiveness.
- **D&A as % of revenue** — this ratio reflects the capital intensity of the business. An increasing ratio may indicate the company is investing heavily (higher PP&E base), while a decreasing ratio may suggest under-investment.

> **Key Concept:** The key insight from cost ratio analysis is the **trend direction**. A cost ratio that has been stable for three years is a reasonable basis for projection. A ratio that is trending in one direction requires a thesis: will the trend continue, stabilise, or reverse?

### Reading the historical cost ratios

The historical ratios reveal the company's cost structure:

* **COGS ratio (COGS / Revenue):** Expresses the cost of producing each dollar of revenue. The complement (1 − COGS ratio) is the gross margin. Stability indicates pricing power; trends require explanation.

* **SG&A ratio (SG&A / Revenue):** Shows operational efficiency. A declining ratio means SG&A is growing slower than revenue (operating leverage); a rising ratio means costs are outpacing growth.

* **R&D ratio (R&D / Revenue):** Reflects strategic priorities. A constant ratio suggests proportional reinvestment; a rising ratio indicates increased commitment to innovation; a declining ratio may signal harvesting of past investments.

For Apex, the ratios reveal a company with:
* Stable but slightly compressing gross margin (input cost pressure)
* Stable SG&A ratio (operational consistency)
* Stable R&D ratio (committed innovation budget)

### From ratios to projections

The historical ratios become the projection inputs. Two options for each item:
1. **Hold the ratio constant** at the historical level (simplest, assumes no operational change)
2. **Trend the ratio** in line with historical movement (assumes recent trend continues)

For Apex, the slight gross margin compression suggests that holding COGS at the most recent ratio is reasonable, while trending it down further would assume the compression continues. This is a judgment call that the analyst must make and document.

> **CFA Exam Tip:** When projecting costs, distinguish between *operational* changes (the company is becoming more or less efficient) and *strategic* changes (management is making deliberate decisions to invest more in marketing, R&D, etc.). Operational changes typically continue the historical trend; strategic changes require explicit guidance from management or industry analysis.

In [ ]:
# ── Project costs ──
# COGS: slight improvement (scale economies) — linear decline from 62.5% to 61.0%
cogs_pct_proj = np.linspace(0.625, 0.610, n_proj)

# SG&A: fixed component of ~$100M + variable ~10% of revenue
sga_fixed = 100.0
sga_variable_pct = 0.10
sga_proj = sga_fixed + sga_variable_pct * revenue_proj

# R&D: constant 5% of revenue
rd_pct_proj = 0.05
rd_proj = rd_pct_proj * revenue_proj

cogs_proj = cogs_pct_proj * revenue_proj
gross_profit_proj = revenue_proj - cogs_proj

print("Projected Cost Structure ($M):")
print(f"{'Year':>6} {'Revenue':>10} {'COGS':>10} {'COGS%':>7} {'GP':>10} {'GP%':>7} {'SGA':>10} {'R&D':>10}")
for i in range(n_proj):
    print(f"  Y{years_proj[i]}: {revenue_proj[i]:>9.1f} {cogs_proj[i]:>9.1f} "
          f"{cogs_pct_proj[i]*100:>6.1f}% {gross_profit_proj[i]:>9.1f} "
          f"{gross_profit_proj[i]/revenue_proj[i]*100:>6.1f}% "
          f"{sga_proj[i]:>9.1f} {rd_proj[i]:>9.1f}")

### Interpretation: Projected Costs

The cost projections reflect our assumptions about how each expense category will evolve:

- **COGS** — a slight improvement in COGS-to-revenue ratio reflects expected scale economies as revenue grows. This is a common and defensible assumption for an industrial company with significant fixed production costs.
- **SG&A** — the fixed/variable decomposition explicitly captures operating leverage. Fixed SG&A grows at a rate below revenue growth, causing the SG&A-to-revenue ratio to decline gradually.
- **R&D** — held as a stable percentage of revenue, reflecting management's commitment to innovation spending. This is a conservative assumption.

The **combined effect** of these assumptions is **margin expansion** — gross margin and operating margin should both improve in the projection period. This is the operating leverage thesis: as Apex grows, its fixed costs are spread over a larger revenue base.

> **Common Mistake:** Be careful not to project indefinite margin expansion. Margins are bounded — COGS cannot go to zero, and SG&A cannot decline forever. Always check that projected margins remain within a reasonable range compared to best-in-class peers.

### Validating the cost projection

The projected cost numbers should pass several sanity checks:

1. **Margin progression:** Compute projected gross margin, operating margin, and net margin. Are they in line with history? Is the trajectory (expansion, compression, stability) defensible?

2. **Operating leverage check:** If revenue is projected to grow faster than fixed costs, operating margin should expand. If margin is projected to compress while revenue grows, the implicit assumption is that variable costs are growing faster than revenue — which warrants justification.

3. **Peer comparison:** Compare projected margins to industry peers. Projecting Apex to achieve software-like 30% operating margins would be implausible without a fundamental business model change.

4. **Historical bounds:** Projected margins should generally fall within or near the historical range, unless there's a specific reason to expect a structural change (acquisition, divestiture, major cost program).

### The danger of "hockey stick" projections

A common modelling error is the **hockey stick projection** — flat or declining performance historically, but rapidly improving in the projection period:

```
Historical:  +5%   +5%   +4%   +4%   +3%   +3%
Projection:  +8%   +12%  +15%  +15%  +15%
```

This pattern almost always signals optimism bias rather than analytical rigour. Why would performance suddenly inflect? Specific catalysts must be identified (new product launch, completed cost program, market recovery) — otherwise, the hockey stick should be flattened.

> **Common Mistake:** Models often assume that operational improvements (margin expansion, working capital improvements) "just happen" over time. In reality, every improvement requires specific actions — a cost program, a pricing initiative, a working capital task force. If the model projects improvement, the analyst should be able to articulate the specific drivers.

---
## 6. Working Capital Modeling

Working capital — the difference between current assets and current liabilities — is the lifeblood of operations. To project it, we use **efficiency ratios** derived from historical data.

### 6.1 Key efficiency ratios

**Days Sales Outstanding (DSO)** — how quickly the company collects receivables:
$$\text{DSO} = \frac{\text{Accounts Receivable}}{\text{Revenue}} \times 365$$

**Days Inventory Outstanding (DIO)** — how long inventory sits before being sold:
$$\text{DIO} = \frac{\text{Inventory}}{\text{COGS}} \times 365$$

**Days Payable Outstanding (DPO)** — how long the company takes to pay suppliers:
$$\text{DPO} = \frac{\text{Accounts Payable}}{\text{COGS}} \times 365$$

**Cash Conversion Cycle (CCC):**
$$\text{CCC} = \text{DSO} + \text{DIO} - \text{DPO}$$

The CCC measures how many days it takes to convert a dollar spent on inventory into a dollar collected from customers.

### 6.2 Projection method

To project working capital items, we hold efficiency ratios at their historical averages (or trend them) and solve backwards:

$$\text{AR}_t = \frac{\text{DSO}}{365} \times \text{Revenue}_t$$

$$\text{Inventory}_t = \frac{\text{DIO}}{365} \times \text{COGS}_t$$

$$\text{AP}_t = \frac{\text{DPO}}{365} \times \text{COGS}_t$$

> **Key Concept:** Changes in working capital directly affect operating cash flow. An increase in receivables means the company earned revenue it has not collected — cash flow is *lower* than net income. Conversely, an increase in payables means the company consumed inputs it has not paid for — cash flow is *higher* than net income.

> **CFA Exam Tip:** The cash conversion cycle is a favourite CFA exam topic. A shorter CCC is generally better — it means the company converts inventory to cash faster. But be careful: a very low DPO might mean the company is paying suppliers too quickly, or a very low DIO might indicate stock-out risk.

### 6.3 The Cash Conversion Cycle (CCC)

The CCC measures the time (in days) between paying for raw materials and collecting cash from customers:

$$\text{CCC} = \text{DIO} + \text{DSO} - \text{DPO}$$

| Component | Measures | Decrease Means |
|-----------|----------|---------------|
| **DIO** (Days Inventory Outstanding) | How long inventory sits before sale | Faster inventory turnover |
| **DSO** (Days Sales Outstanding) | How long receivables take to collect | Faster customer payment |
| **DPO** (Days Payable Outstanding) | How long the company takes to pay suppliers | More favourable payment terms |

A **shorter CCC** means the company needs less working capital to support operations, freeing cash for investment or return to shareholders. A **longer CCC** ties up more cash in the operating cycle.

> **Key Concept:** Working capital management is a zero-sum game within the supply chain. Extending your DPO (paying suppliers later) improves *your* CCC but worsens *their* DSO. Analysts should be alert to companies that improve working capital metrics by squeezing suppliers.

### 6.4 Projecting Working Capital from Efficiency Ratios

The standard approach is to project each working capital item using the efficiency ratio:

$$\text{Projected AR} = \frac{\text{DSO}}{365} \times \text{Projected Revenue}$$

$$\text{Projected Inventory} = \frac{\text{DIO}}{365} \times \text{Projected COGS}$$

$$\text{Projected AP} = \frac{\text{DPO}}{365} \times \text{Projected COGS}$$

The assumption for each ratio (DSO, DIO, DPO) is typically:
- **Stable** — carry forward the most recent year's ratio, or the 3-year average
- **Improving** — assume efficiency gains (often guided by management targets)
- **Deteriorating** — assume competitive pressure or growth-related strain

> **CFA Exam Tip:** The CFA exam frequently tests the relationship between working capital changes and cash flow. Remember: an *increase* in a current asset (AR, inventory) is a *use* of cash, while an *increase* in a current liability (AP) is a *source* of cash. This is the working capital adjustment in the indirect cash flow method.

### 6.5 Working Capital and Growth

A critical insight for modelling is that **growing companies typically consume working capital**. As revenue increases, the company needs more inventory to support higher sales, extends more credit to customers (higher AR), and these increases typically outpace the benefit from higher payables.

The **working capital investment** in a given year is:

$$\Delta\text{Net Working Capital} = \Delta\text{AR} + \Delta\text{Inventory} - \Delta\text{AP}$$

This is a direct cash outflow that reduces free cash flow. For fast-growing companies, working capital investment can be the largest single use of cash after capital expenditure.

> **Common Mistake:** Forgetting to model working capital changes is one of the most common errors in financial modelling. A model that projects strong net income growth but ignores the working capital needed to support that growth will *overstate* free cash flow.

### Why working capital deserves its own section

Working capital is the most overlooked source of cash generation (or consumption) in financial modelling. While the income statement focuses on profit, the balance sheet — specifically working capital — determines whether profits convert to cash.

### The working capital projection framework

The standard approach uses the historical efficiency ratios — DSO, DIO, DPO — as projection inputs:

$$\text{Accounts Receivable}_t = \frac{\text{DSO}}{365} \times \text{Revenue}_t$$

$$\text{Inventory}_t = \frac{\text{DIO}}{365} \times \text{COGS}_t$$

$$\text{Accounts Payable}_t = \frac{\text{DPO}}{365} \times \text{COGS}_t$$

This approach has two key properties:
1. **Working capital scales with the business:** As revenue (and COGS) grow, AR, inventory, and AP grow proportionally. This is realistic — a larger business needs proportionally more working capital.
2. **Efficiency assumptions are explicit:** By using DSO, DIO, DPO directly, the analyst is making a specific statement about how efficiently the company manages each component.

### Connection to the cash conversion cycle

The CCC (covered in the Balance Sheet & Working Capital notebook) ties these three ratios together:

$$\text{CCC} = \text{DIO} + \text{DSO} - \text{DPO}$$

In the projection, the CCC tells you how many days of operating costs are tied up in working capital. A 50-day CCC on a \$1B revenue base means roughly \$137M of working capital investment ($1B × 50/365). When revenue grows by 10%, working capital needs grow by approximately 10% as well — a real cash drain that the income statement does not show.

> **Key Concept:** The working capital projection is the bridge between the income statement and the cash flow statement. Without it, projecting cash flow is impossible. The income statement tells you what the company will *earn*; the working capital ratios tell you how much of those earnings will *convert to cash*.

### Common projection scenarios

Analysts typically model working capital under three regimes:

1. **Constant efficiency:** Hold DSO, DIO, DPO at recent levels. Working capital scales with revenue. The most common assumption.

2. **Improving efficiency:** Project DSO and DIO to decline over time (faster collections, leaner inventory). DPO may extend (negotiate longer terms with suppliers). Reduces working capital investment, boosting FCF.

3. **Deteriorating efficiency:** Project DSO and DIO to increase (slower collections, inventory buildup). May reflect competitive weakness, demand softening, or aging product lines. Increases working capital investment, reducing FCF.

The choice depends on the business outlook. A company implementing a working capital improvement program may justify Scenario 2; a company facing demand challenges may warrant Scenario 3.

In [ ]:
# ── Compute historical efficiency ratios ──
dso_hist = ar_hist / revenue_hist * 365
dio_hist = inventory_hist / cogs_hist * 365
dpo_hist = ap_hist / cogs_hist * 365
ccc_hist = dso_hist + dio_hist - dpo_hist

print("Historical Efficiency Ratios (days):")
print(f"{'':>6} {'DSO':>8} {'DIO':>8} {'DPO':>8} {'CCC':>8}")
for i in range(3):
    print(f"  Y{i+1}: {dso_hist[i]:>7.1f} {dio_hist[i]:>7.1f} {dpo_hist[i]:>7.1f} {ccc_hist[i]:>7.1f}")
print(f"  Avg: {dso_hist.mean():>7.1f} {dio_hist.mean():>7.1f} {dpo_hist.mean():>7.1f} {ccc_hist.mean():>7.1f}")

# Use average ratios for projection
dso_proj = dso_hist.mean()
dio_proj = dio_hist.mean()
dpo_proj = dpo_hist.mean()

# Project working capital items
ar_proj  = dso_proj / 365 * revenue_proj
inv_proj = dio_proj / 365 * cogs_proj
ap_proj  = dpo_proj / 365 * cogs_proj

print(f"\nProjected Working Capital ($M):")
print(f"{'Year':>6} {'AR':>10} {'Inventory':>10} {'AP':>10} {'NWC':>10}")
for i in range(n_proj):
    nwc = ar_proj[i] + inv_proj[i] - ap_proj[i]
    print(f"  Y{years_proj[i]}: {ar_proj[i]:>9.1f} {inv_proj[i]:>9.1f} {ap_proj[i]:>9.1f} {nwc:>9.1f}")

### Interpretation: Historical Efficiency Ratios

The computed efficiency ratios reveal how Apex manages its working capital:

- **DSO** indicates how many days, on average, it takes to collect receivables. Compare to industry benchmarks — industrial companies typically have DSOs of 40-60 days. A rising DSO may indicate deteriorating credit quality of customers or loosening credit terms to boost sales.
- **DIO** measures inventory management efficiency. Higher DIO means more capital tied up in inventory. For a manufacturer, DIO depends on production cycle time and inventory management practices (just-in-time vs buffer stock).
- **DPO** reflects the company's payment practices with suppliers. Higher DPO conserves cash but may strain supplier relationships. A DPO significantly above industry norms could indicate the company is stretching payables to manage cash flow.

The **trends** in these ratios are as important as their levels. Improving efficiency (lower DSO, lower DIO, higher DPO) is positive for cash flow; deteriorating efficiency consumes cash.

> **Key Concept:** The efficiency ratios form the basis of our working capital projections. By assuming specific DSO, DIO, and DPO levels for the forecast period, we mechanically derive the projected AR, inventory, and AP balances — which in turn determine working capital changes on the cash flow statement.

### Reading the efficiency ratios

The historical efficiency ratios for Apex Manufacturing reveal:

* **DSO (Days Sales Outstanding):** How many days, on average, does it take to collect from customers? An industrial manufacturer typically has DSOs of 50-70 days (longer than retail, shorter than construction). Track the trend — rising DSO may indicate aging receivables or weakening customer credit quality.

* **DIO (Days Inventory Outstanding):** How many days does inventory sit before being sold? Industrial manufacturers often have 60-100 day DIOs (much higher than perishable retail, lower than aged spirits). Rising DIO may indicate slowing demand or product obsolescence.

* **DPO (Days Payable Outstanding):** How long does the company take to pay suppliers? Typical range is 30-60 days. Rising DPO may indicate working capital optimisation (good) or cash strain (bad) — the interpretation depends on context.

### Computing the cash conversion cycle

For Apex, the CCC computation reveals how much working capital investment is required to support its operations. A CCC of approximately 50-70 days means that for every \$1 of revenue, the company has roughly 50-70 cents of revenue locked in working capital at any moment.

This is real cash that cannot be invested elsewhere — and it must grow as the company grows. Modelling this explicitly is essential to producing realistic free cash flow projections.

> **CFA Exam Tip:** A common exam scenario presents two companies with identical income statements but different working capital efficiency. The company with the lower CCC will generate more free cash flow despite having identical accounting profits. The exam tests whether you understand that *operating profit ≠ cash flow* and that working capital efficiency is the primary explanation for the gap.

In [ ]:
# ── Visualise the Cash Conversion Cycle ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: historical trends
x = np.arange(3)
width = 0.25
ax1.bar(x - width, dso_hist, width, label='DSO', color=PRIMARY)
ax1.bar(x, dio_hist, width, label='DIO', color=SECONDARY)
ax1.bar(x + width, dpo_hist, width, label='DPO', color=TERTIARY)
ax1.plot(x, ccc_hist, 'D-', color=ACCENT, markersize=8, linewidth=2, label='CCC')
ax1.set_xticks(x)
ax1.set_xticklabels(labels_hist)
ax1.set_ylabel('Days')
ax1.set_title('Historical Efficiency Ratios')
ax1.legend()

# Right: projected NWC
nwc_proj = ar_proj + inv_proj - ap_proj
nwc_hist_vals = ar_hist + inventory_hist - ap_hist
all_nwc = np.concatenate([nwc_hist_vals, nwc_proj])
all_x = np.concatenate([years_hist, years_proj])
colours = [PRIMARY]*3 + [SECONDARY]*5
ax2.bar(all_x, all_nwc, color=colours, edgecolor='white', linewidth=0.5)
ax2.axvline(x=3.5, color='gray', linestyle=':', alpha=0.7)
ax2.set_xticks(all_x)
ax2.set_xticklabels([f'Y{y}' for y in all_x])
ax2.set_ylabel('Net Working Capital ($M)')
ax2.set_title('Historical and Projected NWC')

plt.tight_layout()
plt.show()

### Interpretation: Cash Conversion Cycle Visualisation

The charts above illustrate Apex's working capital efficiency over the historical period:

- The **individual components** (DSO, DIO, DPO) show whether improvements in the CCC are broad-based or concentrated in one area.
- The **overall CCC** trend reveals whether the company is becoming more or less efficient at converting its operating cycle into cash.
- A declining CCC is generally positive — it means the company needs less working capital per dollar of revenue, freeing cash for other purposes.
- However, examine *how* the CCC improved. A decline driven by stretching payables (higher DPO) is less sustainable than one driven by genuine operational improvements (lower DIO).

For our projections, we assume the efficiency ratios remain close to their recent levels, with modest improvements reflecting management's working capital optimisation initiatives.

> **CFA Exam Tip:** On the CFA exam, you may be asked to assess the quality of a company's working capital improvement. Always decompose the CCC into its three components to determine whether the improvement is sustainable or merely a timing shift.

### Visualising the CCC over time

The CCC visualisation reveals whether working capital efficiency is improving, deteriorating, or stable. Three-year history is too short to draw definitive conclusions, but the direction of change is informative:

* **Improving CCC:** Each year's bar is shorter than the previous. The company is collecting faster, holding less inventory, or paying suppliers slower (or some combination). This frees cash for other uses.

* **Deteriorating CCC:** Each year's bar is longer. Working capital is consuming more cash relative to revenue. May indicate competitive pressure (longer customer terms), supply chain issues (inventory buildup), or supplier renegotiation.

* **Stable CCC:** Working capital efficiency is consistent. The company's working capital investment scales linearly with revenue.

### Implications for projection

If the historical CCC is stable, projecting at the most recent level is defensible. If it's trending in either direction, the analyst must decide whether to:
1. **Extrapolate the trend** (assume the change continues) — most aggressive
2. **Hold at the most recent level** (assume the trend stabilises) — most common
3. **Revert to a long-run mean** (assume the recent change is temporary) — most conservative

Each choice has different implications for projected free cash flow. A company with deteriorating CCC will look more cash-flow-positive under option 3 (mean reversion) than under option 1 (extrapolation).

> **Key Concept:** Working capital management is one of the few areas where a company has direct control over cash flow without affecting reported profits. A CFO can launch a "DSO reduction initiative" or an "inventory rationalisation program" and meaningfully boost free cash flow within 6-12 months. The model should reflect such initiatives if they are credible — but the analyst must be sceptical of management promises about working capital improvements that haven't yet materialised.

---
## 7. Capital Expenditure and Depreciation

### 7.1 Maintenance vs growth capex

Capital expenditure can be decomposed into two components:

$$\text{Total Capex} = \text{Maintenance Capex} + \text{Growth Capex}$$

- **Maintenance capex** replaces worn-out assets to sustain current capacity. A common proxy is depreciation expense: $\text{Maintenance Capex} \approx \text{D\&A}$.
- **Growth capex** expands capacity. It depends on the company's strategic plans and revenue growth.

### 7.2 Capex-to-depreciation ratio

The ratio of Capex to D&A is a useful diagnostic:

| Ratio | Interpretation |
|-------|---------------|
| Approximately 1.0 | Company is spending just enough to maintain its asset base |
| Greater than 1.5 | Significant growth investment — capacity is expanding |
| Less than 1.0 | Under-investment — the asset base is shrinking |

### 7.3 Depreciation modelling

For a simple model, depreciation is estimated as:

$$\text{D\&A}_t = \text{Depreciation Rate} \times \frac{\text{Net PP\&E}_{t-1} + \text{Net PP\&E}_t}{2}$$

### 7.4 PP&E roll-forward

$$\text{Net PP\&E}_t = \text{Net PP\&E}_{t-1} + \text{Capex}_t - \text{D\&A}_t$$

> **Key Concept:** Capex and depreciation form a feedback loop with the balance sheet. Capex increases PP&E (investing cash flow), while depreciation decreases PP&E (a non-cash charge that reduces net income but is added back in operating cash flow). Understanding this loop is essential for model integrity.

> **Common Mistake:** Do not confuse depreciation (an accounting allocation of past capex) with current capex. A company can have high depreciation (from past investments) but low current capex (under-investing now). The capex-to-D&A ratio catches this.

### 7.3 Capex-to-Depreciation Ratio

A key metric for assessing capital investment intensity is the **capex-to-depreciation ratio**:

$$\text{Capex/D\&A Ratio} = \frac{\text{Capital Expenditure}}{\text{Depreciation \& Amortisation}}$$

| Ratio Value | Interpretation |
|-------------|---------------|
| < 1.0 | Company is **under-investing** — not replacing assets as they depreciate. PP&E base is shrinking. |
| ≈ 1.0 | Company is **maintaining** its asset base — replacing depreciated assets but not expanding. |
| > 1.0 | Company is **growing** its asset base — investing beyond replacement. PP&E base is expanding. |
| > 2.0 | **Aggressive expansion** — significant growth capex. Common during capacity build-outs. |

> **Key Concept:** For a growing company, the capex/D&A ratio should be consistently above 1.0. If it falls below 1.0 for an extended period, the company's productive capacity is shrinking — which may eventually constrain revenue growth.

### 7.4 Depreciation Schedule Mechanics

Depreciation is driven by the PP&E balance and the assumed useful life of assets:

$$\text{PP\&E}_{t} = \text{PP\&E}_{t-1} + \text{Capex}_{t} - \text{Depreciation}_{t}$$

Rearranging:

$$\text{Depreciation}_{t} = \text{PP\&E}_{t-1} + \text{Capex}_{t} - \text{PP\&E}_{t}$$

In practice, depreciation is computed as:

$$\text{Depreciation}_{t} = \frac{\text{Gross PP\&E}_{t}}{\text{Weighted Average Useful Life}}$$

For simplicity, we model depreciation as a percentage of the average PP&E balance, calibrated to historical depreciation rates.

### 7.5 Maintenance Capex Estimation

Estimating maintenance capex — the minimum investment needed to sustain current operations — is important for understanding **sustainable free cash flow**:

$$\text{Maintenance Capex} \approx \text{Depreciation Expense}$$

$$\text{Growth Capex} = \text{Total Capex} - \text{Maintenance Capex}$$

> **Common Mistake:** Using total capex in a free cash flow calculation without distinguishing maintenance from growth capex can understate the company's sustainable cash-generating ability. For valuation purposes, maintenance capex (not total capex) should be used to compute **normalised FCF**.

### CapEx and depreciation — the asset lifecycle

Capital expenditure and depreciation are linked through the asset lifecycle:
1. **Capex:** Cash spent today to acquire long-lived assets (machinery, buildings, technology)
2. **Depreciation:** Allocation of asset cost across its useful life (non-cash expense)
3. **Eventually:** Asset is fully depreciated and (typically) replaced — a new round of capex begins

In a steady-state business, capex and depreciation are roughly equal — the company is just maintaining capacity. Above that, capex represents growth investment.

### Maintenance vs growth capex

A critical analytical distinction:

$$\text{Total CapEx} = \text{Maintenance CapEx} + \text{Growth CapEx}$$

* **Maintenance CapEx:** Required to maintain current productive capacity. Generally tracks depreciation (since depreciation is the rate at which assets are "used up").
* **Growth CapEx:** Discretionary investment to expand capacity, enter new markets, or modernise.

Why does this matter? Because **free cash flow** should ideally subtract only maintenance capex — growth capex creates future earnings and should be evaluated separately:

$$\text{Owner Earnings} = \text{Net Income} + \text{Depreciation} - \text{Maintenance CapEx}$$

(Warren Buffett's famous formulation)

Companies rarely disclose the maintenance/growth split, so analysts use heuristics:
* Maintenance CapEx ≈ Depreciation (most common)
* Maintenance CapEx ≈ Industry-average CapEx/Revenue ratio
* Maintenance CapEx ≈ Long-run CapEx/Revenue ratio for the company

### The CapEx-to-Revenue ratio

A simple but useful projection input:

$$\text{CapEx Intensity} = \frac{\text{CapEx}}{\text{Revenue}}$$

For capital-intensive industries (utilities, telecom, oil & gas), this ratio runs 10-25%. For asset-light industries (software, services), it can be 1-5%. Stability of this ratio over time indicates a steady business model; volatility may indicate lumpy investment or strategic shifts.

> **Key Concept:** CapEx is one of the largest cash outflows for capital-intensive companies — often exceeding net income. A small change in CapEx assumptions can dramatically affect projected free cash flow. The analyst must justify the chosen capex trajectory based on historical patterns, management guidance, and the implied capacity expansion.

In [ ]:
# ── Historical capex analysis ──
capex_to_da = capex_hist / da_hist

print("Historical Capex Analysis ($M):")
print(f"{'':>6} {'Capex':>8} {'D&A':>8} {'Capex/DA':>10} {'Net PP&E':>10}")
for i in range(3):
    print(f"  Y{i+1}: {capex_hist[i]:>7.1f} {da_hist[i]:>7.1f} {capex_to_da[i]:>9.2f}x {ppe_hist[i]:>9.1f}")

# ── Project capex and depreciation ──
# Capex = maintenance (1.0x D&A) + growth (linked to revenue growth)
# We target a capex/D&A ratio of ~1.3x (moderate growth phase)
capex_da_target = 1.30
depr_rate = 0.085  # ~8.5% of average net PP&E

# Iterative: D&A depends on PP&E, PP&E depends on capex, capex depends on D&A
ppe_proj = np.zeros(n_proj)
da_proj = np.zeros(n_proj)
capex_proj = np.zeros(n_proj)

ppe_prev = ppe_hist[-1]
for t in range(n_proj):
    # Estimate D&A from previous PP&E
    da_est = depr_rate * ppe_prev
    # Capex = target ratio * D&A
    capex_est = capex_da_target * da_est
    # New PP&E
    ppe_new = ppe_prev + capex_est - da_est
    # Refine D&A using average PP&E
    da_refined = depr_rate * (ppe_prev + ppe_new) / 2
    capex_refined = capex_da_target * da_refined
    ppe_final = ppe_prev + capex_refined - da_refined

    da_proj[t] = da_refined
    capex_proj[t] = capex_refined
    ppe_proj[t] = ppe_final
    ppe_prev = ppe_final

print(f"\nProjected Capex & Depreciation ($M):")
print(f"{'Year':>6} {'Capex':>10} {'D&A':>10} {'Capex/DA':>10} {'Net PP&E':>10}")
for i in range(n_proj):
    print(f"  Y{years_proj[i]}: {capex_proj[i]:>9.1f} {da_proj[i]:>9.1f} "
          f"{capex_proj[i]/da_proj[i]:>9.2f}x {ppe_proj[i]:>9.1f}")

### Interpretation: Capital Expenditure Analysis

The capex analysis for Apex Manufacturing reveals the company's investment trajectory:

- The **capex-to-D&A ratio** above 1.0 confirms that Apex is a net investor — it spends more on capex than its assets depreciate, meaning the PP&E base is growing.
- The **capex-to-revenue ratio** indicates the capital intensity of the business. For an industrial company, capex/revenue of 4-6% is typical. Significantly higher suggests expansion; significantly lower suggests under-investment.
- The **implied growth capex** (total capex minus depreciation) represents the incremental investment in productive capacity. This should be directionally consistent with the revenue growth we project.
- The **depreciation rate** (D&A / average PP&E) indicates the average useful life assumption embedded in the financials. A rate of 8-10% implies a 10-12 year average useful life, reasonable for industrial equipment.

For the projection period, we model capex as a percentage of revenue, with the ratio calibrated to maintain a capex/D&A ratio consistent with Apex's growth plans. Depreciation is modelled as a percentage of the prior-year PP&E balance, which mechanically updates as new capex is added.

> **Key Concept:** Capex and depreciation are the link between the income statement and the balance sheet for fixed assets. Capex increases PP&E (balance sheet) and eventually flows through as depreciation expense (income statement). Getting this relationship right is essential for model integrity.

### Reading the CapEx history

For Apex Manufacturing, the historical CapEx data reveals the investment intensity of the business. Key observations:

* **CapEx-to-Revenue ratio:** Stable around 5-7% for industrial manufacturers — consistent with the industry's capital intensity profile.
* **CapEx vs Depreciation:** If CapEx exceeds Depreciation consistently, the company is *growing* its asset base (real expansion). If they are roughly equal, the company is in steady-state. If CapEx is below Depreciation, the company is shrinking its productive capacity.

For a healthy growing company like Apex, CapEx > Depreciation is normal — the excess represents growth investment.

### Projecting CapEx and Depreciation

The standard approach:

1. **CapEx:** Project as a percentage of revenue, using the historical average. Modify if there are known major investments (new factory, capacity expansion).

2. **Depreciation:** Derive from the PP&E schedule:
   * Beginning PP&E + CapEx = Gross PP&E available
   * Gross PP&E × Average depreciation rate = Annual Depreciation
   * Net PP&E = Beginning PP&E + CapEx − Depreciation

3. **Reconciliation:** Verify that the implied useful life (Average PP&E / Annual Depreciation) is consistent with the asset mix.

### The depreciation-to-revenue ratio

A consistency check: in steady-state, depreciation as a percentage of revenue should equal CapEx as a percentage of revenue (both reflecting the same capital intensity). If your projected ratios diverge significantly, the model implies the company is structurally expanding or contracting its asset base — which should match the strategic story.

> **CFA Exam Tip:** The CFA exam tests the relationship between CapEx, depreciation, and free cash flow. Remember: FCFF = CFO − CapEx (with CFO including the depreciation add-back). The depreciation cancels out — what matters for FCFF is the difference between operating cash generation and reinvestment in productive assets.

---
## 8. Income Statement Projection

With revenue, costs, and D&A projected, we can build the complete projected income statement. Note that interest expense will initially use a **placeholder** estimate — we will resolve the circular reference in Section 11.

> **Key Concept:** The income statement projection flows top-down: Revenue minus COGS gives gross profit, minus operating expenses gives EBIT, minus interest gives EBT, minus tax gives net income. Each line connects to assumptions we have already established.

### 8.1 The Assumption Hierarchy

Each line of the projected income statement is derived from specific assumptions, creating a clear hierarchy:

| Line Item | Projection Method | Key Assumption |
|-----------|------------------|----------------|
| **Revenue** | Growth rate / regression / segment buildup | Growth rate trajectory |
| **COGS** | % of revenue (with trend) | Gross margin assumption |
| **Gross Profit** | Revenue − COGS | *Derived* |
| **SG&A** | Fixed + variable % of revenue | Operating leverage assumption |
| **R&D** | % of revenue | Innovation intensity |
| **D&A** | % of prior-year PP&E | Asset useful life |
| **EBIT** | Gross Profit − SG&A − R&D − D&A | *Derived* |
| **Interest Expense** | Interest rate × average debt | Cost of debt, leverage |
| **EBT** | EBIT − Interest Expense | *Derived* |
| **Tax** | Effective tax rate × EBT | Tax rate assumption |
| **Net Income** | EBT − Tax | *Derived* |

Note that several line items are **derived** (computed mechanically from other items) rather than assumed. The model has approximately 6-8 independent assumptions; everything else follows.

> **Key Concept:** A well-structured income statement projection clearly separates **assumptions** (inputs you choose) from **derivations** (outputs computed from assumptions). This makes it easy to trace any output back to the assumptions that drive it — essential for scenario analysis and model auditing.

> **CFA Exam Tip:** Interest expense creates a **circular reference** with the balance sheet — it depends on debt levels, which depend on cash flow, which depends on net income, which depends on interest expense. We use a placeholder in this initial projection and resolve the circularity in Section 11.

### Building the projected income statement

With revenue, costs, and depreciation projected, we can assemble the complete projected income statement. The construction is mechanical, but each step encodes specific assumptions:

| Line Item | Source | Assumption Embedded |
|-----------|--------|---------------------|
| Revenue | Forecasted directly | Growth rate methodology (covered above) |
| COGS | % of revenue | Stable or trending gross margin |
| Gross Profit | Revenue − COGS | Derived |
| SG&A | Fixed + variable | Operational efficiency |
| R&D | % of revenue | Strategic budget commitment |
| Operating Income | GP − OpEx | Derived |
| Interest Expense | Debt × Rate | Capital structure assumption |
| Pre-tax Income | OI − Int | Derived |
| Tax Expense | Effective rate × PTI | Tax strategy stability |
| Net Income | PTI − Tax | The bottom line |

### The articulation property

The projected income statement is *internally consistent* with the assumptions:
* If revenue grows X%, COGS grows X% (under fixed margin assumption)
* Gross profit grows X%
* If fixed OpEx is constant, operating income grows faster than X% (operating leverage)
* Interest expense depends on projected debt (which depends on cash flow — circularity!)

This articulation is the strength of integrated modelling — but also its primary challenge. A change in one assumption ripples through the entire system, often in unexpected ways.

> **Key Concept:** The projected income statement is *not* the final answer. It depends on assumptions about debt levels (for interest expense) that themselves depend on cash flows (which depend on the income statement). This circularity is resolved iteratively, as we'll see in Section 11.

In [ ]:
# ── Build projected income statement ──
# Interest expense placeholder: use Year 3 rate on average total debt
# (Will be refined in circular reference section)
total_debt_hist_y3 = st_debt_hist[-1] + lt_debt_hist[-1]
avg_interest_rate = interest_hist[-1] / total_debt_hist_y3

# For initial projection, assume debt declines linearly
st_debt_proj_init = np.maximum(st_debt_hist[-1] - 5 * np.arange(1, n_proj + 1), 10)
lt_debt_proj_init = np.maximum(lt_debt_hist[-1] - 30 * np.arange(1, n_proj + 1), 200)
total_debt_proj_init = st_debt_proj_init + lt_debt_proj_init

interest_proj_init = avg_interest_rate * total_debt_proj_init

ebit_proj = gross_profit_proj - sga_proj - rd_proj - da_proj
ebt_proj_init = ebit_proj - interest_proj_init
tax_proj_init = np.maximum(ebt_proj_init * tax_rate, 0)
net_income_proj_init = ebt_proj_init - tax_proj_init

print("=== Projected Income Statement (Initial, $M) ===")
print(f"{'':>12} " + " ".join(f"{'Y'+str(y):>10}" for y in years_proj))
print("-" * 70)

rows = [
    ('Revenue', revenue_proj),
    ('COGS', -cogs_proj),
    ('Gross Profit', gross_profit_proj),
    ('SGA', -sga_proj),
    ('R&D', -rd_proj),
    ('D&A', -da_proj),
    ('EBIT', ebit_proj),
    ('Interest', -interest_proj_init),
    ('EBT', ebt_proj_init),
    ('Tax', -tax_proj_init),
    ('Net Income', net_income_proj_init),
]

for name, vals in rows:
    line = f"  {name:<12}" + " ".join(f"{v:>10.1f}" for v in vals)
    print(line)

# Margins
print(f"\n{'GP Margin':<14}" + " ".join(f"{gross_profit_proj[i]/revenue_proj[i]*100:>9.1f}%" for i in range(n_proj)))
print(f"{'EBIT Margin':<14}" + " ".join(f"{ebit_proj[i]/revenue_proj[i]*100:>9.1f}%" for i in range(n_proj)))
print(f"{'Net Margin':<14}" + " ".join(f"{net_income_proj_init[i]/revenue_proj[i]*100:>9.1f}%" for i in range(n_proj)))

### Interpretation: Projected Income Statement

The projected income statement shows the expected profitability trajectory for Apex over the forecast horizon:

- **Revenue growth** follows our selected forecast method. Check that the projected growth rates are reasonable relative to historical performance and industry expectations.
- **Gross margin** should be compared to historical levels. If we project margin expansion, verify that the underlying assumptions (scale economies, pricing power) are defensible.
- **Operating margin** captures the combined effect of gross margin and operating leverage (SG&A, R&D, D&A as percentages of revenue). This is the key profitability metric for the operating business.
- **Interest expense** uses a placeholder estimate at this stage. The final interest expense will be higher or lower depending on the resolved debt levels in Section 11.
- **Net income** is the bottom line that flows to retained earnings on the balance sheet. Its growth rate relative to revenue growth indicates whether profitability leverage is positive or negative.
- **Effective tax rate** should remain approximately constant unless there is a specific reason to change it (tax reform, geographic mix shift).

Verify the following relationships hold:
- Gross Profit = Revenue − COGS
- EBIT = Gross Profit − SG&A − R&D − D&A
- EBT = EBIT − Interest Expense
- Net Income = EBT × (1 − Tax Rate)

> **Common Mistake:** Projecting each income statement line independently without verifying that they are mutually consistent. For example, if revenue grows 8% but COGS grows 5%, the implicit assumption is a 300+ basis point gross margin improvement — which may not be realistic.

### Reading the projected income statement

The projected income statement reveals the implied trajectory of profitability. Key checks:

1. **Margin trajectory:** Are projected margins consistent with history? Expansion requires explanation; compression requires acknowledgment of competitive pressure.

2. **Earnings growth vs revenue growth:** If earnings grow faster than revenue (positive operating leverage), the implicit story is that fixed costs are diluting as the business scales. The opposite (negative operating leverage) suggests cost growth outpacing revenue.

3. **Bottom-line projection:** Does projected net income compound to a reasonable level by Year 5? Companies rarely sustain >20% net income growth for extended periods — if your model implies this, scrutinise the assumptions.

4. **Tax expense reasonableness:** Project tax as a stable effective rate. Wild swings in projected tax expense suggest a calculation error or an aggressive tax planning assumption.

### Common projection errors

Several errors recur in beginner models:

* **Compounding optimism:** Each year's growth rate is plausible alone, but compounded over 5 years they imply unrealistic terminal values.
* **Margin expansion without cause:** Projecting steady margin expansion without identifying the source (cost program, mix shift, scale benefits).
* **Ignoring competitive response:** A model that assumes the company gains share without competitors responding is unrealistic in efficient industries.
* **One-time items as recurring:** Including last year's one-time gain in the recurring forecast inflates all subsequent years.

> **Common Mistake:** The projected income statement should not be the *output* of the model — it should be the *input* to the rest of the analysis. Once you have a defensible income statement projection, the balance sheet and cash flow projections follow mechanically. Spending time on income statement assumptions is the highest-leverage activity in financial modelling.

---
## 9. Balance Sheet Projection

The balance sheet is the most complex of the three statements to project because it must **balance** — total assets must equal total liabilities plus equity in every projected year. This requires a **plug variable** that absorbs any surplus or shortfall.

### 9.1 The balancing mechanism

After projecting all individual line items:

$$\text{Total Assets} = \text{Cash} + \text{AR} + \text{Inventory} + \text{PP\&E}$$
$$\text{Total L+E} = \text{AP} + \text{ST Debt} + \text{LT Debt} + \text{Equity}$$

If Total Assets exceeds Total L+E, the company needs more financing (debt or equity).
If Total Assets is less than Total L+E, the company has excess cash.

### 9.2 Cash as the plug

The most common approach for a profitable, growing company is to use **cash** as the plug:

$$\text{Cash}_t = \text{Total L+E}_t - \text{AR}_t - \text{Inventory}_t - \text{PP\&E}_t$$

If the plug is negative (the company needs more cash than it has), we switch to a **debt revolver** as the plug.

### 9.3 Equity roll-forward

Retained earnings (and therefore equity) builds through:

$$\text{Equity}_t = \text{Equity}_{t-1} + \text{Net Income}_t - \text{Dividends}_t$$

We assume a **payout ratio** (dividends as a percentage of net income) based on historical behaviour.

> **Key Concept:** The balance sheet plug is the mechanism that ensures internal consistency. Without it, the model will have a "gap" between assets and liabilities that grows larger each year. Professional models always include a plug — usually cash (for surplus) or a revolver (for deficit).

> **Common Mistake:** A common error is to project cash independently (e.g., "cash grows 5% per year") rather than letting it be determined by the model. Cash is the *residual* — what is left after all operating, investing, and financing activities. Projecting it independently breaks the model's internal logic.

### 9.3 Why the Balance Sheet Must Balance

The fundamental accounting identity is not a suggestion — it is a **mathematical requirement**:

$$\text{Assets} = \text{Liabilities} + \text{Shareholders' Equity}$$

This identity holds because of **double-entry bookkeeping**: every transaction affects at least two accounts, and the effects always balance. In our model:

- **Revenue recognition** increases AR (asset) and retained earnings (equity) by the same amount
- **Capex** increases PP&E (asset) and decreases cash (asset) by the same amount
- **Debt issuance** increases cash (asset) and debt (liability) by the same amount
- **Dividend payment** decreases cash (asset) and retained earnings (equity) by the same amount

### 9.4 The Plug Variable

After projecting all line items, the balance sheet will almost certainly **not balance** — total assets will differ from total liabilities plus equity. The **plug variable** absorbs the difference:

- If total assets > total liabilities + equity (cash shortfall): **increase debt** to fund the gap
- If total assets < total liabilities + equity (cash surplus): **increase cash** or **reduce debt**

The choice of plug variable is a modelling decision:

| Plug Variable | When to Use | Implication |
|---------------|------------|-------------|
| **Cash (excess cash)** | Company has surplus cash flow | Debt is fixed; excess cash accumulates |
| **Revolver / Short-term debt** | Company has funding needs | Cash is set to a minimum; debt fluctuates |
| **Combination** | Most realistic | Cash has a minimum floor; excess goes to debt repayment; shortfall increases borrowing |

> **Key Concept:** The plug variable is what makes the model **self-balancing**. Without it, you would need to manually adjust entries until the balance sheet balances — which is what Excel modellers did before revolving credit line plugs became standard practice.

> **Common Mistake:** Forgetting to link retained earnings to the income statement is the most common cause of an unbalanced balance sheet. Retained earnings must equal prior retained earnings plus net income minus dividends — this is the critical link between the IS and BS.

### The balance sheet projection challenge

The balance sheet projection has a unique challenge: the accounting equation must hold *exactly* in every projected period:

$$\text{Assets}_t = \text{Liabilities}_t + \text{Equity}_t$$

If you project each line item independently, the equation will not balance — there will be a "plug" of unexplained value. The two standard solutions:

### The balancing mechanism — debt or cash as plug

Method 1: **Debt as plug** (most common)
1. Project all asset items based on operational drivers (working capital ratios, capex)
2. Project equity from beginning equity + projected net income − dividends
3. Project most liabilities (AP, accrued expenses) from operational drivers
4. **Debt is the plug:** Debt = Total Assets − Equity − Other Liabilities

This approach assumes the company will borrow whatever is needed to fund operations and investment. If the result is negative, the company has surplus cash — which converts to a cash plug.

Method 2: **Cash as plug** (alternative)
1. Project all asset items *except cash* based on drivers
2. Project all liabilities and equity directly
3. **Cash is the plug:** Cash = Total Liab + Equity − Non-cash Assets

This approach assumes a fixed debt level. If projections imply more cash than the balance sheet supports, the excess accumulates as cash.

### The integrated solution

A sophisticated model uses *both* mechanisms:
* **Cash flow generated → accumulates as cash** (up to a target level)
* **Cash above target → reduces debt** (if debt exists)
* **Cash flow consumed → drains existing cash first**
* **Cash exhausted → revolver debt is drawn**

This ensures realistic capital structure dynamics. The model should reflect management's actual capital allocation policy, not arbitrary plugs.

> **Key Concept:** The balancing mechanism is not just a computational trick — it represents real economic decisions. When the model needs more cash to balance, real-world management would have to either: (a) borrow money, (b) issue equity, (c) sell assets, or (d) cut dividends. The plug should match management's likely action.

> **CFA Exam Tip:** Modelers must be explicit about which item plugs the balance sheet. The convention varies by industry: for financials, debt is usually the plug; for technology, cash often serves; for utilities, both may rotate. Documenting the choice and its implications is part of professional model construction.

In [ ]:
# ── Project balance sheet (initial, before circular resolution) ──
# Dividend payout ratio from historical
dividends_pct = dividends_hist / net_income_hist
avg_payout = dividends_pct.mean()
print(f"Historical payout ratios: {dividends_pct}")
print(f"Average payout ratio: {avg_payout:.4f} ({avg_payout*100:.2f}%)")

# Project equity
dividends_proj_init = avg_payout * net_income_proj_init
equity_proj_init = np.zeros(n_proj)
eq_prev = equity_hist[-1]
for t in range(n_proj):
    equity_proj_init[t] = eq_prev + net_income_proj_init[t] - dividends_proj_init[t]
    eq_prev = equity_proj_init[t]

# Non-cash assets
non_cash_assets_proj = ar_proj + inv_proj + ppe_proj

# Liabilities (excluding cash-side plug)
total_liab_proj_init = ap_proj + st_debt_proj_init + lt_debt_proj_init

# Cash as plug: Cash = Equity + Total Liab - Non-cash assets
cash_proj_init = equity_proj_init + total_liab_proj_init - non_cash_assets_proj

# Total assets
total_assets_proj_init = cash_proj_init + ar_proj + inv_proj + ppe_proj
total_le_proj_init = total_liab_proj_init + equity_proj_init

print(f"\n=== Projected Balance Sheet (Initial, $M) ===")
print(f"{'':>18} " + " ".join(f"{'Y'+str(y):>10}" for y in years_proj))
print("-" * 75)

bs_rows = [
    ('Cash (plug)', cash_proj_init),
    ('Accts Receivable', ar_proj),
    ('Inventory', inv_proj),
    ('Net PP&E', ppe_proj),
    ('Total Assets', total_assets_proj_init),
    ('---', None),
    ('Accts Payable', ap_proj),
    ('ST Debt', st_debt_proj_init),
    ('LT Debt', lt_debt_proj_init),
    ('Total Liabilities', total_liab_proj_init),
    ('Equity', equity_proj_init),
    ('Total L+E', total_le_proj_init),
]

for name, vals in bs_rows:
    if vals is None:
        print(f"  {'---'*20}")
        continue
    line = f"  {name:<18}" + " ".join(f"{v:>10.1f}" for v in vals)
    print(line)

# Balance check
print(f"\n  Balance Check (A - L - E):")
for i in range(n_proj):
    diff = total_assets_proj_init[i] - total_le_proj_init[i]
    check = 'PASS' if abs(diff) < ATOL else 'FAIL'
    print(f"    Y{years_proj[i]}: {diff:.6f} [{check}]")

### Interpretation: Projected Balance Sheet (Initial)

The initial balance sheet projection — before circular reference resolution — provides a first approximation:

- **Cash** is derived as the plug variable or set to a policy minimum. The initial value may change significantly after we resolve the circular reference.
- **AR and Inventory** are mechanically derived from revenue, COGS, and the efficiency ratios (DSO, DIO). They should grow roughly in line with revenue and COGS respectively.
- **PP&E** follows the capex and depreciation schedule: PP&E = Prior PP&E + Capex − D&A.
- **AP** is derived from COGS and DPO, similar to AR and inventory.
- **Debt** is initially held constant or follows a simple schedule. After circular resolution, it will adjust based on actual cash flow needs.
- **Equity** = Prior Equity + Net Income − Dividends. This link to the income statement is essential.

**Check the balance:** Verify that Total Assets = Total Liabilities + Equity. If not, identify the source of the imbalance. At this stage, a small imbalance is expected because we have not yet resolved the interest expense circular reference.

> **CFA Exam Tip:** On the CFA exam, balance sheet projection questions often test whether you understand the *direction* of each item's change. An increase in revenue should increase AR (asset) and retained earnings (equity), keeping the balance sheet balanced. Always trace the full chain of effects.

### Reading the initial projected balance sheet

The "initial" balance sheet projection is computed *before* resolving the circular reference between debt and interest expense. It uses placeholder values that will be refined through iteration.

Several items deserve attention:

* **Current assets:** Cash, receivables, inventory all scale with revenue (working capital ratios applied to projected revenue and COGS).

* **PP&E:** Builds up via the formula: Net PP&E_t = Net PP&E_{t-1} + CapEx_t − Depreciation_t. The growth in PP&E reflects the company's capacity expansion.

* **Total liabilities and equity:** Equity grows by retained earnings (NI − Dividends). The plug determines whether the company needs more debt or accumulates excess cash.

### What "initial" means

Before circular resolution, the projection has an internal inconsistency: interest expense was projected from prior-year debt, but the new debt (the plug) hasn't been factored in. So the actual interest expense in the projection year should be slightly higher (or lower) than what was used in the income statement.

The next step is to iterate: feed the new debt level back into interest expense, recompute net income, recompute the plug, repeat until convergence. This is covered in Section 11.

> **Common Mistake:** Beginners sometimes ignore the circular reference and accept the "initial" balance sheet as the answer. The error is small for low-leverage companies but can become significant for highly leveraged companies where interest expense is a major item. Always iterate to convergence for professional-quality models.

---
## 10. Cash Flow Statement Projection

The projected cash flow statement is **derived** from the projected income statement and balance sheet — it is not independently forecasted. This ensures internal consistency.

### 10.1 Indirect method structure

$$\text{CFO} = \text{Net Income} + \text{D\&A} - \Delta\text{AR} - \Delta\text{Inventory} + \Delta\text{AP}$$

$$\text{CFI} = -\text{Capex}$$

$$\text{CFF} = \Delta\text{Debt} - \text{Dividends}$$

$$\Delta\text{Cash} = \text{CFO} + \text{CFI} + \text{CFF}$$

> **Key Concept:** The cash flow statement is a *check* on the model — if it does not reconcile to the change in cash on the balance sheet, there is an error somewhere. This is one of the most powerful model integrity tests.

### 10.2 Deriving Cash Flow from IS and BS

The key insight is that the cash flow statement is **entirely derived** — it contains no independent assumptions. Every line comes from either the income statement or the change in a balance sheet account:

**Operating Cash Flow (Indirect Method):**

$$\text{CFO} = \underbrace{\text{Net Income}}_{\text{from IS}} + \underbrace{\text{D\&A}}_{\text{non-cash add-back}} - \underbrace{\Delta\text{AR}}_{\text{from BS}} - \underbrace{\Delta\text{Inventory}}_{\text{from BS}} + \underbrace{\Delta\text{AP}}_{\text{from BS}}$$

**Investing Cash Flow:**

$$\text{CFI} = -\text{Capex}$$

**Financing Cash Flow:**

$$\text{CFF} = \underbrace{\Delta\text{Debt}}_{\text{from BS}} - \underbrace{\text{Dividends}}_{\text{from IS/BS}}$$

**Free Cash Flow:**

$$\text{FCF} = \text{CFO} + \text{CFI} = \text{CFO} - \text{Capex}$$

### 10.3 Cash Flow Verification

The ultimate integrity check for the cash flow statement is:

$$\text{Cash}_{t} = \text{Cash}_{t-1} + \text{CFO}_t + \text{CFI}_t + \text{CFF}_t$$

If the ending cash balance derived from cash flows does not match the cash balance on the projected balance sheet, there is an error in the model. This reconciliation is performed automatically in our integrity checks (Section 14).

> **Key Concept:** Because the cash flow statement is derived from the IS and BS, it serves as a powerful **cross-check**. If the three statements are internally consistent, the cash flow reconciliation will hold automatically. Any failure points to a modelling error.

> **CFA Exam Tip:** The indirect method of computing CFO starts with net income and adjusts for non-cash items and working capital changes. The direct method starts with cash collected from customers and subtracts cash paid. Both yield the same CFO — the indirect method is more common in models because it explicitly shows the linkages to the IS and BS.

### Deriving the projected cash flow statement

The cash flow statement is *derived* from the projected income statement and projected balance sheet. This is the indirect method covered in the Cash Flow Statement Analysis notebook, applied prospectively:

**CFO components:**
1. Start with projected Net Income
2. Add projected Depreciation (non-cash)
3. Subtract projected increases in current assets (AR, inventory)
4. Add projected increases in current liabilities (AP, accrued expenses)

**CFI components:**
1. Subtract projected CapEx
2. Add proceeds from any projected divestitures (typically zero unless modeled)

**CFF components:**
1. Add net new borrowing (or subtract net repayment)
2. Subtract projected dividends
3. Add proceeds from equity issuance (or subtract buybacks)

**Net change in cash:**

$$\Delta\text{Cash} = \text{CFO} + \text{CFI} + \text{CFF}$$

### The integrity check

The projected ending cash on the balance sheet must equal:

$$\text{Cash}_t = \text{Cash}_{t-1} + \Delta\text{Cash from CFS}$$

If this fails, the model has an error. The most common cause is a mismatch between balance sheet changes used in the cash flow statement and those reflected on the balance sheet itself.

> **Key Concept:** Building a self-balancing 3-statement model requires the cash flow statement to be *derived* rather than *forecasted independently*. The cash flow statement is the algebraic consequence of the income statement and balance sheet projections. If you forecast it separately, you almost guarantee inconsistency.

### Free cash flow as the model's payoff

The ultimate output of the model is projected free cash flow:

$$\text{FCF} = \text{CFO} - \text{CapEx}$$
$$\text{FCFF} = \text{CFO} + \text{Int}(1-t) - \text{CapEx}$$
$$\text{FCFE} = \text{FCFF} - \text{Int}(1-t) + \text{Net Borrowing}$$

These cash flows feed directly into DCF valuation models. The financial model converts operating assumptions into the cash flows that determine intrinsic value.

In [ ]:
# ── Project cash flow statement (initial) ──
# Changes in working capital
ar_all = np.concatenate([[ar_hist[-1]], ar_proj])
inv_all = np.concatenate([[inventory_hist[-1]], inv_proj])
ap_all = np.concatenate([[ap_hist[-1]], ap_proj])

delta_ar_proj = np.diff(ar_all)
delta_inv_proj = np.diff(inv_all)
delta_ap_proj = np.diff(ap_all)

# CFO
cfo_proj_init = (net_income_proj_init + da_proj
                 - delta_ar_proj - delta_inv_proj + delta_ap_proj)

# CFI
cfi_proj = -capex_proj

# CFF
st_all = np.concatenate([[st_debt_hist[-1]], st_debt_proj_init])
lt_all = np.concatenate([[lt_debt_hist[-1]], lt_debt_proj_init])
delta_st_proj = np.diff(st_all)
delta_lt_proj = np.diff(lt_all)
cff_proj_init = delta_st_proj + delta_lt_proj - dividends_proj_init

# Net change
net_change_proj_init = cfo_proj_init + cfi_proj + cff_proj_init

print("=== Projected Cash Flow Statement (Initial, $M) ===")
print(f"{'':>22} " + " ".join(f"{'Y'+str(y):>10}" for y in years_proj))
print("-" * 78)

cf_rows = [
    ('Net Income', net_income_proj_init),
    ('+ D&A', da_proj),
    ('- Incr. AR', -delta_ar_proj),
    ('- Incr. Inventory', -delta_inv_proj),
    ('+ Incr. AP', delta_ap_proj),
    ('Cash from Ops', cfo_proj_init),
    ('Capital Expend.', -capex_proj),
    ('Cash from Invest', cfi_proj),
    ('Chg ST Debt', delta_st_proj),
    ('Chg LT Debt', delta_lt_proj),
    ('Dividends', -dividends_proj_init),
    ('Cash from Finance', cff_proj_init),
    ('Net Cash Change', net_change_proj_init),
]

for name, vals in cf_rows:
    line = f"  {name:<22}" + " ".join(f"{v:>10.1f}" for v in vals)
    print(line)

# Verify reconciliation
cash_all_init = np.concatenate([[cash_hist[-1]], cash_proj_init])
actual_change = np.diff(cash_all_init)
print(f"\n  Cash Reconciliation:")
for i in range(n_proj):
    print(f"    Y{years_proj[i]}: Computed change = {net_change_proj_init[i]:.1f}, "
          f"Actual change = {actual_change[i]:.1f}, "
          f"Diff = {net_change_proj_init[i] - actual_change[i]:.6f}")

### Interpretation: Projected Cash Flow Statement (Initial)

The initial cash flow projection provides a first look at Apex's cash generation capacity:

- **CFO** should be positive and growing for a healthy company. Compare CFO to net income — if CFO exceeds net income, earnings quality is high (non-cash charges like D&A are adding back).
- **Working capital changes** represent the cash cost of growth. If Apex is growing revenue, it needs more AR and inventory, which consumes cash. The magnitude of this drag depends on the efficiency ratios.
- **Capex** is the primary investing outflow. Compare to CFO — if CFO comfortably exceeds capex, the company generates positive free cash flow.
- **FCF (Free Cash Flow)** = CFO − Capex. This is the cash available to service debt, pay dividends, and accumulate as cash on the balance sheet.
- **CFF** captures debt changes and dividends. If FCF is insufficient to cover dividends, the company must borrow (increase debt) or draw down cash.

The cash flow reconciliation should approximately hold:

$$\text{Ending Cash} \approx \text{Beginning Cash} + \text{CFO} + \text{CFI} + \text{CFF}$$

Any discrepancy at this stage is due to the placeholder interest expense. After circular resolution, the reconciliation will be exact.

> **Common Mistake:** Forgetting that capex is a *negative* cash flow. In the investing section, capex appears as a negative number (cash outflow). When computing FCF = CFO + CFI, the signs are already correct. When computing FCF = CFO − Capex, capex is entered as a positive number.

### Reading the projected cash flow statement

The projected cash flow statement reveals whether the projected operations actually convert to cash. Several diagnostic questions:

1. **Is CFO > Net Income?** A healthy growing business with significant depreciation should typically have CFO > NI. Apex Manufacturing's depreciation add-back should exceed the working capital investment.

2. **Is FCF positive throughout?** If projected FCF is negative in any year, the company will need external financing (debt or equity) to fund operations. The model should reflect this in the financing section.

3. **Does FCF grow or shrink?** Even if positive, FCF that grows slower than revenue indicates declining capital efficiency. FCF growth is the ultimate measure of value creation.

### The investment intensity story

A useful summary metric: **CapEx-to-Revenue ratio**. If projected at 6% (similar to history), and revenue grows 8%, then CapEx grows 8% as well — capacity is expanding linearly with revenue.

Compare this to operating cash flow growth. If CFO grows at 12% (operating leverage compounding), the gap between CFO growth and CapEx growth — 12% − 8% = 4% — represents accelerating free cash flow expansion. This is the financial model's way of capturing operating leverage.

> **CFA Exam Tip:** When projecting cash flows, always check the relationship between Net Income, CFO, and FCF. A model where these three diverge significantly without explanation is suspect. The relationships should be: Net Income → CFO via depreciation add-back and working capital adjustments → FCF via CapEx subtraction. Each step has a specific economic meaning.

---
## 11. Circular Reference Resolution

### 11.1 The circularity problem

In an integrated model, there is a fundamental **circular dependency**:

1. **Interest expense** depends on the level of **debt** (and cash).
2. **Debt** (or the cash plug) depends on **cash flow**, which determines whether the company has a surplus or deficit.
3. **Cash flow** depends on **net income**, which depends on **interest expense**.

$$\text{Interest} \rightarrow \text{Net Income} \rightarrow \text{Cash Flow} \rightarrow \text{Debt/Cash} \rightarrow \text{Interest}$$

This is a genuine mathematical circularity — each variable depends on the others.

### 11.2 Resolution methods

**Method 1: Iterative convergence** (what we implement)

Start with an initial guess for interest expense, compute the full model, derive the implied debt/cash levels, recompute interest expense, and repeat until convergence:

$$\text{Interest}^{(k+1)} = r \times \frac{\text{Debt}^{(k)}_{t-1} + \text{Debt}^{(k)}_t}{2}$$

This converges because the feedback loop is **contractive** — a one-dollar increase in interest expense reduces net income by $0.75 (after tax), which changes cash by a similar amount, which changes debt/cash, which changes interest by only a fraction of a dollar.

**Method 2: Algebraic solution**

For simple models, you can solve the system of equations simultaneously. However, this becomes unwieldy as the model grows.

**Method 3: Previous-period debt**

Use the *beginning-of-period* debt balance (which is known) to compute interest, breaking the circularity. This is simpler but less accurate.

> **Key Concept:** The circular reference is not a bug — it reflects economic reality. Interest expense genuinely depends on debt levels, which depend on cash flows, which depend on interest expense. A model that ignores this circularity will have a systematic bias.

> **CFA Exam Tip:** The CFA curriculum notes that circular references in financial models should be resolved through iteration. In Excel, this means enabling iterative calculations. In our Python model, we implement this explicitly with a convergence loop.

> **Common Mistake:** Some modellers "break" the circularity by hard-coding interest expense rather than linking it to debt. While this avoids the iteration, it means the model's interest expense is inconsistent with its debt levels — a violation of internal consistency.

### 11.3 Mathematical Formulation

The circular reference can be expressed as a **fixed-point equation**. Define $x$ as the interest expense. Then:

1. Net Income $= (\text{EBIT} - x)(1 - t)$ where $t$ is the tax rate
2. Retained Earnings $= \text{Prior RE} + \text{Net Income} - \text{Dividends}$
3. Equity $= \text{Prior Equity} + \text{Retained Earnings Change}$
4. Cash flow and the plug determine Debt (or Cash)
5. Interest $= r \times \text{Average Debt}$ — which gives us a *new* estimate of $x$

We seek $x^*$ such that:

$$x^* = f(x^*)$$

This is a **fixed-point problem**. The function $f$ maps an interest expense guess to the implied interest expense after flowing through the entire model. The solution $x^*$ is the fixed point where the model is self-consistent.

### 11.4 Convergence Guarantee

The fixed-point iteration converges because the **feedback loop is contractive**. Consider a \$1 increase in interest expense:

1. Net income decreases by $\$1 \times (1-t) = \$0.75$ (assuming 25% tax rate)
2. Cash flow decreases by \$0.75 (less cash available)
3. Debt increases by \$0.75 (need to borrow more)
4. Interest expense increases by $\$0.75 \times r \approx \$0.04$ (assuming 5% interest rate)

The feedback factor is approximately $(1-t) \times r \approx 0.04$, which is much less than 1. This means the iteration converges rapidly — typically within 3-5 iterations to machine precision.

### 11.5 Excel vs Python Approach

In **Excel**, circular references are resolved by enabling iterative calculation (File → Options → Formulas → Enable iterative calculation). Excel then internally performs fixed-point iteration, usually converging in 100 iterations with a tolerance of 0.001.

In **Python**, we implement the iteration explicitly, which gives us full control over convergence criteria and the ability to monitor the convergence process. Our approach:

```
for each projected year:
    guess interest_expense = prior year interest
    repeat:
        compute IS → BS → CF → new debt/cash → new interest
        if |new interest - old interest| < tolerance:
            break
        update interest = new interest
```

> **Key Concept:** The circular reference is not a bug — it is a fundamental feature of integrated financial models. It arises because interest expense (an IS item) depends on debt (a BS item) which depends on cash flow (derived from IS and BS). Resolving it ensures the model is **internally self-consistent**.

> **CFA Exam Tip:** While the CFA exam is unlikely to ask you to solve a circular reference numerically, it may test your understanding of *why* the circularity exists and how it affects model outputs. The key insight is that interest expense and debt are simultaneously determined — you cannot know one without knowing the other.

### 11.6 Impact of Circular Resolution

The difference between the initial (placeholder) and converged (resolved) results can be material, especially for:

- **Highly leveraged companies** — more debt means higher interest, stronger feedback loop
- **Low-margin companies** — interest expense is a larger fraction of EBIT, so changes matter more
- **Companies with variable-rate debt** — interest rate changes amplify the effect

For Apex, with moderate leverage and healthy margins, the impact is typically small (a few million dollars of interest expense difference). But for a leveraged buyout target with 6× debt/EBITDA, the resolution can change net income by 10% or more.

> **Common Mistake:** Some modellers "break" the circular reference by hardcoding the interest rate and never resolving the circularity. This produces a model that appears to work but is not internally consistent — the debt level may not actually support the assumed interest expense. Always resolve the circularity.

### The circular reference problem

A 3-statement model contains an inherent circularity:

```
Net Income depends on Interest Expense
Interest Expense depends on Debt
Debt depends on the balance sheet plug
Plug depends on Cash
Cash depends on the Cash Flow Statement
Cash Flow Statement depends on Net Income
                         ↑
                         └── back to where we started
```

This isn't a modelling error — it reflects a real economic relationship. A company's debt level depends on its cash flow, but its cash flow depends on its interest expense, which depends on debt level. The variables are simultaneously determined.

### The two solution approaches

**Approach 1: Iterative calculation (Excel)**
* Set the model in iterative mode (File → Options → Formulas → Enable iterative calculation)
* Excel iterates the formulas until convergence
* Risk: convergence may fail or be unstable; results may oscillate

**Approach 2: Explicit iteration loop (Python/programming)**
* Initialize debt and interest expense at prior-year values
* Solve the entire model with these values
* Update debt based on the resulting plug
* Recompute interest expense from new debt
* Repeat until debt level stabilises (convergence threshold: typically 0.1% change or less)

Our model uses Approach 2 because it is more transparent and reliable.

### Convergence properties

The iteration is mathematically a fixed-point iteration. For typical companies, convergence is rapid (3-5 iterations) because the feedback effect is small — interest expense is usually a small fraction of net income.

For highly leveraged companies (interest expense > 50% of operating income), convergence can be slower and the result more sensitive to input assumptions. In extreme cases (interest > operating income), the iteration may diverge — indicating the company is unable to service its debt.

> **Key Concept:** The circular reference is a *feature*, not a bug. It represents the genuine simultaneity of corporate financial decisions: cash flow determines debt capacity, but interest cost affects cash flow. A model that ignores this circularity (e.g., by using only beginning-period debt) will produce slightly inaccurate results, but the magnitude depends on how leveraged the company is.

> **Common Mistake:** Some modellers avoid the circular reference by hard-coding interest expense as a fixed number rather than calculating it from debt. This works for simple models but breaks down when debt levels change significantly across the projection period — exactly the cases where the model is most useful.

In [ ]:
def build_integrated_model(revenue, cogs_pct, sga_fixed, sga_var_pct, rd_pct,
                           depr_rate, capex_da_ratio, tax_rate, avg_int_rate,
                           payout_ratio, dso, dio, dpo,
                           st_debt_schedule, lt_debt_schedule,
                           hist_data, n_years=5, max_iter=100, tol=1e-6):
    '''Build a fully integrated three-statement model with circular reference resolution.

    Parameters: revenue, cogs_pct, sga_fixed, sga_var_pct, rd_pct, depr_rate,
    capex_da_ratio, tax_rate, avg_int_rate, payout_ratio, dso, dio, dpo,
    st_debt_schedule, lt_debt_schedule, hist_data, n_years, max_iter, tol.

    Returns: dict with all projected financial statement arrays.'''
    n = n_years

    # Unpack historical base-year values
    ppe_base = hist_data['ppe']
    equity_base = hist_data['equity']
    ar_base = hist_data['ar']
    inv_base = hist_data['inventory']
    ap_base = hist_data['ap']
    cash_base = hist_data['cash']
    st_debt_base = hist_data['st_debt']
    lt_debt_base = hist_data['lt_debt']

    # ── Items that do not depend on circular reference ──
    cogs = cogs_pct * revenue
    gross_profit = revenue - cogs
    sga = sga_fixed + sga_var_pct * revenue
    rd = rd_pct * revenue

    # Working capital
    ar = dso / 365 * revenue
    inv = dio / 365 * cogs
    ap = dpo / 365 * cogs

    # PP&E and D&A (iterative within themselves)
    ppe = np.zeros(n)
    da = np.zeros(n)
    capex = np.zeros(n)
    ppe_prev_val = ppe_base
    for t in range(n):
        da_est = depr_rate * ppe_prev_val
        capex_est = capex_da_ratio * da_est
        ppe_new = ppe_prev_val + capex_est - da_est
        da_ref = depr_rate * (ppe_prev_val + ppe_new) / 2
        capex_ref = capex_da_ratio * da_ref
        ppe[t] = ppe_prev_val + capex_ref - da_ref
        da[t] = da_ref
        capex[t] = capex_ref
        ppe_prev_val = ppe[t]

    ebit = gross_profit - sga - rd - da

    # ── Circular reference iteration ──
    # Initial guess: interest = 0
    interest = np.zeros(n)

    for iteration in range(max_iter):
        interest_old = interest.copy()

        # Income statement
        ebt = ebit - interest
        tax = np.maximum(ebt * tax_rate, 0)
        net_income = ebt - tax
        dividends = payout_ratio * net_income

        # Equity roll-forward
        equity = np.zeros(n)
        eq_prev = equity_base
        for t in range(n):
            equity[t] = eq_prev + net_income[t] - dividends[t]
            eq_prev = equity[t]

        # Balance sheet: cash as plug
        non_cash = ar + inv + ppe
        total_liab = ap + st_debt_schedule + lt_debt_schedule
        cash = equity + total_liab - non_cash

        # Recompute interest on average debt
        total_debt = st_debt_schedule + lt_debt_schedule
        total_debt_prev = np.concatenate([[st_debt_base + lt_debt_base], total_debt[:-1]])
        avg_debt = (total_debt_prev + total_debt) / 2

        # Interest income on cash (at a lower rate, e.g., 1/3 of debt rate)
        cash_prev = np.concatenate([[cash_base], cash[:-1]])
        avg_cash = (cash_prev + cash) / 2
        interest_on_debt = avg_int_rate * avg_debt
        interest_income = (avg_int_rate / 3) * np.maximum(avg_cash, 0)
        interest = interest_on_debt - interest_income  # Net interest expense

        # Check convergence
        if np.max(np.abs(interest - interest_old)) < tol:
            break

    # Cash flow statement
    delta_ar = np.diff(np.concatenate([[ar_base], ar]))
    delta_inv = np.diff(np.concatenate([[inv_base], inv]))
    delta_ap = np.diff(np.concatenate([[ap_base], ap]))

    cfo = net_income + da - delta_ar - delta_inv + delta_ap
    cfi = -capex
    delta_st = np.diff(np.concatenate([[st_debt_base], st_debt_schedule]))
    delta_lt = np.diff(np.concatenate([[lt_debt_base], lt_debt_schedule]))
    cff = delta_st + delta_lt - dividends
    net_cash_change = cfo + cfi + cff

    total_assets = cash + ar + inv + ppe
    total_le = total_liab + equity

    return {
        'revenue': revenue, 'cogs': cogs, 'gross_profit': gross_profit,
        'sga': sga, 'rd': rd, 'da': da, 'ebit': ebit,
        'interest': interest, 'ebt': ebt, 'tax': tax,
        'net_income': net_income, 'dividends': dividends,
        'cash': cash, 'ar': ar, 'inventory': inv, 'ppe': ppe,
        'total_assets': total_assets,
        'ap': ap, 'st_debt': st_debt_schedule, 'lt_debt': lt_debt_schedule,
        'total_liab': total_liab, 'equity': equity, 'total_le': total_le,
        'capex': capex,
        'cfo': cfo, 'cfi': cfi, 'cff': cff, 'net_cash_change': net_cash_change,
        'delta_ar': delta_ar, 'delta_inv': delta_inv, 'delta_ap': delta_ap,
        'iterations': iteration + 1,
    }

print("Integrated model function defined.")

### Interpretation: The Integrated Model Function

The `build_integrated_model` function is the heart of our notebook — it encapsulates the entire three-statement model with circular reference resolution in a single callable function. Key design choices:

- **Parameterised assumptions** — every assumption (growth, margins, efficiency ratios, capex, tax, interest rate, payout ratio) is passed as a parameter, making the function suitable for scenario analysis and Monte Carlo simulation.
- **Year-by-year iteration** — the model builds each projection year sequentially, using the prior year's balance sheet as a starting point.
- **Convergence loop** — within each year, the interest expense circular reference is resolved iteratively. The convergence criterion (tolerance) ensures accuracy without excessive computation.
- **Complete output** — the function returns all three statements plus derived metrics (FCF, key ratios), providing everything needed for analysis.

The function's structure mirrors the manual model-building process:
1. Project revenue and costs → income statement
2. Project working capital and PP&E → balance sheet
3. Derive cash flows → cash flow statement
4. Resolve interest circular reference → iterate until convergence
5. Compute the cash/debt plug to balance the balance sheet

> **Key Concept:** By encapsulating the model in a function, we can run it thousands of times with different inputs — this is what enables the scenario analysis (Section 12) and Monte Carlo simulation (Section 13) that follow. A well-structured model function is the foundation of rigorous financial analysis.

### Reading the integrated model function

The `build_integrated_model` function is the heart of the financial model. It takes assumptions as inputs and produces complete projected statements as outputs.

**Function inputs (the "drivers"):**
* Revenue assumption (forecast trajectory)
* Cost ratios (COGS%, SG&A breakdown, R&D%)
* Working capital efficiency (DSO, DIO, DPO)
* CapEx intensity (CapEx as % of revenue)
* Tax rate, dividend policy, interest rate
* Initial balance sheet (starting point)

**Function logic:**
1. Initialise all output arrays
2. For each projection year:
   a. Project income statement using current debt assumption
   b. Project working capital from revenue/COGS using ratios
   c. Project PP&E from beginning + capex − depreciation
   d. Project equity from beginning + NI − dividends
   e. Compute the plug (revolver debt or excess cash)
   f. Iterate to convergence (resolve circularity)
3. Return complete projected statements

**Function outputs:**
* Projected income statement (revenue, all costs, NI)
* Projected balance sheet (all asset/liability/equity items)
* Projected cash flow statement (CFO, CFI, CFF, ending cash)

### The power of parameterisation

By structuring the model as a function, scenario analysis becomes trivial: just call the function with different inputs. Want to see what happens with revenue growth at 5% instead of 8%? Pass a different revenue array. Want to test a working capital improvement? Pass lower DSO and DIO assumptions.

This parameterisation is what enables:
* **Sensitivity analysis** (vary one input at a time)
* **Scenario analysis** (change multiple inputs in coherent groups)
* **Monte Carlo simulation** (vary all inputs randomly across realistic distributions)

> **CFA Exam Tip:** Real-world financial models in Excel are typically structured similarly — with a clear "assumptions" section (the inputs), formulas that derive everything else (the function), and outputs that present the results. The Python implementation here is conceptually identical but more rigorous because it forces explicit handling of the circular reference.

In [ ]:
# ── Run the integrated model (base case) ──
hist_data = {
    'ppe': ppe_hist[-1], 'equity': equity_hist[-1],
    'ar': ar_hist[-1], 'inventory': inventory_hist[-1],
    'ap': ap_hist[-1], 'cash': cash_hist[-1],
    'st_debt': st_debt_hist[-1], 'lt_debt': lt_debt_hist[-1],
}

base = build_integrated_model(
    revenue=revenue_proj,
    cogs_pct=cogs_pct_proj,
    sga_fixed=sga_fixed, sga_var_pct=sga_variable_pct,
    rd_pct=rd_pct_proj,
    depr_rate=depr_rate, capex_da_ratio=capex_da_target,
    tax_rate=tax_rate, avg_int_rate=avg_interest_rate,
    payout_ratio=avg_payout,
    dso=dso_proj, dio=dio_proj, dpo=dpo_proj,
    st_debt_schedule=st_debt_proj_init,
    lt_debt_schedule=lt_debt_proj_init,
    hist_data=hist_data,
)

print(f"Circular reference resolved in {base['iterations']} iterations.\n")

# Print final income statement
print("=== FINAL Projected Income Statement ($M) ===")
print(f"{'':>14} " + " ".join(f"{'Y'+str(y):>10}" for y in years_proj))
print("-" * 70)
for name, key in [('Revenue','revenue'), ('COGS','cogs'), ('Gross Profit','gross_profit'),
                   ('SGA','sga'), ('R&D','rd'), ('D&A','da'), ('EBIT','ebit'),
                   ('Interest','interest'), ('EBT','ebt'), ('Tax','tax'),
                   ('Net Income','net_income')]:
    sign = -1 if key in ('cogs','sga','rd','da','interest','tax') else 1
    vals = base[key] * sign
    print(f"  {name:<14}" + " ".join(f"{v:>10.1f}" for v in vals))

### Interpretation: Base Case Results

The base case results represent our best estimate of Apex's financial trajectory. Key metrics to examine:

- **Revenue trajectory** — verify the projected growth rates match our assumptions and produce revenue levels consistent with the company's market position.
- **Margin progression** — check whether operating and net margins are expanding, stable, or compressing. The trend should be consistent with our cost structure assumptions.
- **Interest expense convergence** — the resolved interest expense should differ from the initial placeholder. A large difference indicates high leverage sensitivity; a small difference confirms moderate leverage.
- **Cash/debt plug** — examine whether the model projects cash accumulation (financial strength) or increasing debt (funding needs). This depends on whether FCF exceeds or falls short of dividends and debt repayments.
- **FCF generation** — positive and growing FCF is the hallmark of a healthy company. Compare FCF to net income — a ratio near 1.0 or above indicates strong cash conversion.

> **CFA Exam Tip:** When analysing model outputs, always compute the **key valuation multiples** implied by the projections: P/E, EV/EBITDA, FCF yield. If these imply unrealistic valuations, your assumptions may need revisiting.

### Reading the base case results

The base case projection represents the "expected" trajectory given a single set of assumptions. Several diagnostic questions:

1. **Does revenue grow at the assumed rate?** Mechanical check — verify that the model is implementing the inputs correctly.

2. **Does net income compound to a reasonable level?** Compare 5-year terminal NI to current NI. A 1.5x to 2x range is typical for moderate-growth companies.

3. **Does the balance sheet balance every year?** Total assets must equal total liabilities plus equity. If not, the model has a bug.

4. **Is the projected debt level reasonable?** If debt is rising rapidly, the model implies that cash flow is insufficient to fund growth — a credit concern.

5. **Is FCF positive and growing?** The whole point of the model is to project FCF. Negative or declining FCF is a red flag.

### Interpreting projected free cash flow

FCF is the most important output of the model. It represents the cash that *could* be returned to capital providers (after maintaining the business). It feeds directly into:

* **DCF valuation:** Discount projected FCF at WACC to compute enterprise value
* **Credit analysis:** Compare FCF to debt service to assess coverage
* **Capital allocation:** Determine how much can be returned via dividends/buybacks vs reinvested

A projected FCF trajectory of \$X → \$Y over 5 years implies a CAGR of (Y/X)^(1/5) − 1. Compare this to revenue CAGR — if FCF grows faster, the company is becoming more capital-efficient; if slower, the opposite.

> **Common Mistake:** Beginners sometimes interpret the base case as "the prediction." It is not. The base case is one *possible* future given a coherent set of assumptions. The real value of the model is in scenario analysis — exploring how outcomes change as assumptions vary. The base case is the reference point, not the answer.

In [ ]:
# Print final balance sheet
print("=== FINAL Projected Balance Sheet ($M) ===")
print(f"{'':>18} " + " ".join(f"{'Y'+str(y):>10}" for y in years_proj))
print("-" * 75)
for name, key in [('Cash','cash'), ('Accts Receivable','ar'), ('Inventory','inventory'),
                   ('Net PP&E','ppe'), ('Total Assets','total_assets'),
                   ('---', None),
                   ('Accts Payable','ap'), ('ST Debt','st_debt'), ('LT Debt','lt_debt'),
                   ('Total Liabilities','total_liab'), ('Equity','equity'),
                   ('Total L+E','total_le')]:
    if key is None:
        print(f"  {'---'*20}")
        continue
    print(f"  {name:<18}" + " ".join(f"{v:>10.1f}" for v in base[key]))

print(f"\n  Balance Check (A - L - E):")
for i in range(n_proj):
    diff = base['total_assets'][i] - base['total_le'][i]
    check = 'PASS' if abs(diff) < ATOL else 'FAIL'
    print(f"    Y{years_proj[i]}: {diff:.10f} [{check}]")

### Interpretation: Final Projected Balance Sheet

The converged balance sheet — after circular reference resolution — is the definitive projection:

- **Verify the balance:** Total Assets must equal Total Liabilities + Equity in every year. Any imbalance indicates a modelling error.
- **Cash trend** — is cash growing (positive FCF after debt service and dividends) or declining (FCF insufficient to cover obligations)?
- **Debt trend** — is the company deleveraging (paying down debt from excess cash flow) or levering up (borrowing to fund growth or dividends)?
- **Equity growth** — retained earnings accumulation drives equity growth. If the company pays out most of its earnings as dividends, equity growth will be slow.
- **Asset composition** — as the company grows, the mix between current assets (AR, inventory) and fixed assets (PP&E) may shift. For a capital-intensive industrial company, PP&E typically dominates.

> **Common Mistake:** Checking only that the balance sheet balances in the first projection year. It must balance in *every* year. A model that balances in Year 4 but not Year 5 has a subtle error — often related to how the debt/cash plug is updated year-to-year.

### Reading the converged balance sheet

After resolving the circular reference, the final projected balance sheet shows the company's financial position at each future year-end. Key checks:

1. **Asset growth:** Compare terminal year assets to current year assets. The growth rate should be consistent with revenue growth (assets typically grow with revenue) or slightly faster (working capital and PP&E both expand).

2. **Capital structure evolution:** Track Debt/Equity ratio across the projection. Is leverage rising, falling, or stable? Compare to historical levels and industry norms.

3. **Cash trajectory:** Cash should grow if FCF exceeds debt repayment and dividends. A growing cash pile may be deployed strategically (M&A, buybacks) — or may be a source of investor concern (capital inefficiency).

4. **Retained earnings build:** Each year's retained earnings should equal beginning RE + NI − Dividends. Verify mechanically.

### The "balance sheet integrity" check

Total assets must equal total liabilities + equity, exactly. The model should display this check explicitly — a balance sheet that doesn't balance is a critical error that propagates through every subsequent calculation.

The check is simple:
* Compute Total Assets = sum of all asset line items
* Compute Total L+E = sum of all liability line items + total equity
* Difference should be zero (or within rounding tolerance, e.g., < \$0.01M)

If the check fails, debug by isolating which line items aren't articulating correctly. The most common culprits are working capital adjustments and the debt plug.

> **Key Concept:** A balanced balance sheet is necessary but not sufficient for model correctness. A model can balance while still having errors in the income statement projection, working capital ratios, or capex schedule. The balance sheet check confirms internal consistency; it does not validate the assumptions themselves.

In [ ]:
# Print final cash flow statement
print("=== FINAL Projected Cash Flow Statement ($M) ===")
print(f"{'':>22} " + " ".join(f"{'Y'+str(y):>10}" for y in years_proj))
print("-" * 78)
for name, key, sign in [
    ('Net Income', 'net_income', 1), ('+ D&A', 'da', 1),
    ('- Incr. AR', 'delta_ar', -1), ('- Incr. Inv', 'delta_inv', -1),
    ('+ Incr. AP', 'delta_ap', 1), ('Cash from Ops', 'cfo', 1),
    ('Capex', 'capex', -1), ('Cash from Invest', 'cfi', 1),
    ('Cash from Finance', 'cff', 1), ('Net Cash Change', 'net_cash_change', 1),
]:
    vals = base[key] * sign
    print(f"  {name:<22}" + " ".join(f"{v:>10.1f}" for v in vals))

# Verify
cash_all = np.concatenate([[cash_hist[-1]], base['cash']])
actual_delta = np.diff(cash_all)
print(f"\n  Reconciliation:")
for i in range(n_proj):
    diff = base['net_cash_change'][i] - actual_delta[i]
    print(f"    Y{years_proj[i]}: CF change = {base['net_cash_change'][i]:.2f}, "
          f"BS change = {actual_delta[i]:.2f}, Diff = {diff:.8f}")

### Interpretation: Final Projected Cash Flow Statement

The converged cash flow statement ties everything together:

- **CFO composition** — net income plus D&A minus working capital investment. The D&A add-back is typically the largest single adjustment, followed by working capital changes.
- **Working capital drag** — for a growing company, the annual increase in working capital is a persistent cash drain. Quantify this as a percentage of revenue growth to understand the marginal working capital requirement.
- **Free cash flow** — this is the ultimate measure of the company's cash-generating ability. Positive and growing FCF supports the company's ability to service debt, pay dividends, and fund acquisitions.
- **Cash flow reconciliation** — verify that Beginning Cash + CFO + CFI + CFF = Ending Cash. This reconciliation is the strongest test of model integrity.

> **Key Concept:** The cash flow statement is the "truth serum" of financial analysis. A company can report strong earnings through aggressive accounting, but it cannot fabricate cash. When net income and cash flow diverge, the cash flow statement tells the true story.

### Reading the converged cash flow statement

The converged cash flow statement shows the projected cash dynamics at the resolved capital structure. Several diagnostic questions:

1. **CFO trajectory:** Is operating cash flow growing? At what rate compared to net income?

2. **CFI consistency:** Investment cash flows should be primarily CapEx, scaling roughly with revenue.

3. **CFF interpretation:** What is the company doing with its cash? Paying dividends, repaying debt, building cash, or all three?

4. **Net change in cash:** The sum CFO + CFI + CFF gives the year's cash change. This must reconcile with the balance sheet's cash movement.

### The FCFF and FCFE calculations

The model also computes:

$$\text{FCFF} = \text{CFO} + \text{Int}(1-t) - \text{CapEx}$$
$$\text{FCFE} = \text{FCFF} - \text{Int}(1-t) + \text{Net Borrowing}$$

These figures are the inputs to DCF valuation. A 5-year FCFF projection plus a terminal value assumption gives the enterprise value; FCFE plus terminal value gives equity value directly.

### Validating against history

A useful sanity check: compare projected CFO/Net Income ratio to the historical ratio. If the model projects a dramatically different ratio, investigate why. Possible reasons:
* Working capital efficiency assumed to improve significantly
* Depreciation policy changing (different capex/asset mix)
* Tax rate changes
* Non-recurring items in history that were excluded from projection

If you can't articulate the reason, the assumption may be flawed.

> **CFA Exam Tip:** Always verify that the projected statements are *internally consistent*. The simplest check: the change in cash from the cash flow statement should equal the change in cash on the balance sheet. If it doesn't, the model has an error somewhere — usually in the working capital adjustments.

---
## 12. Scenario Analysis

### 12.1 Why scenarios matter

A single-point forecast creates a false sense of precision. In reality, the future is uncertain, and modellers should explore a **range of outcomes**.

The standard approach is to define three scenarios:

| Scenario | Description | Revenue Growth | COGS Adjustment | Capex/D&A |
|----------|-------------|---------------|-----------------|-----------|
| **Bull** | Strong economy, market share gains | +2pp above base | -1pp below base | 1.5x |
| **Base** | Continuation of recent trends | As projected | As projected | 1.3x |
| **Bear** | Recession, margin compression | -3pp below base | +2pp above base | 1.1x |

### 12.2 Fan charts

A **fan chart** displays the range of outcomes as coloured bands, with the base case in the centre and increasingly extreme scenarios in lighter shades. This is a powerful communication tool for boards and investors.

> **Key Concept:** Scenario analysis forces the modeller to think about *what could go wrong* (and what could go right). It transforms a model from a single prediction into a **decision tool** that shows the sensitivity of outcomes to key assumptions.

> **CFA Exam Tip:** The CFA curriculum distinguishes between **scenario analysis** (discrete cases with specific assumptions) and **sensitivity analysis** (varying one input at a time). Both are tested. Scenario analysis changes multiple inputs simultaneously to tell a coherent "story" about a possible future state.

### 12.2 Constructing Scenarios

Each scenario should be **internally consistent** — the assumptions should tell a coherent economic story:

**Bull Case Story:** The economy strengthens, demand for industrial products surges, Apex gains market share through operational excellence, and commodity costs remain contained. Higher revenue combines with operating leverage to produce significant margin expansion.

**Base Case Story:** The economy grows at trend, Apex maintains its market position, costs rise modestly but are partially offset by efficiency gains. This is the "most likely" scenario.

**Bear Case Story:** Economic slowdown reduces demand, competitive pressure intensifies, input costs rise. Revenue growth slows significantly, and margins compress as fixed costs are spread over a smaller base.

> **Key Concept:** Good scenario analysis is not about picking arbitrary numbers. Each scenario should be a plausible **narrative** about the future, with assumptions that are mutually consistent. A scenario with high revenue growth *and* high cost pressure is unlikely unless there are specific, identifiable reasons.

### 12.3 Scenario Selection Methodology

The choice of scenario parameters should be grounded in data:

| Parameter | Base | Bull | Bear | Basis |
|-----------|------|------|------|-------|
| Revenue growth | Historical average | 75th percentile | 25th percentile | Historical distribution |
| Gross margin | Recent level | Peer best-in-class | Recessionary trough | Historical range |
| Capex/Revenue | Management guidance | Expansion plan | Maintenance only | Capital plan |

> **CFA Exam Tip:** Scenario analysis on the CFA exam often asks you to assess the *sensitivity* of a valuation or credit metric to different scenarios. Be prepared to explain which assumptions have the greatest impact on the output metric — this connects to the tornado chart analysis in Section 13.

### 12.4 Interpreting Fan Charts

The fan chart visualisation shows the range of outcomes across scenarios. Key things to look for:

- **Width of the fan** — wider fans indicate greater uncertainty. Metrics with wide fans deserve closer attention.
- **Symmetry** — if the bull case is much further from base than the bear case (or vice versa), the distribution of outcomes is skewed.
- **Convergence or divergence** — if scenarios diverge over time, uncertainty compounds. If they converge, there is a self-correcting mechanism.

### Why scenario analysis matters

The base case is one possible future. Scenario analysis — running the model under multiple coherent sets of assumptions — reveals:

1. **The range of plausible outcomes:** Best case to worst case
2. **The sensitivity to key assumptions:** Which drivers matter most?
3. **The risk profile:** How much downside is possible? How likely?
4. **The optionality:** What's the upside if everything goes right?

### The traditional three-scenario framework

Most models use three named scenarios:

| Scenario | Purpose | Typical Assumptions |
|----------|---------|---------------------|
| **Bear** | Stress test | Revenue declines, margins compress, working capital deteriorates |
| **Base** | Central expectation | Historical trends continue at moderate pace |
| **Bull** | Upside case | Revenue accelerates, margins expand, efficiency improves |

The choice of three scenarios is somewhat arbitrary — some firms use five (very bear, bear, base, bull, very bull) or two (base, downside). The key is to define meaningful, coherent variations rather than arbitrary numerical perturbations.

### Coherence is critical

A common error is to construct scenarios by just changing one variable. Real economic scenarios involve correlated changes:

**Bad bear case:** Revenue declines but margins stay the same (unrealistic — operating leverage means margins compress in downturns)

**Good bear case:** Revenue declines, gross margin compresses (input cost pressure can't be passed through), operating margin compresses more (fixed costs become a larger share), working capital deteriorates (slower collections, inventory buildup), tax expense doesn't fall proportionally (some costs are not deductible)

The good bear case captures the *correlated* deterioration that occurs in real downturns. The bad bear case underestimates risk.

> **Key Concept:** Scenarios should tell coherent economic stories. The bear case should describe a specific bad outcome (recession, competitive entry, commodity price spike) and the bull case should describe a specific good outcome (market expansion, successful product launch, cost program). The numerical assumptions should follow from the narrative.

### Confidence intervals via fan charts

The fan chart visualisation displays the range of outcomes graphically:
* The middle line is the base case
* The shaded region above and below shows the bear-to-bull range
* Sometimes intermediate scenarios add additional bands

This visual representation makes the *uncertainty* explicit, which a single base case projection conceals.

> **CFA Exam Tip:** When presenting financial projections, always include scenario analysis. A point estimate alone is meaningless without context about how confident you are and what range of outcomes is possible. The exam may ask you to identify the assumption with the largest impact on the projected outcome — this is what scenario analysis reveals.

In [ ]:
# ── Define scenario assumptions ──
# Bull: higher growth, lower costs, more investment
bull_growth_adj = 0.02  # +2pp revenue growth
bull_cogs_adj = -0.01   # -1pp COGS ratio
bull_capex_da = 1.50

# Bear: lower growth, higher costs, less investment
bear_growth_adj = -0.03
bear_cogs_adj = 0.02
bear_capex_da = 1.10

# Compute scenario revenues
base_growth_rates = np.diff(np.concatenate([[revenue_hist[-1]], revenue_proj])) / \
                    np.concatenate([[revenue_hist[-1]], revenue_proj[:-1]])

bull_revenue = np.zeros(n_proj)
bear_revenue = np.zeros(n_proj)
rev_prev_bull = revenue_hist[-1]
rev_prev_bear = revenue_hist[-1]
for t in range(n_proj):
    bull_revenue[t] = rev_prev_bull * (1 + base_growth_rates[t] + bull_growth_adj)
    bear_revenue[t] = rev_prev_bear * (1 + max(base_growth_rates[t] + bear_growth_adj, -0.05))
    rev_prev_bull = bull_revenue[t]
    rev_prev_bear = bear_revenue[t]

# Run bull model
bull = build_integrated_model(
    revenue=bull_revenue,
    cogs_pct=cogs_pct_proj + bull_cogs_adj,
    sga_fixed=sga_fixed, sga_var_pct=sga_variable_pct,
    rd_pct=rd_pct_proj, depr_rate=depr_rate, capex_da_ratio=bull_capex_da,
    tax_rate=tax_rate, avg_int_rate=avg_interest_rate,
    payout_ratio=avg_payout, dso=dso_proj, dio=dio_proj, dpo=dpo_proj,
    st_debt_schedule=st_debt_proj_init, lt_debt_schedule=lt_debt_proj_init,
    hist_data=hist_data,
)

# Run bear model
bear = build_integrated_model(
    revenue=bear_revenue,
    cogs_pct=cogs_pct_proj + bear_cogs_adj,
    sga_fixed=sga_fixed, sga_var_pct=sga_variable_pct,
    rd_pct=rd_pct_proj, depr_rate=depr_rate, capex_da_ratio=bear_capex_da,
    tax_rate=tax_rate, avg_int_rate=avg_interest_rate,
    payout_ratio=avg_payout, dso=dso_proj, dio=dio_proj, dpo=dpo_proj,
    st_debt_schedule=st_debt_proj_init, lt_debt_schedule=lt_debt_proj_init,
    hist_data=hist_data,
)

print("Scenario Summary -- Net Income ($M):")
print(f"{'Year':>6} {'Bear':>10} {'Base':>10} {'Bull':>10}")
for i in range(n_proj):
    print(f"  Y{years_proj[i]}: {bear['net_income'][i]:>9.1f} "
          f"{base['net_income'][i]:>9.1f} {bull['net_income'][i]:>9.1f}")

### Interpretation: Scenario Assumptions

The scenario definitions capture three plausible futures for Apex. Examining the assumptions:

- **Revenue growth** varies most across scenarios — this is typically the largest driver of uncertainty in a financial model. The bull case assumes acceleration from historical trends, while the bear case assumes deceleration.
- **Cost assumptions** are adjusted consistently with the revenue story. In the bull case, higher volumes produce scale economies (lower COGS %); in the bear case, underutilisation raises per-unit costs.
- **Capex** varies to reflect management's response to different demand environments. Higher growth justifies higher investment; slower growth leads to capex restraint.

The scenarios are passed to our `build_integrated_model` function, which runs the full three-statement model for each scenario with circular reference resolution. This ensures that every scenario produces internally consistent financial statements.

> **Common Mistake:** Running scenarios with only one parameter changed at a time (ceteris paribus) is sensitivity analysis, not scenario analysis. True scenarios change *multiple* parameters simultaneously to reflect a coherent economic narrative.

### Reading the scenario specifications

The three scenarios for Apex Manufacturing capture different economic narratives:

**Bull case:** "Apex captures market share as competitors struggle"
* Revenue growth above historical trend
* Margin expansion as scale benefits compound
* Working capital efficiency improves through digitisation initiatives

**Base case:** "Apex continues current trajectory"
* Revenue growth in line with industry
* Stable margins (input cost pressure offset by pricing)
* Working capital ratios stable

**Bear case:** "Industrial downturn squeezes Apex"
* Revenue declines or stagnates
* Margin compression as fixed costs become a larger share
* Working capital deteriorates (customers stretching payments)

### Notice the correlated changes

In the bull case, all variables move favourably together: revenue, margins, and working capital all improve. In the bear case, all variables move unfavourably together. This is realistic — economic environments tend to drive correlated movements in financial metrics.

Modelling these correlations explicitly produces more realistic projections than independently varying each input.

> **Common Mistake:** Some analysts test a "bear case" with only revenue declining, leaving all other metrics at base case levels. This understates downside risk because in actual recessions, margins typically compress and working capital deteriorates simultaneously. A robust bear case captures all the correlated negative effects.

In [ ]:
# ── Fan chart visualisation ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = [
    ('Revenue', 'revenue', revenue_hist),
    ('Net Income', 'net_income', net_income_hist),
    ('EBIT Margin (%)', None, ebit_hist / revenue_hist * 100),
    ('Free Cash Flow', None, cfo_hist + cfi_hist),
]

for ax, (title, key, hist_vals) in zip(axes.flat, metrics):
    if title == 'EBIT Margin (%)':
        base_vals = base['ebit'] / base['revenue'] * 100
        bull_vals = bull['ebit'] / bull['revenue'] * 100
        bear_vals = bear['ebit'] / bear['revenue'] * 100
    elif title == 'Free Cash Flow':
        base_vals = base['cfo'] + base['cfi']
        bull_vals = bull['cfo'] + bull['cfi']
        bear_vals = bear['cfo'] + bear['cfi']
    else:
        base_vals = base[key]
        bull_vals = bull[key]
        bear_vals = bear[key]

    # Historical
    ax.plot(years_hist, hist_vals, 'ko-', markersize=6, linewidth=2, label='Historical')

    # Fan: shade between bear and bull
    ax.fill_between(years_proj, bear_vals, bull_vals, alpha=0.15, color=PRIMARY)
    ax.fill_between(years_proj,
                    base_vals - 0.3*(base_vals - bear_vals),
                    base_vals + 0.3*(bull_vals - base_vals),
                    alpha=0.3, color=PRIMARY)

    ax.plot(years_proj, base_vals, 's-', color=PRIMARY, markersize=5, label='Base')
    ax.plot(years_proj, bull_vals, '^--', color=TERTIARY, markersize=5, alpha=0.7, label='Bull')
    ax.plot(years_proj, bear_vals, 'v--', color=SECONDARY, markersize=5, alpha=0.7, label='Bear')

    ax.axvline(x=3.5, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xticks(np.concatenate([years_hist, years_proj]))
    ax.set_xticklabels([f'Y{y}' for y in np.concatenate([years_hist, years_proj])], fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Scenario Analysis -- Fan Charts', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Interpretation: Fan Chart Visualisation

The fan charts display the range of outcomes across bull, base, and bear scenarios:

- **Revenue fan** — shows the widening range of revenue outcomes over the projection horizon. The spread increases with time because growth rate differences compound.
- **EBIT fan** — typically wider (in percentage terms) than the revenue fan due to **operating leverage**. A 10% revenue difference might translate to a 20-30% EBIT difference because fixed costs amplify revenue changes.
- **FCF fan** — the widest fan, reflecting the combined uncertainty of revenue, margins, working capital, and capex. FCF is the most volatile metric because it captures all sources of variation.
- **Debt/Cash fan** — shows the cumulative financial position. In the bull case, the company may be debt-free by Year 5; in the bear case, debt may increase significantly.

The fan charts make it immediately clear which metrics are most sensitive to scenario assumptions and how uncertainty compounds over time.

> **Key Concept:** The fan chart is a communication tool — it conveys the **range of uncertainty** to decision-makers far more effectively than a table of numbers. Always include fan charts or waterfall charts when presenting model results to stakeholders.

### Reading the fan chart

The fan chart visualises the range of projected outcomes across scenarios. Key features:

* **The central line** (often base case) represents the modal expectation
* **The shaded region** shows the bear-to-bull range
* **Width** of the fan increases over time, reflecting growing uncertainty in farther-out projections

### What the fan chart reveals

Several insights emerge from the visualisation:

1. **Year 1 spread:** The bear-to-bull range in Year 1 reflects near-term uncertainty. For most variables, this is relatively narrow — current conditions are knowable.

2. **Year 5 spread:** The terminal-year range captures cumulative uncertainty. Even small annual differences compound dramatically over 5 years.

3. **Asymmetry:** If the upside (bull-to-base) is larger than the downside (base-to-bear), the company has more upside optionality. The opposite indicates more downside risk.

4. **Inflection points:** Look for years where the fan widens or narrows abruptly. These typically reflect specific events (new product launch, major investment, refinancing) that the model has captured.

### Using fan charts for valuation

Fan charts directly inform valuation:
* **Discounting bull case FCF at WACC** gives a high-end equity value
* **Discounting bear case FCF at WACC** gives a low-end equity value
* **The base case** gives the central estimate
* **Probability-weighted average** can produce an "expected value"

The width of the resulting valuation range (from bear-discounted to bull-discounted) is the *intrinsic uncertainty* in the company's value — typically quoted as a range rather than a single point estimate.

> **Key Concept:** Fan charts make uncertainty visible. A single base case projection invites the viewer to treat it as "the answer." A fan chart correctly conveys that the future is uncertain and that responsible analysis must consider the range of possibilities, not just the central estimate.

---
## 13. Monte Carlo Simulation

### 13.1 Beyond three scenarios

Scenario analysis considers a handful of discrete cases. **Monte Carlo simulation** goes further by randomly sampling thousands of possible futures, each defined by a different set of input assumptions drawn from probability distributions.

### 13.2 Methodology

For each simulation $i = 1, \ldots, N$:

1. **Draw random assumptions:**
   - Revenue growth rate: $g_i \sim \mathcal{N}(\mu_g, \sigma_g^2)$
   - COGS ratio adjustment: $c_i \sim \mathcal{N}(0, \sigma_c^2)$
   - Capex/D&A ratio: $k_i \sim \mathcal{N}(\mu_k, \sigma_k^2)$

2. **Run the full integrated model** with these assumptions.

3. **Record outputs** of interest (e.g., terminal-year free cash flow, ending equity).

After $N$ simulations, we have a **distribution** of outcomes, from which we can compute percentiles, expected values, and risk measures.

### 13.3 Sensitivity analysis

A **tornado chart** ranks the input variables by their impact on a key output. For each input, we compute the output at the 10th and 90th percentile of that input while holding others at their median, revealing which assumptions matter most.

> **Key Concept:** Monte Carlo simulation converts a deterministic model into a probabilistic one. Instead of asking "what is our forecast?", we ask "what is the *distribution* of possible outcomes?" This is fundamentally more honest and more useful for decision-making.

> **CFA Exam Tip:** Monte Carlo simulation is covered in the CFA curriculum under quantitative methods. The key exam points are: (1) it requires specifying probability distributions for inputs, (2) it produces a distribution of outputs, not a point estimate, and (3) the quality of results depends entirely on the quality of input assumptions.

> **Common Mistake:** Running Monte Carlo with independent random draws when the inputs are actually correlated. If revenue growth and COGS ratio are correlated (they often are — booms raise both revenue and input costs), ignoring this correlation will understate the tails of the distribution.

### 13.3 Advantages of Monte Carlo Over Discrete Scenarios

| Feature | Scenario Analysis | Monte Carlo Simulation |
|---------|------------------|----------------------|
| Number of cases | 3-5 | 1,000-100,000 |
| Output | Point estimates per scenario | Full probability distributions |
| Probability quantification | Subjective (if any) | Explicit percentiles and confidence intervals |
| Correlation modelling | Limited | Can model inter-assumption correlations |
| Tail risk assessment | Miss extreme outcomes | Captures fat tails if distribution is specified |

> **Key Concept:** Monte Carlo simulation does not replace judgement — it **quantifies** the uncertainty that scenario analysis can only describe qualitatively. When a CFO asks "what is the probability that FCF will be negative next year?", only Monte Carlo can provide a rigorous answer.

### 13.4 Input Distributions

For each stochastic assumption, we specify a probability distribution:

| Assumption | Distribution | Parameters | Rationale |
|------------|-------------|------------|-----------|
| Revenue growth | Normal | $\mu = 6\%, \sigma = 2\%$ | Symmetric around base case |
| COGS % of revenue | Triangular | (60%, 62%, 65%) | Bounded with slight upside skew |
| SG&A variable % | Normal | $\mu = 10\%, \sigma = 1\%$ | Small variation around stable level |
| Capex/Revenue | Uniform | (4%, 7%) | Management discretion within range |

The **choice of distribution** matters:
- **Normal** — appropriate when the variable can be higher or lower with equal probability, and extreme outcomes are rare
- **Triangular** — appropriate when you can specify minimum, most likely, and maximum values
- **Uniform** — appropriate when all values in a range are equally likely (maximum uncertainty within bounds)
- **Lognormal** — appropriate for variables that are strictly positive and right-skewed (e.g., revenue levels)

### 13.5 Correlation Between Assumptions

In reality, assumptions are not independent. When the economy is strong:
- Revenue growth is higher
- Costs may also be higher (labour, materials)
- Capex may increase (expansion investment)

Ignoring correlations can **overstate** the probability of extreme outcomes (where everything goes right or everything goes wrong simultaneously).

To model correlations, we can use a **correlation matrix** and generate correlated random draws using Cholesky decomposition:

$$\mathbf{L} \mathbf{L}^T = \boldsymbol{\Sigma}$$

$$\text{Correlated draws} = \mathbf{L} \cdot \text{Independent draws}$$

> **CFA Exam Tip:** The CFA curriculum discusses Monte Carlo simulation in the context of risk management and derivatives pricing. The key concepts — random sampling from specified distributions, large sample sizes for convergence, and the importance of input assumptions — apply equally to financial modelling.

### 13.6 Interpreting Monte Carlo Output

Monte Carlo simulation produces distributions, not point estimates. Key statistics to examine:

- **Mean and median** — central tendency of the output distribution
- **Standard deviation** — spread of outcomes
- **Percentiles** (5th, 25th, 75th, 95th) — define confidence intervals
- **Probability of specific events** — e.g., P(FCF < 0), P(debt coverage ratio < 1.5)
- **Shape of distribution** — symmetric (normal-like) or skewed (more upside or downside risk)

### 13.7 Tornado Chart Theory

A **tornado chart** (or sensitivity chart) ranks assumptions by their impact on a key output metric. For each assumption:

1. Fix all other assumptions at their base case values
2. Set the assumption to its 10th percentile — record the output
3. Set the assumption to its 90th percentile — record the output
4. The **width of the bar** = difference between the two outputs

The assumptions are ranked by bar width, with the most impactful at the top — creating the characteristic "tornado" shape.

> **Key Concept:** The tornado chart answers the question "which assumptions matter most?" This is invaluable for prioritising research effort — spend more time refining assumptions that have a large impact on the output, and less time on assumptions that barely move the needle.

### Why Monte Carlo extends scenario analysis

Discrete scenarios (bear/base/bull) are useful but limited:
* Only 3 (or a few) discrete cases
* Each scenario picks specific values for each driver — but reality is continuous
* No probability weighting (which scenario is most likely?)

Monte Carlo simulation addresses these limitations:
* **Continuous distributions** for each driver (e.g., revenue growth ~ Normal(8%, 2%))
* **Thousands of simulations** sampling from these distributions
* **Output distribution** showing the full range of possible outcomes with frequencies

### The Monte Carlo workflow

1. **Specify input distributions:** For each driver (revenue growth, COGS%, working capital efficiency, etc.), define a probability distribution. Common choices:
   * **Normal:** For drivers expected to be roughly symmetric (revenue growth)
   * **Triangular:** For drivers with known min/max bounds (margin)
   * **Uniform:** For drivers with no preferred value (rare)
   * **Lognormal:** For multiplicative drivers (compound growth)

2. **Specify correlations:** Real-world drivers move together. Recessions tend to compress margins AND deteriorate working capital simultaneously. The model should capture this via a correlation matrix.

3. **Run many simulations:** Typically 10,000+ Monte Carlo paths, each sampling from the input distributions and running the full 3-statement model.

4. **Analyse output distributions:** Key statistics:
   * Mean, median (central tendency)
   * Standard deviation (volatility)
   * 5th, 95th percentile (range)
   * Probability of various thresholds (P(NPV > 0), P(FCF < 0), etc.)

### Why Monte Carlo is more rigorous than discrete scenarios

Three discrete scenarios force the analyst to choose specific values for each driver. Monte Carlo:
* **Considers the full distribution** of possible values, not just three points
* **Reveals the probability** of various outcomes (not just "best/worst case")
* **Captures interactions** between variables (a low-margin AND low-revenue scenario may be much worse than either alone)

The trade-off: Monte Carlo is more computationally intensive and harder to explain to non-technical stakeholders. Discrete scenarios remain useful for communication; Monte Carlo for rigorous risk analysis.

> **Key Concept:** Monte Carlo simulation transforms financial modelling from deterministic prediction to probabilistic analysis. Instead of asking "what will FCF be?" (impossible to answer), it asks "what is the distribution of possible FCF outcomes, and what's the probability that FCF exceeds X?" This is the right framing for any decision under uncertainty.

> **Common Mistake:** Beginners sometimes use Monte Carlo with arbitrary distributions (e.g., uniform between min and max) without justification. The chosen distribution should reflect actual beliefs about the variable. Historical data, industry studies, or explicit subjective probability assessment should inform distribution choices — not convenience.

In [ ]:
# ── Monte Carlo simulation ──
N_SIM = 10_000

# Random parameters
growth_mean = avg_growth
growth_std = 0.03
cogs_adj_std = 0.015
capex_da_mean = 1.30
capex_da_std = 0.15

# Storage
terminal_fcf = np.zeros(N_SIM)
terminal_ni = np.zeros(N_SIM)
terminal_revenue = np.zeros(N_SIM)

# Store parameter draws for sensitivity analysis
param_draws = np.zeros((N_SIM, 3))  # growth, cogs_adj, capex_da

print(f"Running {N_SIM:,} Monte Carlo simulations...")

for sim in range(N_SIM):
    # Draw random parameters
    g_draw = rng.normal(growth_mean, growth_std)
    c_draw = rng.normal(0, cogs_adj_std)
    k_draw = np.clip(rng.normal(capex_da_mean, capex_da_std), 0.8, 2.0)

    param_draws[sim] = [g_draw, c_draw, k_draw]

    # Build revenue with random growth
    rev_sim = revenue_hist[-1] * (1 + g_draw) ** np.arange(1, n_proj + 1)

    # Run model
    try:
        result = build_integrated_model(
            revenue=rev_sim,
            cogs_pct=cogs_pct_proj + c_draw,
            sga_fixed=sga_fixed, sga_var_pct=sga_variable_pct,
            rd_pct=rd_pct_proj, depr_rate=depr_rate, capex_da_ratio=k_draw,
            tax_rate=tax_rate, avg_int_rate=avg_interest_rate,
            payout_ratio=avg_payout, dso=dso_proj, dio=dio_proj, dpo=dpo_proj,
            st_debt_schedule=st_debt_proj_init, lt_debt_schedule=lt_debt_proj_init,
            hist_data=hist_data,
        )
        terminal_fcf[sim] = result['cfo'][-1] + result['cfi'][-1]
        terminal_ni[sim] = result['net_income'][-1]
        terminal_revenue[sim] = result['revenue'][-1]
    except Exception:
        terminal_fcf[sim] = np.nan
        terminal_ni[sim] = np.nan
        terminal_revenue[sim] = np.nan

# Remove any failed simulations
valid = ~np.isnan(terminal_fcf)
terminal_fcf = terminal_fcf[valid]
terminal_ni = terminal_ni[valid]
terminal_revenue = terminal_revenue[valid]
param_draws = param_draws[valid]

print(f"Completed {valid.sum():,} valid simulations.")
print(f"\nTerminal-Year FCF Distribution ($M):")
print(f"  Mean:   ${terminal_fcf.mean():>8.1f}M")
print(f"  Median: ${np.median(terminal_fcf):>8.1f}M")
print(f"  Std:    ${terminal_fcf.std():>8.1f}M")
pcts = [5, 25, 75, 95]
for p in pcts:
    print(f"  {p}th pct: ${np.percentile(terminal_fcf, p):>8.1f}M")

### Interpretation: Monte Carlo Simulation Results

The Monte Carlo simulation generates a distribution of outcomes for each financial metric. Key findings:

- **Mean vs Base Case** — the mean of the simulated distribution may differ from the base case deterministic forecast due to **Jensen's inequality** (non-linear transformations of random variables shift the mean). If the mean FCF from simulation is lower than the base case FCF, the model exhibits negative convexity — downside scenarios hurt more than upside scenarios help.
- **Standard deviation** — the spread of outcomes quantifies total uncertainty. Compare the standard deviation of FCF to the mean — a coefficient of variation above 0.5 suggests high uncertainty.
- **Percentile range** — the 5th-to-95th percentile range captures 90% of simulated outcomes. If this range is very wide, the model is highly sensitive to assumptions, and the base case should be interpreted with caution.
- **Probability of distress** — compute P(FCF < 0) or P(interest coverage < 1.0) to assess financial risk. Even a 5-10% probability of cash flow shortfall is significant for credit analysis.

The simulation results should be presented alongside the discrete scenarios to provide both narrative (scenarios) and probabilistic (Monte Carlo) perspectives on the forecast.

> **Common Mistake:** Running too few simulations produces unstable results. With 1,000 simulations, the mean is reasonably stable but tail percentiles (1st, 99th) are not. For reliable tail statistics, use 10,000+ simulations.

### Reading the Monte Carlo simulation results

The simulation produces a distribution of outcomes for each modelled metric (typically terminal-year revenue, net income, and free cash flow). Key statistics:

1. **Mean and median:** The central tendency. If they differ significantly, the distribution is skewed (often the case for compound metrics).

2. **Standard deviation:** A measure of uncertainty. Higher SD means wider range of possible outcomes.

3. **Percentiles:** The 5th-95th percentile range captures 90% of likely outcomes. The 25th-75th percentile range (interquartile) captures the central 50%.

4. **Tail risks:** What's the probability of extreme outcomes? P(FCF < 0)? P(NI declines from current?)

### Comparing to discrete scenarios

A useful exercise: compare the discrete bear/base/bull values to the Monte Carlo percentiles.
* The base case might correspond to roughly the 50th percentile (median)
* The bear case might correspond to roughly the 10th-15th percentile
* The bull case might correspond to roughly the 85th-90th percentile

If the discrete scenarios match these percentiles approximately, the scenario analysis is well-calibrated. If they don't (e.g., the bear case is at the 1st percentile of Monte Carlo), the discrete scenarios are too extreme to be likely.

### Convergence and number of simulations

Monte Carlo results are themselves random — running 10,000 simulations vs 1,000 vs 100,000 produces somewhat different output distributions. As N → ∞, the empirical distribution converges to the true theoretical distribution.

In practice:
* **N = 1,000:** Useful for quick exploration
* **N = 10,000:** Standard for professional analysis
* **N = 100,000+:** When tail probabilities (5th, 95th percentile) require high precision

The standard error of the mean shrinks as 1/√N, so quadrupling the simulations halves the precision improvement — diminishing returns.

> **Key Concept:** Monte Carlo simulation is most valuable for **risk analysis**, not point prediction. The mean of the distribution is similar to the deterministic base case, but the *spread* and *tails* of the distribution reveal information about risk that no single projection can convey.

In [ ]:
# ── Histogram of terminal-year FCF ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, data, title, color in zip(
    axes,
    [terminal_revenue, terminal_ni, terminal_fcf],
    ['Terminal Revenue', 'Terminal Net Income', 'Terminal FCF'],
    [PRIMARY, TERTIARY, SECONDARY]
):
    ax.hist(data, bins=80, color=color, alpha=0.7, edgecolor='white', linewidth=0.3)
    mean_val = np.mean(data)
    p5_val = np.percentile(data, 5)
    p95_val = np.percentile(data, 95)
    ax.axvline(mean_val, color='black', linestyle='--', linewidth=1.5,
               label=f'Mean: ${mean_val:.0f}M')
    ax.axvline(p5_val, color='red', linestyle=':', linewidth=1.2,
               label=f'5th: ${p5_val:.0f}M')
    ax.axvline(p95_val, color='red', linestyle=':', linewidth=1.2,
               label=f'95th: ${p95_val:.0f}M')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('$M')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.suptitle('Monte Carlo Results -- ' + f'{valid.sum():,} Simulations (Year {years_proj[-1]})',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Interpretation: Output Distributions

The histograms visualise the probability distributions of key terminal-year metrics:

- **Distribution shape** — a roughly symmetric (bell-shaped) distribution suggests that upside and downside risks are balanced. A right-skewed distribution means there is more upside potential; a left-skewed distribution means more downside risk.
- **Width** — wider distributions indicate greater uncertainty. Compare the width (standard deviation) across metrics — FCF typically has the widest distribution because it aggregates uncertainty from all assumptions.
- **Tails** — examine the extreme outcomes. Are there scenarios where FCF is deeply negative (financial distress)? Are there scenarios with exceptionally high returns? The tails drive the expected value of options and contingent claims.
- **Comparison to base case** — mark the base case value on the distribution. Is it near the centre (median) or shifted to one side? If shifted, the base case may be optimistic or pessimistic relative to the full distribution.

> **Key Concept:** The histogram transforms a single-point forecast into a **probability distribution**, which is far more informative for decision-making. A CFO who sees that there is a 15% chance of negative FCF will make different capital allocation decisions than one who only sees a positive base case FCF.

> **CFA Exam Tip:** Monte Carlo output is often reported as "Value at Risk" (VaR) — for example, "there is a 5% probability that FCF will be below $X." This is the same concept used in market risk management, applied here to financial planning.

### Reading the histogram of Monte Carlo outputs

The histogram displays the empirical distribution of the simulated outcomes. Several features deserve attention:

1. **Shape:** Is it roughly normal (bell-shaped)? Skewed (longer tail on one side)? Bimodal (two peaks)?

2. **Spread:** How wide is the range? Compare the difference between min and max to the mean.

3. **Skewness:** Compound effects (revenue × margin × leverage) tend to produce right-skewed distributions for upside metrics (NI, FCF) — large positive outcomes are possible but less likely. The opposite for cost metrics.

4. **Tail behaviour:** Are there extreme outcomes? Tail risks (5%-15% probability of very bad outcomes) are the most important risks for risk management.

### Practical statistics to compute

From the histogram, several practical statistics are derived:

| Statistic | Use Case |
|-----------|----------|
| Mean | Expected value (input to expected NPV) |
| Median | Middle outcome (less affected by tails) |
| Std Dev | Volatility measure |
| 5th percentile | "Bear" stress level (95% confidence floor) |
| 95th percentile | "Bull" upside level |
| P(X < threshold) | Probability of downside event |

### Connecting back to valuation

If the simulated metric is **terminal-year FCF**, this distribution can be discounted to present value to produce a **value distribution**. The mean of the value distribution gives the expected enterprise value; the percentiles give a confidence interval for value.

This contrasts with traditional DCF which produces a single point value. The Monte Carlo approach produces a *distribution of values*, with the spread reflecting genuine intrinsic uncertainty in the business.

> **CFA Exam Tip:** While the CFA exam typically focuses on deterministic DCF (single base case), the curriculum increasingly emphasises understanding uncertainty. Knowing how to conceptually transform a deterministic model into a probabilistic one — even without calculating it on the exam — demonstrates the analytical maturity expected at higher CFA levels.

In [ ]:
# ── Tornado sensitivity chart ──
# For each parameter, compute FCF at its 10th and 90th percentile
# while holding others at median

param_names = ['Revenue Growth', 'COGS Adj.', 'Capex/D&A']
param_medians = np.median(param_draws, axis=0)

tornado_lo = np.zeros(3)
tornado_hi = np.zeros(3)
base_fcf = np.median(terminal_fcf)

for p in range(3):
    p10 = np.percentile(param_draws[:, p], 10)
    p90 = np.percentile(param_draws[:, p], 90)

    for pval, store_idx in [(p10, 0), (p90, 1)]:
        params = param_medians.copy()
        params[p] = pval
        g, c, k = params

        rev_sim = revenue_hist[-1] * (1 + g) ** np.arange(1, n_proj + 1)
        result = build_integrated_model(
            revenue=rev_sim,
            cogs_pct=cogs_pct_proj + c,
            sga_fixed=sga_fixed, sga_var_pct=sga_variable_pct,
            rd_pct=rd_pct_proj, depr_rate=depr_rate, capex_da_ratio=k,
            tax_rate=tax_rate, avg_int_rate=avg_interest_rate,
            payout_ratio=avg_payout, dso=dso_proj, dio=dio_proj, dpo=dpo_proj,
            st_debt_schedule=st_debt_proj_init, lt_debt_schedule=lt_debt_proj_init,
            hist_data=hist_data,
        )
        fcf_val = result['cfo'][-1] + result['cfi'][-1]
        if store_idx == 0:
            tornado_lo[p] = fcf_val
        else:
            tornado_hi[p] = fcf_val

# Sort by range
ranges = tornado_hi - tornado_lo
sort_idx = np.argsort(ranges)

fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(len(param_names))

for i, idx in enumerate(sort_idx):
    ax.barh(i, tornado_hi[idx] - base_fcf, left=base_fcf, height=0.5,
            color=TERTIARY, alpha=0.8)
    ax.barh(i, tornado_lo[idx] - base_fcf, left=base_fcf, height=0.5,
            color=SECONDARY, alpha=0.8)
    ax.text(tornado_hi[idx] + 2, i, f'${tornado_hi[idx]:.0f}M', va='center', fontsize=10)
    ax.text(tornado_lo[idx] - 2, i, f'${tornado_lo[idx]:.0f}M', va='center',
            ha='right', fontsize=10)

ax.set_yticks(y_pos)
ax.set_yticklabels([param_names[idx] for idx in sort_idx])
ax.axvline(base_fcf, color='black', linestyle='-', linewidth=1.5)
ax.set_xlabel('Terminal-Year FCF ($M)')
ax.set_title('Tornado Chart -- Sensitivity of FCF to Input Assumptions', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nBase case FCF (median): ${base_fcf:.1f}M")
print(f"\nSensitivity ranges (P10 to P90):")
for idx in reversed(sort_idx):
    print(f"  {param_names[idx]:>18s}: ${tornado_lo[idx]:.1f}M to ${tornado_hi[idx]:.1f}M "
          f"(range: ${ranges[idx]:.1f}M)")

### Interpretation: Tornado Sensitivity Chart

The tornado chart ranks the assumptions by their impact on terminal-year FCF:

- **Top assumption** — this is the single most important input in the model. Typically, revenue growth dominates because it affects not only the top line but also costs (via operating leverage), working capital (via efficiency ratios), and capex (via revenue-linked investment).
- **Relative magnitudes** — compare the bar widths. If the top assumption's bar is 5× wider than the second, the model is essentially a one-variable model for practical purposes. If several assumptions have similar widths, the model has multiple significant risk factors.
- **Direction of bars** — each bar extends in two directions from the base case: one for the high-assumption scenario and one for the low-assumption scenario. The direction tells you whether a higher assumption value increases or decreases FCF.
- **Research prioritisation** — invest analytical effort proportionally to each assumption's impact. If revenue growth is 10× more impactful than the tax rate, spend 10× more time getting the revenue growth assumption right.

> **Key Concept:** The tornado chart is the bridge between Monte Carlo simulation and actionable insight. It answers the critical question: "Where should we focus our analytical effort to reduce forecast uncertainty?"

> **CFA Exam Tip:** Sensitivity analysis and scenario analysis are both tools for assessing forecast uncertainty, but they answer different questions. Sensitivity analysis (tornado chart) asks "which assumption matters most?" while scenario analysis asks "what happens under plausible alternative futures?" Both are valuable and complementary.

### Reading the tornado sensitivity chart

The tornado chart displays sensitivity of the output (typically terminal-year FCF or NI) to each input driver, ranked from most to least sensitive. Each bar shows:
* The output value when the input is at its low value
* The output value when the input is at its high value
* The bar width = output sensitivity to this input

The longest bars (at the top) represent the most influential drivers. The shortest bars (at the bottom) represent low-sensitivity drivers.

### Using tornado charts to focus analysis

Tornado charts answer: "Where should I spend my analytical effort?"

1. **Top drivers:** Spend disproportionate effort understanding and validating the high-sensitivity assumptions. A 1% error in the most sensitive driver may move the output more than a 10% error in a low-sensitivity driver.

2. **Bottom drivers:** Don't waste effort precisely estimating low-sensitivity inputs. The output is robust to errors in these.

3. **Surprising drivers:** If a driver you didn't expect to be influential ranks high, investigate why. Sometimes the model has unintended sensitivities (e.g., a particular formula that amplifies small changes).

### Common high-sensitivity drivers

For typical industrial companies, the most sensitive drivers are usually:
1. **Revenue growth rate** (compounds across all 5 years)
2. **Gross margin / COGS%** (large absolute dollar impact)
3. **Capital expenditure intensity** (directly affects FCF)
4. **Working capital efficiency** (affects cash conversion)
5. **Tax rate** (linear scaling effect)

Less sensitive drivers typically include:
* Specific working capital ratios (DSO, DIO, DPO individually) — net effect of all three matters more than each alone
* Interest rate (unless highly leveraged)
* Dividend payout (doesn't affect operating fundamentals)

> **Key Concept:** The tornado chart is the analytical bridge between modelling and decision-making. It tells the analyst which assumptions deserve the most scrutiny — focusing valuable analytical time on the variables that actually affect the conclusion.

> **CFA Exam Tip:** When presenting a financial model, always include sensitivity analysis. Stating that "FCF is most sensitive to revenue growth and gross margin" demonstrates analytical rigour. Without this, the model is just a calculation; with it, the model becomes a decision support tool.

---
## 14. Model Integrity Checks

A model without integrity checks is like a bridge without inspections. The two fundamental checks are:

### 14.1 Balance sheet identity

$$\text{Total Assets} = \text{Total Liabilities} + \text{Equity}$$

This must hold in **every projected year**, to the penny (or in our case, to machine precision).

### 14.2 Cash flow reconciliation

$$\text{Cash}_{t} = \text{Cash}_{t-1} + \text{CFO}_t + \text{CFI}_t + \text{CFF}_t$$

The change in cash on the balance sheet must exactly match the net cash flow from the cash flow statement.

### 14.3 Additional checks

- **Retained earnings roll-forward:** $\text{Equity}_t = \text{Equity}_{t-1} + \text{NI}_t - \text{Dividends}_t$
- **PP&E roll-forward:** $\text{PP\&E}_t = \text{PP\&E}_{t-1} + \text{Capex}_t - \text{D\&A}_t$
- **Sign checks:** Cash should be non-negative (unless a revolver is the plug), margins should be within reasonable bounds

> **Key Concept:** Professional financial modellers run integrity checks *automatically* after every model change. If a check fails, the model is broken, and all outputs are unreliable. These checks are the financial modelling equivalent of unit tests.

> **CFA Exam Tip:** If an exam question asks you to identify an error in a financial model, check the balance sheet identity first, then cash flow reconciliation. These two tests catch the vast majority of modelling errors.

### 14.3 Common Model Errors and Diagnostics

Even experienced modellers make mistakes. The most common errors and how to detect them:

| Error | Symptom | Diagnostic |
|-------|---------|-----------|
| Unlinked retained earnings | BS does not balance | Check RE = Prior RE + NI − Div |
| Missing working capital change | CF reconciliation fails | Verify ΔAR, ΔInv, ΔAP match BS changes |
| Double-counted D&A | EBIT or CFO incorrect | Verify D&A appears once in IS and once in CF |
| Sign error in capex | PP&E schedule wrong | Verify PP&E = Prior PP&E + Capex − D&A |
| Circular reference not resolved | Interest expense inconsistent | Check convergence of iterative solver |
| Hardcoded values | Model does not respond to assumption changes | Test by changing one assumption and verifying all linked cells update |
| Mismatched time periods | Revenue ≠ sum of quarters | Verify annual figures match sum of sub-periods |

### 14.4 Balance Sheet Tolerance

In a numerical model (as opposed to an exact-arithmetic spreadsheet), the balance sheet may not balance to the penny due to floating-point arithmetic. Our tolerance is:

$$|\text{Total Assets} - \text{Total Liabilities} - \text{Equity}| < \$0.01 \text{ million}$$

This is standard practice in Python/NumPy financial models. In Excel, the tolerance is typically zero because Excel uses exact decimal arithmetic for simple formulas.

### 14.5 The Model Audit Checklist

Professional financial models undergo formal audits. Key checks include:

1. **Balance sheet balances** in every projected year ✓
2. **Cash flow reconciliation** holds in every year ✓
3. **No hardcoded values** in formula cells ✓
4. **All assumptions clearly labelled** and in one place ✓
5. **Sensitivity analysis** completed ✓
6. **Sign conventions** consistent throughout ✓
7. **Historical data** reconciles to source documents ✓
8. **Circular references** resolved ✓

> **Key Concept:** Model integrity checks are not optional — they are a fundamental part of the modelling process. A model without checks is like a bridge without load testing. Always build checks *into* the model, not as an afterthought.

> **CFA Exam Tip:** The CFA curriculum emphasises the importance of **financial statement articulation** — the linkages between the three statements. A model integrity check that verifies these linkages is testing the same concept the exam tests: do you understand how the statements connect?

### Why model integrity matters

A 3-statement model has dozens of interdependencies. A single error — a wrong sign, a misallocated cell, a missing adjustment — can cascade through every output. Without explicit integrity checks, errors go undetected.

Professional financial models include a **diagnostics section** that runs automated tests. The model only passes if every check returns green. This discipline catches errors before they propagate.

### The four core integrity checks

**Check 1: Balance Sheet Balance**
$$\text{Total Assets} = \text{Total Liabilities} + \text{Total Equity}$$

This must hold in *every* projected year, with zero tolerance (or sub-cent rounding tolerance). If it fails, the working capital adjustments or balance sheet plug has an error.

**Check 2: Cash Flow Reconciliation**
$$\text{Cash}_t - \text{Cash}_{t-1} = \text{CFO}_t + \text{CFI}_t + \text{CFF}_t$$

The change in cash on the balance sheet must match the net cash flow from the cash flow statement. If it fails, the cash flow statement has an error somewhere.

**Check 3: Retained Earnings Articulation**
$$\text{Retained Earnings}_t = \text{Retained Earnings}_{t-1} + \text{Net Income}_t - \text{Dividends}_t$$

The retained earnings build must equal net income minus dividends. If it fails, either the equity rollforward or the income statement has an error.

**Check 4: PP&E Roll-forward**
$$\text{Net PP&E}_t = \text{Net PP&E}_{t-1} + \text{CapEx}_t - \text{Depreciation}_t$$

(Plus disposals, asset writedowns if modelled.) If it fails, the PP&E projection is inconsistent with the income statement (depreciation) or cash flow statement (capex).

### Beyond the core checks

Sophisticated models add additional checks:
* **Tax expense reasonableness** (effective rate near assumed)
* **Margin trajectory** (no unrealistic year-over-year jumps)
* **Working capital trajectory** (DSO, DIO, DPO at assumed levels)
* **Debt schedule** (interest rate × debt = interest expense)

These checks transform the model from a "trust me" calculation into a verifiable analytical tool.

> **Key Concept:** A model that passes all integrity checks is *internally consistent*, but that does not guarantee the assumptions are correct. Integrity checks ensure the math is right; assumption validation ensures the conclusions are meaningful. Both are necessary; neither is sufficient.

> **Common Mistake:** Many beginner models lack explicit integrity checks. The model "looks right" because the analyst trusts the formulas. Professional modelers know that any sufficiently complex model has bugs — the question is whether they're caught before publication. Always include integrity checks; always expect them to occasionally flag errors.

In [ ]:
# ── Comprehensive model integrity checks ──
print("=" * 60)
print("       MODEL INTEGRITY CHECKS -- BASE CASE")
print("=" * 60)

all_passed = True

# Check 1: Balance sheet identity
print("\n1. Balance Sheet Identity (A = L + E):")
for i in range(n_proj):
    diff = base['total_assets'][i] - base['total_le'][i]
    passed = abs(diff) < ATOL
    all_passed &= passed
    symbol = 'PASS' if passed else 'FAIL'
    print(f"   Y{years_proj[i]}: A={base['total_assets'][i]:.4f}, "
          f"L+E={base['total_le'][i]:.4f}, Diff={diff:.2e} [{symbol}]")

# Check 2: Cash flow reconciliation
print("\n2. Cash Flow Reconciliation:")
cash_arr = np.concatenate([[cash_hist[-1]], base['cash']])
for i in range(n_proj):
    bs_change = cash_arr[i+1] - cash_arr[i]
    cf_change = base['net_cash_change'][i]
    diff = bs_change - cf_change
    passed = abs(diff) < ATOL
    all_passed &= passed
    symbol = 'PASS' if passed else 'FAIL'
    print(f"   Y{years_proj[i]}: BS change={bs_change:.4f}, "
          f"CF change={cf_change:.4f}, Diff={diff:.2e} [{symbol}]")

# Check 3: Equity roll-forward
print("\n3. Equity Roll-forward:")
eq_arr = np.concatenate([[equity_hist[-1]], base['equity']])
for i in range(n_proj):
    implied = eq_arr[i] + base['net_income'][i] - base['dividends'][i]
    diff = implied - base['equity'][i]
    passed = abs(diff) < ATOL
    all_passed &= passed
    symbol = 'PASS' if passed else 'FAIL'
    print(f"   Y{years_proj[i]}: Prev+NI-Div={implied:.4f}, "
          f"Actual={base['equity'][i]:.4f}, Diff={diff:.2e} [{symbol}]")

# Check 4: PP&E roll-forward
print("\n4. PP&E Roll-forward:")
ppe_arr = np.concatenate([[ppe_hist[-1]], base['ppe']])
for i in range(n_proj):
    implied = ppe_arr[i] + base['capex'][i] - base['da'][i]
    diff = implied - base['ppe'][i]
    passed = abs(diff) < ATOL
    all_passed &= passed
    symbol = 'PASS' if passed else 'FAIL'
    print(f"   Y{years_proj[i]}: Prev+Capex-DA={implied:.4f}, "
          f"Actual={base['ppe'][i]:.4f}, Diff={diff:.2e} [{symbol}]")

# Check 5: Sign checks
print("\n5. Sign / Reasonableness Checks:")
cash_positive = np.all(base['cash'] > 0)
margins_ok = (np.all(base['ebit'] / base['revenue'] > 0) and
              np.all(base['ebit'] / base['revenue'] < 0.5))
print(f"   Cash positive all years: {'PASS' if cash_positive else 'FAIL'}")
print(f"   EBIT margins in (0%, 50%): {'PASS' if margins_ok else 'FAIL'}")
all_passed &= cash_positive and margins_ok

print(f"\n{'='*60}")
result_msg = 'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED'
print(f"  OVERALL: {result_msg}")
print(f"{'='*60}")

### Interpretation: Model Integrity Results

The integrity check results confirm whether our integrated model is structurally sound:

- **Balance sheet check** — if "PASS" in every year, the fundamental accounting identity holds and our plug variable (cash/debt) is working correctly. A failure here indicates a broken link between the income statement and balance sheet (usually retained earnings).
- **Cash flow reconciliation** — if the derived ending cash matches the balance sheet cash in every year, the cash flow statement is correctly derived from IS and BS changes. A failure indicates a missing or double-counted item.
- **Convergence check** — if the circular reference solver converged in all years, the interest expense is self-consistent with the debt levels. Non-convergence (rare with our contractive iteration) would indicate an extremely leveraged company or a coding error.

If all checks pass, we can have confidence that the model is **internally consistent** — meaning the three statements tell a coherent, self-reinforcing story. This does not mean the *assumptions* are correct, only that the *mechanics* are sound.

> **Common Mistake:** Passing integrity checks does not validate assumptions. A model can be perfectly internally consistent yet produce wildly wrong forecasts if the revenue growth assumption is off by 5%. Integrity checks ensure **structural correctness**; assumption quality requires separate analysis (scenario analysis, peer comparison, management guidance).

> **Key Concept:** The professional standard is to include an integrity check dashboard — a visible section that immediately flags any errors. This is especially important when multiple people use the model, as modifications can inadvertently break linkages.

### Reading the integrity check results

A passing model shows all checks at zero (or near-zero, within rounding tolerance). The integrity check output typically displays:

| Check | Tolerance | Status |
|-------|-----------|--------|
| Balance Sheet Balance | < \$0.01M | Pass / Fail |
| Cash Reconciliation | < \$0.01M | Pass / Fail |
| Retained Earnings | < \$0.01M | Pass / Fail |
| PP&E Roll-forward | < \$0.01M | Pass / Fail |

A model that passes all checks for all projection years is internally consistent. If any check fails, the model has a bug that must be fixed before the projections can be trusted.

### What to do when a check fails

A failed integrity check is not a disaster — it's diagnostic information. The systematic debugging approach:

1. **Identify the year of failure:** Did it fail in Year 1 or Year 5? Year 1 failures suggest formula errors; Year 5 failures may indicate accumulating rounding errors (less concerning if magnitude is small).

2. **Identify the magnitude:** A \$0.01M discrepancy is usually rounding. A \$1B discrepancy is a structural error.

3. **Trace the components:** For a balance sheet failure, sum each side independently and identify which side is "off."

4. **Check the recent change:** What was last modified? Most bugs are in recent edits.

5. **Test boundary conditions:** Run the model with simple inputs (revenue grows 0%, all ratios constant) and verify the trivial case works.

### Beyond integrity — model validation

Even a model that passes all integrity checks may produce nonsensical projections. Validation goes beyond integrity:

* **Plausibility checks:** Are projected margins, ratios, and growth rates reasonable?
* **Peer comparison:** Do projected metrics match industry norms?
* **Historical comparison:** Does the projected trajectory differ implausibly from history?
* **Stress testing:** Does the model produce reasonable outputs even under extreme inputs?

A truly robust model survives both rigorous integrity checking AND careful plausibility review.

> **CFA Exam Tip:** The CFA curriculum emphasises both quantitative rigour and qualitative judgment. Integrity checks belong to the quantitative side; plausibility belongs to the qualitative side. Strong analysts excel at both — they build models that are mathematically correct AND tell coherent economic stories.

---
## 15. References

1. **CFA Institute** (2024). *CFA Program Curriculum Level I: Financial Statement Analysis*. CFA Institute.
2. **Benninga, S.** (2014). *Financial Modeling*, 4th ed. MIT Press. — The standard reference for Excel-based three-statement modelling.
3. **Koller, T., Goedhart, M., & Wessels, D.** (2020). *Valuation: Measuring and Managing the Value of Companies*, 7th ed. Wiley. — McKinsey's approach to integrated modelling and DCF valuation.
4. **Rosenbaum, J. & Pearl, J.** (2020). *Investment Banking: Valuation, LBOs, M&A, and IPOs*, 3rd ed. Wiley. — Practical guide to building financial models in investment banking.
5. **Damodaran, A.** (2012). *Investment Valuation*, 3rd ed. Wiley. — Comprehensive treatment of forecasting and valuation methods.
6. **Palepu, K. & Healy, P.** (2013). *Business Analysis and Valuation Using Financial Statements*, 5th ed. Cengage. — Framework for linking financial analysis to valuation.

---

*This notebook is the capstone of the Financial Statement Analysis series. It integrates concepts from all prior notebooks — income statement analysis, balance sheet mechanics, cash flow derivation, ratio analysis, and earnings quality — into a single, coherent modelling framework. The iterative circular reference resolution and Monte Carlo simulation demonstrate how a deterministic accounting model can be extended into a probabilistic decision tool.*

### Key Formulas Summary

| Formula | Description |
|---------|-------------|
| $\text{Gross Profit} = \text{Revenue} - \text{COGS}$ | Top-line profitability |
| $\text{EBIT} = \text{Gross Profit} - \text{SG\&A} - \text{R\&D} - \text{D\&A}$ | Operating profitability |
| $\text{Net Income} = (\text{EBIT} - \text{Interest})(1 - t)$ | Bottom-line profitability |
| $\text{CFO} = \text{NI} + \text{D\&A} - \Delta\text{WC}$ | Operating cash flow (indirect) |
| $\text{FCF} = \text{CFO} - \text{Capex}$ | Free cash flow to firm |
| $\text{PP\&E}_t = \text{PP\&E}_{t-1} + \text{Capex} - \text{D\&A}$ | Fixed asset schedule |
| $\text{RE}_t = \text{RE}_{t-1} + \text{NI} - \text{Div}$ | Retained earnings |
| $\text{DSO} = \frac{\text{AR}}{\text{Revenue}} \times 365$ | Collection efficiency |
| $\text{DIO} = \frac{\text{Inventory}}{\text{COGS}} \times 365$ | Inventory efficiency |
| $\text{DPO} = \frac{\text{AP}}{\text{COGS}} \times 365$ | Payment practices |
| $\text{CCC} = \text{DIO} + \text{DSO} - \text{DPO}$ | Cash conversion cycle |
| $\text{DOL} = \frac{\%\Delta\text{EBIT}}{\%\Delta\text{Revenue}}$ | Operating leverage |

### CFA Exam Preparation Notes

This notebook covers material from several CFA Level 1 study sessions:

1. **Financial Statement Analysis** (SS6-9) — Understanding, analysing, and projecting financial statements
2. **Corporate Issuers** (SS10) — Capital allocation, working capital management
3. **Equity Investments** (SS14) — Equity valuation using DCF (which requires a financial model)
4. **Fixed Income** (SS15-16) — Credit analysis relies on projected debt service coverage

> **CFA Exam Tip:** The integrated three-statement model is not directly tested as a calculation problem, but the *concepts* it embodies — statement articulation, ratio analysis, forecasting methodology, and the impact of assumptions on valuation — are tested extensively. Understanding this notebook means understanding the analytical framework the CFA exam assumes.

Key exam-testable concepts from this notebook:
- How depreciation affects all three statements simultaneously
- The indirect method of computing CFO
- Why working capital changes affect cash flow
- The relationship between capex, depreciation, and PP&E
- How leverage (debt) creates circular dependencies
- The difference between scenario analysis and sensitivity analysis
- Monte Carlo simulation methodology and interpretation

### Capstone reflection — what this notebook accomplished

This notebook integrated every concept from the prior six notebooks in the Financial Statement Analysis series:

| Concept | Source Notebook | Used Here |
|---------|----------------|-----------|
| Three statements | Intro | Built simultaneously |
| Income statement structure | IS Analysis | Projected with margin assumptions |
| Inventory and depreciation | BS & Working Capital | Projected via DIO and PP&E roll |
| Cash flow construction | CF Analysis | Indirect method projection |
| Ratio analysis | Ratio & DuPont | Projection drivers (DSO, DIO, DPO, ROE) |
| Earnings quality | Earnings Quality | CFO/NI ratio, FCF/NI ratio checks |

The result is a working financial model that takes operating assumptions and produces complete projected statements with full cross-statement integrity. This is the capstone skill of professional financial analysis.

### Key formulas to remember

| Formula | Application |
|---------|-------------|
| Revenue × Cost% = Cost item | Projecting cost components |
| (DSO/365) × Revenue = AR | Projecting receivables |
| (DIO/365) × COGS = Inventory | Projecting inventory |
| (DPO/365) × COGS = AP | Projecting payables |
| Beginning + CapEx − Depreciation = Net PP&E | Asset roll-forward |
| Beginning + NI − Dividends = Retained Earnings | Equity roll-forward |
| Total Assets − Other Liab − Equity = Debt (plug) | Balance sheet plug |
| FCFF = CFO + Int(1−t) − CapEx | Free cash flow to firm |
| FCFE = FCFF − Int(1−t) + Net Borrowing | Free cash flow to equity |

### CFA exam preparation notes

While the CFA Level 1 exam does not test full 3-statement modelling (that's typically Level 2), the underlying concepts are all examinable:

1. **Articulation between statements** — be able to derive one statement's numbers from the others
2. **Working capital projection** — given DSO, DIO, DPO and revenue, project AR, inventory, AP
3. **Retained earnings build** — given NI, dividends, and beginning RE, compute ending RE
4. **PP&E roll-forward** — given CapEx, depreciation, and beginning PP&E, compute ending PP&E
5. **Balance sheet plug logic** — understand how cash or debt absorbs the imbalance
6. **Free cash flow computation** — from any of the three starting points

### Where to go from here

This notebook completes the Financial Statement Analysis series. The natural next steps in the CFA curriculum:

* **Equity Valuation:** Use the projected FCFF/FCFE in DCF models to estimate intrinsic value
* **Industry & Company Analysis:** Apply Porter's 5 Forces, SWOT, and competitive analysis to frame projections
* **Credit Analysis:** Use the projected EBITDA, FCF, and leverage ratios to assess credit risk
* **Quantitative Analysis:** Apply statistical methods to validate assumptions and assess sensitivity